# Intelligent Task Offloading in 5G Heterogeneous Edge-Fog-Cloud Environments
## HERO / LSH-GRL-HPFOO: Reproducible Single-Notebook Implementation (V5, reviewer revision)

**Research scholar:** Poonam Arjun Undre (PRN 23079001137) · **Guide:** Dr. Kanhaiya Sharma
Symbiosis Institute of Technology, Symbiosis International (Deemed University)

---

### What changed in V5 (reviewer comments → where they are answered)

| Reviewer item | What V5 adds | Section |
|---|---|---|
| 02 Fairness / equity | Formal fairness metrics (Jain's index over tenants and servers, per-tenant miss rate, miss-rate gap, p95/p99 and worst-tenant latency, high-priority and URLLC miss rates) for **every** method, plus a **max-min fair** baseline | §2, §9b |
| 03 State-of-the-art comparison | **GA, PSO and GWO** refiners as drop-in replacements for HPFOO (identical shortlist, reward, seeds and fitness-evaluation budget); **GAT-PPO** (GAPO-style graph RL) baseline | §5b, §5c, §15 |
| 04 Ablation evidence | One unified 10-seed run for all ablations; paired 95% CIs, Cohen's d, Holm correction for every component; graph-adjacency bug fixed; LSH lookup timed against exhaustive scan **at the memory size actually used** | §9, §9d |
| 05 Multi-seed robustness | Mobility, 15 perturbed conditions and 4 arrival-time distributions all repeated over the 10 seeds, with Holm-corrected tests | §9c |
| 06 Generalisation | Leave-one-source-out: train on 3 traces, test on the unseen 4th, with all calibration refitted on the training traces only | §16 |
| 07 Complexity / scalability | Decision time, training time, throughput and peak memory as the network grows (19 → 145 machines) and the task count grows; empirical complexity exponents | §17 |
| 08 Cost reporting | Mean infrastructure cost (absolute and relative to cloud) reported for every method | §2, §9 |

**Bug fixed in V5.** In V4, `build_agent` always passed the *seed-0* network graph to the agent, while the
environment for seeds 1–9 had a different graph. The graph encoder therefore saw the wrong topology in 9 of 10
seeds. V5 passes each seed's own graph (`ADJ_FROM_SEED_ENV = True`). Set it to `False` to reproduce V4 exactly.

### Objective mapping (from the research proposal)

| Proposal objective | Where it is addressed | Evidence produced |
|---|---|---|
| **O1.** Adaptive offloading algorithm for efficient clustering | §4, §9d | LSH retrieval recall, query-time scaling, in-pipeline lookup cost |
| **O2.** System for dynamic, context-aware offloading decisions | §5–§7 | Graph-RL agent with per-node scoring, held-out evaluation |
| **O3.** Optimise allocation for latency, energy, resource utilisation | §2, §3, §9b | Four-term reward; cost; server-utilisation fairness; oracle headroom |
| **O4.** Test/validate vs baselines under varying conditions | §9, §15–§17 | 10 baselines, ablation, multi-seed statistics, robustness, generalisation, scaling |

### Two mandatory gates
The notebook **stops** if either fails (except in the `smoke` profile, which only tests that the code runs).

### Runtime (Colab, same GPU runtime as V4)
`PROFILE = "full"` ≈ **1.5–2 h**. `PROFILE = "quick"` ≈ 25–35 min (fewer seeds, for a dry run). Finished per-seed
results are cached on Drive, so after a disconnect simply **Run all** again and finished work is skipped.

## §0 — Configuration and seed control

All constants live here. `set_seed()` is the **only** place the RNG is seeded: constructors
never reset it, so multi-seed runs in §9 are genuinely independent.

In [1]:
# Colab: torch is preinstalled. If running elsewhere and this fails:
#   !pip install torch pandas numpy matplotlib scipy scikit-learn openpyxl
import torch, scipy, matplotlib
print("torch", torch.__version__, "| scipy", scipy.__version__)

torch 2.11.0+cu130 | scipy 1.16.3


In [2]:
# ===== SECTION 0: CONFIG =====
import os, sys, json, time, math, random, warnings, pickle, copy
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
try:
    from IPython.display import display
except ImportError:                      # plain-Python execution
    display = print

# ---- POINT THIS AT YOUR dataset/ FOLDER -------------------------------------
DRIVE_BASE = "/content/drive/MyDrive/5G"
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False
DATA_DIR  = os.environ.get("LSHGRL_DATA", os.path.join(DRIVE_BASE, "dataset"))
OUT_DIR   = os.environ.get("LSHGRL_OUT", "./results_V5")
DRIVE_OUT = os.path.join(DRIVE_BASE, "results_V5")
os.makedirs(OUT_DIR, exist_ok=True)

# "full"  : the run whose numbers go into the paper (~1.5-2 h)
# "quick" : dry run with fewer seeds (~25-35 min)
# "smoke" : tiny settings, only checks that every cell executes (gates not enforced)
PROFILE = os.environ.get("LSHGRL_PROFILE", "full")
P = {
 "smoke": dict(N_TASKS=2000, EPISODES=4, EP_LEN=100, SEEDS=[0, 1], EVAL_EVERY=2,
               SCHED_SEEDS=[0, 1], REFINER_SEEDS=[0, 1], LOSO_SEEDS=[0, 1], SCALE_SEEDS=[0],
               TOPOLOGIES=[(10, 5, 3), (20, 10, 6)], TASK_COUNTS=[250, 500],
               BUDGET_EPISODES=[2, 4]),
 "quick": dict(N_TASKS=4000, EPISODES=12, EP_LEN=200, SEEDS=[0, 1, 2], EVAL_EVERY=3,
               SCHED_SEEDS=[0, 1, 2], REFINER_SEEDS=[0, 1, 2], LOSO_SEEDS=[0, 1], SCALE_SEEDS=[0],
               TOPOLOGIES=[(10, 5, 3), (20, 10, 6), (40, 20, 12)], TASK_COUNTS=[1000, 2000, 4000],
               BUDGET_EPISODES=[3, 6, 12]),
 "full":  dict(N_TASKS=8000, EPISODES=25, EP_LEN=250, SEEDS=list(range(10)), EVAL_EVERY=5,
               SCHED_SEEDS=[0, 1, 2, 3, 4], REFINER_SEEDS=list(range(10)), LOSO_SEEDS=[0, 1, 2, 3, 4],
               SCALE_SEEDS=[0, 1, 2], TOPOLOGIES=[(10, 5, 3), (20, 10, 6), (40, 20, 12), (80, 40, 24)],
               TASK_COUNTS=[1000, 2000, 4000, 8000, 16000], BUDGET_EPISODES=[6, 12, 25, 50]),
}[PROFILE]

# Experiments added in V5 (switch any off to save time)
RUN = dict(refiners=True, loso=True, scalability=True)
# V4 bug fix: give each seed's agent its own environment graph (V4 always used seed 0's graph).
# False reproduces V4 numbers exactly; True is the scientifically correct setting.
ADJ_FROM_SEED_ENV = True
# Reuse finished per-seed results after a Colab disconnect. Delete the cache folder after
# changing any code or setting, otherwise old results are reused.
RESUME = True
CODE_VERSION = "V5.0-adj%d" % int(ADJ_FROM_SEED_ENV)

NUM_EDGE, NUM_FOG, NUM_CLOUD = 10, 5, 3
N_NODES = 1 + NUM_EDGE + NUM_FOG + NUM_CLOUD   # index 0 = LOCAL device
LOCAL, EDGE, FOG, CLOUD = 0, 1, 2, 3
TIER = np.array([LOCAL] + [EDGE]*NUM_EDGE + [FOG]*NUM_FOG + [CLOUD]*NUM_CLOUD)

BANDWIDTH_HZ = 100e6
SNR_DB_BY_MOBILITY = {"static":25.0, "low":18.0, "medium":11.0, "high":5.0}
BACKHAUL_BPS = {LOCAL:np.inf, EDGE:10e9, FOG:5e9, CLOUD:1e9}
P_TX_W, P_IDLE_W = 0.6, 0.40      # 0.40 W = 5G NR RRC-connected modem draw while waiting
KAPPA_DEVICE = 1e-9            # Joules per device CPU cycle
E_REF_J = 0.5                  # energy normalisation reference
W_SUCCESS, W_LAT, W_ENERGY, W_COST = 0.35, 0.15, 0.15, 0.35
# Resource price per tier (proposal Research Gap #2: latency-vs-energy AND bandwidth-vs-cost).
# Edge is cheap and scarce, cloud is expensive and abundant.
PRICE_CYCLE = {LOCAL:0.0, EDGE:1.0e-11, FOG:3.0e-11, CLOUD:1.0e-10}
PRICE_BIT   = {LOCAL:0.0, EDGE:1.0e-11, FOG:4.0e-11, CLOUD:1.5e-10}
RESULT_RATIO = 0.05

CAP_RANGE = {LOCAL:(2e9,2e9), EDGE:(6e9,14e9), FOG:(35e9,65e9), CLOUD:(150e9,250e9)}
PROP_RANGE = {LOCAL:(0.0,0.0), EDGE:(2.0,6.0), FOG:(12.0,22.0), CLOUD:(40.0,70.0)}

STRICT_DATA = True             # abort rather than silently use synthetic data
STRICT_GATES = PROFILE != "smoke"
TARGET_LOCAL_MEDIAN_MS = 250.0 # calibrate workload intensity to the device class
TARGET_ORACLE_MISS = 0.06
TARGET_EDGE_UTIL   = 0.90


def set_seed(s):
    random.seed(s); np.random.seed(s)
    try:
        import torch; torch.manual_seed(s)
    except Exception: pass


PLOTS_DIR = os.path.join(OUT_DIR, "plots"); os.makedirs(PLOTS_DIR, exist_ok=True)
PLOT_DPI, PLOT_FIGSIZE = (1000 if PROFILE != "smoke" else 110), (8, 5)
COLORS = {"lsh_grl_hpfoo":"#0D47A1","lsh_grl":"#1565C0","grl":"#EF6C00","no_graph":"#8D6E63",
          "dqn":"#2E7D32","greedy":"#6A1B9A","gat_ppo":"#00838F","maxmin":"#AD1457",
          "lsh_grl_ga":"#F9A825","lsh_grl_pso":"#7CB342","lsh_grl_gwo":"#5E35B1",
          "random":"#C62828","local_only":"#5D4037","all_cloud":"#455A64","oracle":"#000000"}
LABELS = {"lsh_grl_hpfoo":"LSH-GRL-HPFOO (Proposed)","lsh_grl":"LSH-GRL (no HPFOO)",
          "grl":"GRL only","no_graph":"No-graph scoring","dqn":"Standard DQN",
          "gat_ppo":"GAT-PPO (graph RL)","maxmin":"Max-min fair",
          "lsh_grl_ga":"LSH-GRL-GA","lsh_grl_pso":"LSH-GRL-PSO","lsh_grl_gwo":"LSH-GRL-GWO",
          "greedy":"Greedy (min-latency)","random":"Random Offloading",
          "local_only":"Local-Only","all_cloud":"All-to-Cloud","oracle":"Myopic Oracle (bound)"}
# ---- Requirement 2: Pufferfish perturbation-radius schedule ---------------------
# "legacy"   : (ub-lb)/t with t monotonic across the whole run (v3 behaviour). ADOPTED.
# "reset"    : (ub-lb)/tau with tau = 1..iters, reset every invocation.
# "annealed" : (ub-lb)/tau * 1/sqrt(1+k), k = invocation index.
PUFFERFISH_SCHEDULE = "legacy"

# ---- result cache (survives Colab disconnects when Drive is mounted) ----------
CACHE_DIR = os.path.join(DRIVE_OUT if IN_COLAB else OUT_DIR, "cache", PROFILE)
os.makedirs(CACHE_DIR, exist_ok=True)

def cached(name, fn):
    """Run fn() once and store its (picklable) result; reuse it on later runs."""
    path = os.path.join(CACHE_DIR, name + ".pkl")
    if RESUME and os.path.exists(path):
        with open(path, "rb") as f: obj = pickle.load(f)
        if obj.get("_version") == CODE_VERSION:
            print("  [cache] reused", name); return obj["data"]
    data = fn()
    with open(path, "wb") as f: pickle.dump({"_version": CODE_VERSION, "data": data}, f)
    return data

RESULTS = {}
print("profile:", PROFILE, P)
print("pufferfish schedule:", PUFFERFISH_SCHEDULE, "| per-seed graph fix:", ADJ_FROM_SEED_ENV)
print("cache folder:", CACHE_DIR)

Mounted at /content/drive
profile: full {'N_TASKS': 8000, 'EPISODES': 25, 'EP_LEN': 250, 'SEEDS': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'EVAL_EVERY': 5, 'SCHED_SEEDS': [0, 1, 2, 3, 4], 'REFINER_SEEDS': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9], 'LOSO_SEEDS': [0, 1, 2, 3, 4], 'SCALE_SEEDS': [0, 1, 2], 'TOPOLOGIES': [(10, 5, 3), (20, 10, 6), (40, 20, 12), (80, 40, 24)], 'TASK_COUNTS': [1000, 2000, 4000, 8000, 16000], 'BUDGET_EPISODES': [6, 12, 25, 50]}
pufferfish schedule: legacy | per-seed graph fix: True
cache folder: /content/drive/MyDrive/5G/results_V5/cache/full


## §1 — Data harmonisation

Four real traces are mapped into one task schema. Trace fields do not map one-to-one onto a
task model, so the conversions below are **stated modelling assumptions**, not measurements.
If a file is missing, a clearly-labelled synthetic substitute is used and reported — it is
never silently swapped in.

In [3]:
# ===== SECTION 1: DATA HARMONISATION =====
MOB_IDX = {"static":0.05, "low":0.2, "medium":0.45, "high":0.8}

def _load_task_offloading(n):
    p = os.path.join(DATA_DIR, "task_offloading_dataset.csv")
    df = pd.read_csv(p).sample(n=min(n,10**7), random_state=42)
    traf = (pd.to_numeric(df["Downlink_Traffic"], errors="coerce").fillna(0) +
            pd.to_numeric(df["Uplink_Traffic"], errors="coerce").fillna(0))
    dur = pd.to_numeric(df["Duration"], errors="coerce").fillna(1).clip(1, 600)
    return pd.DataFrame({
        "cpu_cycles": (traf*2.0e4 + dur*3.0e7).clip(5e6, 3e9),
        "data_size_bytes": (traf*1024.0).clip(1e3, 2e7),
        "mobility_index": 0.45,
        "priority": pd.to_numeric(df["RAT"], errors="coerce").fillna(1)*50,
        "source": "task_offloading"})

def _load_vec(n):
    p = os.path.join(DATA_DIR, "VEC_edge_data.csv")
    df = pd.read_csv(p).sample(n=min(n,10**7), random_state=42)
    ex = pd.to_numeric(df["execution_time"], errors="coerce").fillna(50).clip(1, 2000)
    return pd.DataFrame({
        "cpu_cycles": (ex*1e-3*2e9).clip(5e6, 3e9),
        "data_size_bytes": (ex*2.0e3).clip(1e3, 2e7),
        "mobility_index": df["device_mobility_status"].map(MOB_IDX).fillna(0.45).values,
        "priority": 100.0,
        "source": "vec_edge"})

def _load_azure(n):
    p = os.path.join(DATA_DIR, "New folder", "AzureLLMInferenceTrace_conv.csv")
    df = pd.read_csv(p).sample(n=min(n,10**7), random_state=42)
    ct = pd.to_numeric(df["ContextTokens"], errors="coerce").fillna(500)
    gt = pd.to_numeric(df["GeneratedTokens"], errors="coerce").fillna(100)
    return pd.DataFrame({
        "cpu_cycles": ((ct+gt)*5.0e4).clip(5e6, 3e9),
        "data_size_bytes": (ct*4.0).clip(1e3, 2e7),
        "mobility_index": 0.05,
        "priority": 150.0,
        "source": "azure_llm"})

def _load_borg(n):
    p = os.path.join(DATA_DIR, "borg_traces_data.csv")
    df = pd.read_csv(p, usecols=["start_time","end_time","priority","assigned_memory"],
                     nrows=120000)
    st = pd.to_numeric(df["start_time"], errors="coerce")
    et = pd.to_numeric(df["end_time"], errors="coerce")
    dur_us = (et-st).abs()
    df = df.assign(dur=dur_us).dropna(subset=["dur"])
    df = df[(df["dur"] > 0)].sample(n=min(n, len(df)), random_state=42)
    mem = pd.to_numeric(df["assigned_memory"], errors="coerce").fillna(0.001)
    return pd.DataFrame({
        "cpu_cycles": (df["dur"].values/1e6*5e8).clip(5e6, 3e9),
        "data_size_bytes": (mem.values*1e8).clip(1e3, 2e7),
        "mobility_index": 0.05,
        "priority": pd.to_numeric(df["priority"], errors="coerce").fillna(100).values,
        "source": "borg"})

def _synthetic(n, tag="synthetic"):
    rng = np.random.RandomState(42)
    return pd.DataFrame({
        "cpu_cycles": np.exp(rng.uniform(np.log(5e6), np.log(3e9), n)),
        "data_size_bytes": np.exp(rng.uniform(np.log(1e3), np.log(2e7), n)),
        "mobility_index": rng.choice(list(MOB_IDX.values()), n),
        "priority": rng.uniform(50,300,n), "source": tag})

def harmonize(n_total, verbose=True):
    per = n_total // 4
    frames, log = [], []
    for name, fn in [("task_offloading",_load_task_offloading), ("vec_edge",_load_vec),
                     ("azure_llm",_load_azure), ("borg",_load_borg)]:
        try:
            f = fn(per); frames.append(f); log.append((name, len(f), "loaded"))
        except Exception as e:
            if STRICT_DATA:
                raise FileNotFoundError(
                    "Loader '%s' failed: %s\nDATA_DIR is %r (absolute: %s).\nFix the path: "
                    "synthetic substitutes would make every downstream result meaningless."
                    % (name, e, DATA_DIR, os.path.abspath(DATA_DIR))) from e
            f = _synthetic(per, name+"_synth"); frames.append(f)
            log.append((name, len(f), "SYNTHETIC fallback: %s" % str(e)[:60]))
    df = pd.concat(frames, ignore_index=True)
    rng = np.random.RandomState(42)
    df["mobility_index"] = df["mobility_index"].astype(float)
    # latency budget (deadline) is a workload parameter, calibrated in Section 2
    df["deadline_raw_ms"] = rng.uniform(30, 400, len(df))
    df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
    df["task_id"] = np.arange(len(df))
    if STRICT_DATA:
        bad = [n for n, _, s in log if s != "loaded"]
        assert not bad, "Synthetic fallback used for: %s" % bad
    if verbose:
        print("Harmonisation report")
        for nme, k, st in log: print("  %-16s %6d  %s" % (nme, k, st))
        print("  TOTAL %d tasks" % len(df))
    return df, log


def load_site_topology(n_sites=30):
    try:
        s = pd.read_csv(os.path.join(DATA_DIR,"New folder","site.csv"),
                        usecols=["SITE_ID","LATITUDE","LONGITUDE"]).dropna()
        s = s.sample(n=n_sites, random_state=42)
        return s[["LATITUDE","LONGITUDE"]].values
    except Exception:
        return None

In [4]:
df, load_log = harmonize(P["N_TASKS"])
sites = load_site_topology(30)
print("\nreal site topology:", "loaded %d sites" % len(sites) if sites is not None else "unavailable")

# Bimodal deadline mix: 30% latency-critical (URLLC-like), 70% relaxed (eMBB-like).
_rng = np.random.RandomState(7); _n = len(df)
_urgent = _rng.random(_n) < 0.30
df["deadline_raw_ms"] = np.where(_urgent, _rng.uniform(15, 60, _n), _rng.uniform(150, 600, _n))
df["urgent"] = _urgent
print("urgent (tight-deadline) share: %.1f%%" % (100*_urgent.mean()))

# Compute-intensity calibration: trace-to-cycles mappings are modelling assumptions,
# so absolute compute demand depends on which traces loaded. Scale once so the median
# task is a genuine load on the modelled 2 GIPS device class. Reported, not hidden.
# V5: the unscaled cycles are kept so the leave-one-source-out study (§16) can refit
# this factor on its training traces only.
df["cpu_cycles_raw"] = df["cpu_cycles"].copy()
_med_local_ms = df["cpu_cycles"].median() / CAP_RANGE[LOCAL][0] * 1000
CPU_SCALE = TARGET_LOCAL_MEDIAN_MS / _med_local_ms
df["cpu_cycles"] = (df["cpu_cycles"] * CPU_SCALE).clip(5e6, 1e10)
print("compute-intensity scale : %.2fx  (median local exec %.0f -> %.0f ms)"
      % (CPU_SCALE, _med_local_ms, TARGET_LOCAL_MEDIAN_MS))
assert sites is not None, "site.csv did not load - edge propagation loses real-topology grounding."
display(df[["cpu_cycles","data_size_bytes","deadline_raw_ms","mobility_index","source"]].describe().T)

Harmonisation report
  task_offloading    2000  loaded
  vec_edge           2000  loaded
  azure_llm          2000  loaded
  borg               2000  loaded
  TOTAL 8000 tasks

real site topology: loaded 30 sites
urgent (tight-deadline) share: 30.5%
compute-intensity scale : 1.10x  (median local exec 228 -> 250 ms)


,count,mean,std,min,25%,50%,75%,max
cpu_cycles,8000.0,1.124282e+09,1.269811e+09,5.866228e+06,7.711623e+07,5.000000e+08,1.971552e+09,3.289474e+09
data_size_bytes,8000.0,1.183399e+06,2.995885e+06,1.000000e+03,4.783000e+03,3.072000e+05,1.101685e+06,2.000000e+07
deadline_raw_ms,8000.0,2.710469e+02,1.894780e+02,1.500098e+01,5.125768e+01,2.731104e+02,4.381534e+02,5.998507e+02
mobility_index,8000.0,2.330312e-01,2.334091e-01,5.000000e-02,5.000000e-02,5.000000e-02,4.500000e-01,8.000000e-01


## §2 — Environment model, evaluation measures and **Gate 1**

Latency = uplink (Shannon) + backhaul serialisation + **propagation RTT** + queueing + compute + downlink.

Energy is **device** energy (radio TX + RRC-connected idle, or local CPU), not server energy.

Reward has four terms, matching Research Gap #2 of the proposal (*latency vs energy*, *bandwidth vs cost*):

$$r = w_s\,\mathbb{1}[\ell \le D] + w_l\Big(1-\tfrac{\ell}{D}\Big)^+ + w_e\Big(1-\tfrac{E}{E_{local}}\Big)^+ + w_c\Big(1-\tfrac{c}{c_{cloud}}\Big)^+$$

### V5: cost and fairness measures (computed for every method, every seed)

**Cost.** Mean infrastructure cost $\bar C = \frac1N\sum_k C_{k,a_k}$ and mean cost ratio $\frac1N\sum_k C_{k,a_k}/C^{ref}_k$ (1 = everything priced as cloud, 0 = everything local).

**Tenants.** The four workload sources (cellular, vehicular, Azure LLM, Borg) are treated as four tenants $g \in \mathcal G$, i.e. four user/application groups sharing the same infrastructure.

| Measure | Definition |
|---|---|
| Per-tenant miss rate | $M_g = 1 - \frac{1}{\lvert K_g\rvert}\sum_{k\in K_g}\mathbb 1[\ell_k \le D_k]$ |
| Jain's index over tenants | $J_{\text{ten}} = \dfrac{(\sum_g x_g)^2}{\lvert\mathcal G\rvert\sum_g x_g^2}$ with $x_g = 1-M_g$ (1 = perfectly equal service) |
| Miss-rate gap | $\Delta M = \max_g M_g - \min_g M_g$ |
| Worst-tenant miss / latency | $\max_g M_g$, $\max_g P_{95}(\ell \mid g)$ |
| Tail latency | $P_{95}(\ell)$, $P_{99}(\ell)$, $\max\ell$ |
| Server-load fairness | $J_{\text{srv}}$ = Jain's index over $u_j = \sum_{k: a_k=j} c_k / f_j$ (seconds of work given to server $j$); $J_{\text{edge}}$ over the edge tier only |
| Priority groups | miss rate of latency-critical (URLLC-like) jobs and of top-quartile-priority jobs |

**Max-min fair baseline.** A non-learning policy that targets fairness directly, using two rules.
(1) *Tenant max-min:* each tenant's success rate is tracked as a running estimate (an exponential moving average
of the policy's own predicted deadline outcomes, weight 0.02). A job from the currently worst-served tenant
(lowest estimate, more than 1 point behind the best) is sent to the fastest node, so the minimum tenant success
rate is lifted first.
(2) *Load balancing:* every other job goes to the cheapest tier that can still meet its deadline. Within that
tier it goes to the server with the least queued work after placement, $(Q_j + c_k)/f_j$, which maximises the
minimum spare capacity across servers. If no server can meet the deadline, the job goes to the fastest node.
Like greedy, it uses the exact latency model.

In [5]:
# ===== SECTION 2: ENVIRONMENT =====
class OffloadingEnv:
    # 19 actions: 0 = local device, 1..10 edge, 11..15 fog, 16..18 cloud.
    # Latency = uplink + backhaul + propagation RTT + queueing + compute + downlink.
    # Energy is DEVICE energy: radio TX power * uplink time + idle power * wait,
    # or local CPU energy if executed on device.
    def __init__(self, seed=0, sites=None, deadline_scale=1.0, arrival_scale=1.0):
        rng = np.random.RandomState(seed)
        self.rng = rng
        self.cap = np.zeros(N_NODES); self.prop = np.zeros(N_NODES)
        for i in range(N_NODES):
            lo,hi = CAP_RANGE[TIER[i]];  self.cap[i]  = rng.uniform(lo,hi)
            lo,hi = PROP_RANGE[TIER[i]]; self.prop[i] = rng.uniform(lo,hi)
        if sites is not None and len(sites) >= NUM_EDGE:
            pts = sites[rng.choice(len(sites), NUM_EDGE, replace=False)]
            c = pts.mean(0); d = np.linalg.norm(pts-c, axis=1)
            d = (d-d.min())/(np.ptp(d)+1e-9)
            self.prop[1:1+NUM_EDGE] = 2.0 + 4.0*d      # real site spread -> RTT spread
            self.site_coords = pts
        else:
            self.site_coords = None
        self.backhaul = np.array([BACKHAUL_BPS[TIER[i]] for i in range(N_NODES)])
        self.deadline_scale = deadline_scale
        self.arrival_scale  = arrival_scale
        self.adj = self._build_adj()
        self.reset()

    def _build_adj(self):
        A = np.zeros((N_NODES, N_NODES))
        e0, f0, c0 = 1, 1+NUM_EDGE, 1+NUM_EDGE+NUM_FOG
        for i in range(e0, f0): A[0,i]=A[i,0]=1                    # device <-> edge
        for i in range(e0,f0):
            for j in range(i+1,f0):
                if self.rng.random() < 0.4: A[i,j]=A[j,i]=1        # edge mesh
        for k,i in enumerate(range(e0,f0)):
            j = f0 + (k % NUM_FOG); A[i,j]=A[j,i]=1                # edge -> fog
        for i in range(f0,c0):
            for j in range(c0,N_NODES): A[i,j]=A[j,i]=1            # fog -> cloud
        A += np.eye(N_NODES)
        return (A>0).astype(np.float32)

    def reset(self):
        self.tenant_succ = {}              # V5: used only by the max-min fair baseline
        self.backlog = np.zeros(N_NODES)   # queued cycles per node
        self.nserved = np.zeros(N_NODES)
        self.t_ms = 0.0

    def advance_to(self, t_ms):
        dt = max(0.0, t_ms - self.t_ms)
        self.backlog = np.maximum(0.0, self.backlog - self.cap*dt/1000.0)
        self.t_ms = t_ms

    def uplink_rate(self, mobility_index):
        if   mobility_index <= 0.10: snr = SNR_DB_BY_MOBILITY["static"]
        elif mobility_index <= 0.30: snr = SNR_DB_BY_MOBILITY["low"]
        elif mobility_index <= 0.60: snr = SNR_DB_BY_MOBILITY["medium"]
        else:                        snr = SNR_DB_BY_MOBILITY["high"]
        return BANDWIDTH_HZ * math.log2(1.0 + 10**(snr/10.0))

    def evaluate(self, task, a):
        # returns (latency_ms, device_energy_J) WITHOUT mutating state
        cyc  = task["cpu_cycles"]; bits = task["data_size_bytes"]*8.0
        if a == 0:
            # local runs on the requesting device: no shared queue
            lat = cyc/self.cap[0]*1000.0
            return lat, cyc*KAPPA_DEVICE
        q_ms = self.backlog[a]/self.cap[a]*1000.0
        r_up = self.uplink_rate(task["mobility_index"])
        t_up = bits/r_up*1000.0
        t_bh = bits/self.backhaul[a]*1000.0
        t_cp = cyc/self.cap[a]*1000.0
        t_dn = (bits*RESULT_RATIO)/(2.0*r_up)*1000.0
        lat  = t_up + t_bh + self.prop[a] + q_ms + t_cp + t_dn
        e    = P_TX_W*(t_up/1000.0) + P_IDLE_W*max(0.0,(lat-t_up))/1000.0
        return lat, e

    def cost(self, task, a):
        t = TIER[a]
        return (PRICE_CYCLE[t]*task["cpu_cycles"] +
                PRICE_BIT[t]*task["data_size_bytes"]*8.0)

    def cost_ref(self, task):
        return (PRICE_CYCLE[CLOUD]*task["cpu_cycles"] +
                PRICE_BIT[CLOUD]*task["data_size_bytes"]*8.0)

    def evaluate_all(self, task):
        cyc = task["cpu_cycles"]; bits = task["data_size_bytes"]*8.0
        r_up = self.uplink_rate(task["mobility_index"])
        t_up = bits/r_up*1000.0
        lat = (t_up + bits/self.backhaul*1000.0 + self.prop
               + self.backlog/self.cap*1000.0 + cyc/self.cap*1000.0
               + (bits*RESULT_RATIO)/(2.0*r_up)*1000.0)
        e = P_TX_W*(t_up/1000.0) + P_IDLE_W*np.maximum(0.0, lat-t_up)/1000.0
        lat = lat.copy(); e = e.copy()
        lat[0] = cyc/self.cap[0]*1000.0; e[0] = cyc*KAPPA_DEVICE
        return lat, e

    def reward(self, lat, energy, deadline, e_local, cost, cost_ref):
        ok  = 1.0 if lat <= deadline else 0.0
        rl  = float(np.clip(1.0 - lat/max(deadline,1e-6), 0.0, 1.0))
        re  = float(np.clip(1.0 - energy/max(e_local,1e-9), 0.0, 1.0))
        rc  = float(np.clip(1.0 - cost/max(cost_ref,1e-12), 0.0, 1.0))
        return W_SUCCESS*ok + W_LAT*rl + W_ENERGY*re + W_COST*rc, ok

    def step(self, task, a):
        lat, e = self.evaluate(task, a)
        r, ok = self.reward(lat, e, task["deadline_ms"], task["cpu_cycles"]*KAPPA_DEVICE,
                            self.cost(task,a), self.cost_ref(task))
        if a != 0: self.backlog[a] += task["cpu_cycles"]
        self.nserved[a] += 1
        return r, lat, e, ok

    def node_features(self):
        # [cap_norm, backlog_ratio, queue_ms_norm, prop_norm, tier onehot(4)]
        q = self.backlog/self.cap*1000.0
        f = np.zeros((N_NODES, 8), dtype=np.float32)
        f[:,0] = self.cap/2.5e11
        f[:,1] = np.clip(self.backlog/(self.cap*0.5), 0, 3)
        f[:,2] = np.clip(q/100.0, 0, 3)
        f[:,3] = self.prop/70.0
        for i in range(N_NODES): f[i, 4+TIER[i]] = 1.0
        return f


TASK_FEAT_DIM = 8
def task_features(task, env):
    return np.array([
        math.log10(task["cpu_cycles"])/10.0,
        math.log10(task["data_size_bytes"])/8.0,
        task["deadline_ms"]/400.0,
        task["mobility_index"],
        task["priority"]/300.0,
        min(task["cpu_cycles"]/2e9, 2.0),                 # local compute time (s)
        min(task["data_size_bytes"]*8/env.uplink_rate(task["mobility_index"]), 2.0),
        min(task["deadline_ms"]/(task["cpu_cycles"]/2e9*1000.0+1e-6), 3.0)/3.0,  # local slack
    ], dtype=np.float32)


def interarrivals(n, mean_ms, pattern="poisson", seed=1234):
    # All patterns share the same MEAN gap, so only the burstiness changes.
    #   poisson    : exponential gaps (the V4 default; reproduces V4 streams exactly)
    #   periodic   : constant gaps (no burstiness)
    #   bursty     : two-phase hyper-exponential, squared coefficient of variation 4
    #   heavy_tail : Pareto gaps, shape 1.5 (infinite variance, long idle periods + bursts)
    rng = np.random.RandomState(seed)
    if pattern == "poisson":
        return rng.exponential(mean_ms, n)
    if pattern == "periodic":
        return np.full(n, float(mean_ms))
    if pattern == "bursty":
        c2 = 4.0
        p1 = 0.5*(1.0 + math.sqrt((c2-1.0)/(c2+1.0))); p2 = 1.0 - p1
        m1, m2 = mean_ms/(2*p1), mean_ms/(2*p2)          # balanced-means H2
        pick = rng.random(n) < p1
        return np.where(pick, rng.exponential(m1, n), rng.exponential(m2, n))
    if pattern == "heavy_tail":
        a = 1.5; xm = mean_ms*(a-1.0)/a
        return (rng.pareto(a, n) + 1.0)*xm
    raise ValueError("unknown arrival pattern %r" % pattern)


def make_stream(df, env, mean_interarrival_ms, pattern="poisson"):
    ia = interarrivals(len(df), mean_interarrival_ms, pattern)
    arr = np.cumsum(ia)
    recs = []
    for i, row in enumerate(df.itertuples(index=False)):
        d = dict(cpu_cycles=float(row.cpu_cycles), data_size_bytes=float(row.data_size_bytes),
                 mobility_index=float(row.mobility_index), priority=float(row.priority),
                 deadline_ms=float(row.deadline_raw_ms)*env.deadline_scale,
                 arrival_ms=float(arr[i]), source=row.source, task_id=int(row.task_id),
                 urgent=bool(getattr(row, "urgent", False)))
        recs.append(d)
    return recs


# ---- V5: evaluation summary with cost and fairness measures ----
SOURCES = ["task_offloading", "vec_edge", "azure_llm", "borg"]

def jain(x):
    x = np.asarray(x, dtype=float)
    if len(x) == 0 or not np.any(x): return 1.0
    return float(x.sum()**2 / (len(x)*np.sum(x**2)))

def summarize(stream, env, R, L, E, OK, A, DT):
    R = np.asarray(R, float); L = np.asarray(L, float); E = np.asarray(E, float)
    OK = np.asarray(OK, float); A = np.asarray(A, int); DT = np.asarray(DT, float)
    out = dict(reward=float(np.mean(R)), latency=float(np.mean(L)), energy=float(np.mean(E)),
               success=float(np.mean(OK)), miss=float(1-np.mean(OK)),
               decision_ms=float(np.mean(DT)), decision_p95_ms=float(np.percentile(DT, 95)),
               offload_rate=float(np.mean(A > 0)))
    cyc  = np.array([t["cpu_cycles"] for t in stream])
    bits = np.array([t["data_size_bytes"] for t in stream])*8.0
    cost = _PC[A]*cyc + _PB[A]*bits
    cref = PRICE_CYCLE[CLOUD]*cyc + PRICE_BIT[CLOUD]*bits
    out["cost"] = float(cost.mean()); out["cost_ratio"] = float(np.mean(cost/np.maximum(cref, 1e-12)))
    out["lat_p95"] = float(np.percentile(L, 95)); out["lat_p99"] = float(np.percentile(L, 99))
    out["lat_max"] = float(L.max())
    urg = np.array([bool(t.get("urgent", False)) for t in stream])
    pri = np.array([t["priority"] for t in stream], float)
    hp  = pri >= np.percentile(pri, 75)
    out["miss_urgent"]   = float(1-OK[urg].mean())  if urg.any()    else float("nan")
    out["miss_relaxed"]  = float(1-OK[~urg].mean()) if (~urg).any() else float("nan")
    out["miss_highprio"] = float(1-OK[hp].mean())   if hp.any()     else float("nan")
    src = np.array([t["source"] for t in stream])
    ms, rs, p95 = {}, {}, {}
    for g in SOURCES:
        m = src == g
        if m.any():
            ms[g] = float(1-OK[m].mean()); rs[g] = float(R[m].mean())
            p95[g] = float(np.percentile(L[m], 95))
    out["miss_by_source"], out["reward_by_source"], out["p95_by_source"] = ms, rs, p95
    out["jain_tenant_success"] = jain([1-v for v in ms.values()])
    out["jain_tenant_reward"]  = jain(list(rs.values()))
    out["miss_gap"]          = float(max(ms.values())-min(ms.values())) if ms else float("nan")
    out["worst_tenant_miss"] = float(max(ms.values())) if ms else float("nan")
    out["worst_tenant_p95"]  = float(max(p95.values())) if p95 else float("nan")
    work = np.bincount(A, weights=cyc, minlength=len(env.cap))/env.cap   # seconds of work per node
    out["jain_server_util"] = jain(work[1:])
    out["jain_edge_util"]   = jain(work[1:1+NUM_EDGE])
    out["tier_share"] = {nm: float(np.mean(TIER[A] == t))
                         for nm, t in [("local", LOCAL), ("edge", EDGE), ("fog", FOG), ("cloud", CLOUD)]}
    return out


# ---- policies used for calibration / baselines ----
_PC = np.array([PRICE_CYCLE[t] for t in TIER])
_PB = np.array([PRICE_BIT[t]   for t in TIER])

def reward_all(env, task):
    lat, e = env.evaluate_all(task); D = task["deadline_ms"]
    cyc = task["cpu_cycles"]; bits = task["data_size_bytes"]*8.0
    cost = _PC*cyc + _PB*bits
    cref = PRICE_CYCLE[CLOUD]*cyc + PRICE_BIT[CLOUD]*bits
    return (W_SUCCESS*(lat <= D)
            + W_LAT*np.clip(1.0 - lat/max(D,1e-6), 0, 1)
            + W_ENERGY*np.clip(1.0 - e/max(cyc*KAPPA_DEVICE,1e-9), 0, 1)
            + W_COST*np.clip(1.0 - cost/max(cref,1e-12), 0, 1)), lat, e

def myopic_oracle_action(env, task):
    return int(np.argmax(reward_all(env, task)[0]))

def greedy_latency_action(env, task):
    return int(np.argmin(env.evaluate_all(task)[0]))

def maxmin_fair_action(env, task):
    # Max-min fair baseline (V5). Two rules:
    # (1) tenant max-min: if the job's tenant currently has the LOWEST estimated success rate
    #     (and trails the best tenant by > 1 point), the job gets the fastest node, so the
    #     worst-served tenant is lifted first;
    # (2) otherwise, load balancing: among servers that meet the deadline, use the cheapest
    #     tier that can, and within it the server with the least queued work after placement,
    #     (Q_j + c_k)/f_j. If no server meets the deadline, the fastest node is used.
    # Success estimates are an EMA of the policy's own PREDICTED deadline outcomes.
    lat, _ = env.evaluate_all(task)
    D = task["deadline_ms"]; g = task.get("source", "?"); s = env.tenant_succ
    worst = len(s) > 1 and s.get(g, 1.0) <= min(s.values()) and s.get(g, 1.0) < max(s.values()) - 0.01
    feas = np.where(lat[1:] <= D)[0] + 1
    if worst or len(feas) == 0:
        a = int(np.argmin(lat))
    else:
        tiers = TIER[feas]; cand = feas[tiers == tiers.min()]
        a = int(cand[np.argmin((env.backlog[cand] + task["cpu_cycles"]) / env.cap[cand])])
    ok = float(lat[a] <= D)
    s[g] = ok if g not in s else 0.98*s[g] + 0.02*ok
    return a


def run_policy(env, stream, action_fn, collect=False):
    env.reset()
    R=[]; L=[]; E=[]; OK=[]; A=[]; DT=[]
    for task in stream:
        env.advance_to(task["arrival_ms"])
        t0=time.perf_counter(); a=action_fn(env, task); DT.append((time.perf_counter()-t0)*1000)
        r,lat,e,ok = env.step(task, a)
        R.append(r); L.append(lat); E.append(e); OK.append(ok); A.append(a)
    out = summarize(stream, env, R, L, E, OK, A, DT)
    if collect: out["_raw"]=dict(R=R,L=L,E=E,OK=OK,A=A)
    return out


def load_sites_n(n):
    # n real cell sites for larger topologies (§17). Falls back to resampling with a small
    # jitter only if site.csv has fewer rows than needed (reported).
    s = pd.read_csv(os.path.join(DATA_DIR, "New folder", "site.csv"),
                    usecols=["SITE_ID", "LATITUDE", "LONGITUDE"]).dropna()
    if len(s) >= n:
        return s.sample(n=n, random_state=42)[["LATITUDE", "LONGITUDE"]].values
    print("  site.csv has %d rows < %d needed: resampling with 0.01 deg jitter" % (len(s), n))
    pts = s.sample(n=n, replace=True, random_state=42)[["LATITUDE", "LONGITUDE"]].values
    return pts + np.random.RandomState(0).normal(0, 0.01, pts.shape)


def calibrate(df, sites, verbose=True):
    # 1) arrival rate -> target edge-tier utilisation
    probe = OffloadingEnv(seed=0, sites=sites)
    mean_cyc = float(df["cpu_cycles"].mean())
    edge_cap = probe.cap[1:1+NUM_EDGE].sum()
    ia = (mean_cyc/edge_cap)*1000.0/TARGET_EDGE_UTIL
    # 2) deadline scale -> target myopic-oracle miss rate
    sub = df.head(1200)
    lo, hi = 0.15, 6.0
    for _ in range(14):
        mid = (lo+hi)/2
        env = OffloadingEnv(seed=0, sites=sites, deadline_scale=mid)
        st  = make_stream(sub, env, ia)
        m   = run_policy(env, st, myopic_oracle_action)["miss"]
        if m > TARGET_ORACLE_MISS: lo = mid
        else: hi = mid
    scale = (lo+hi)/2
    if verbose:
        print("Calibration")
        print("  mean inter-arrival     : %.3f ms  (target edge util %.0f%%)" % (ia, TARGET_EDGE_UTIL*100))
        print("  deadline scale factor  : %.3f    (target oracle miss %.0f%%)" % (scale, TARGET_ORACLE_MISS*100))
    return ia, scale

In [6]:
ia_ms, dl_scale = calibrate(df, sites)
env      = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale)
eval_env = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale)
# Three-way split. Checkpoint selection and all hyper-parameter choices use VAL only;
# TEST is touched exactly once per configuration, at the end.
_a, _b = int(len(df)*0.60), int(len(df)*0.75)
TRAIN_DF = df.iloc[:_a].reset_index(drop=True)
VAL_DF   = df.iloc[_a:_b].reset_index(drop=True)
TEST_DF  = df.iloc[_b:].reset_index(drop=True)
train_stream = make_stream(df.iloc[:_a], env, ia_ms)
val_stream   = make_stream(VAL_DF, env, ia_ms)
test_stream  = make_stream(TEST_DF, env, ia_ms)
print("train %d | validation %d | test %d tasks"
      % (len(train_stream), len(val_stream), len(test_stream)))

Calibration
  mean inter-arrival     : 11.888 ms  (target edge util 90%)
  deadline scale factor  : 2.565    (target oracle miss 6%)
train 4800 | validation 1200 | test 2000 tasks


In [7]:
# ============================ GATE 1: PHYSICS SANITY ============================
probe = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale); probe.reset()
light = dict(cpu_cycles=8e6,  data_size_bytes=4e3, deadline_ms=40,  mobility_index=0.05, priority=100)
heavy = dict(cpu_cycles=2.0e9, data_size_bytes=2e5, deadline_ms=300, mobility_index=0.05, priority=100)
bigdata = dict(cpu_cycles=3e7, data_size_bytes=1.5e7, deadline_ms=400, mobility_index=0.80, priority=100)
E1, F1, C1 = 1, 1+NUM_EDGE, 1+NUM_EDGE+NUM_FOG

checks, PHYS = [], {}
for nm, t in [("light",light), ("heavy",heavy), ("big-data",bigdata)]:
    PHYS[nm] = {k: probe.evaluate(t, a)[0] for k, a in
                [("local",0), ("edge",E1), ("fog",F1), ("cloud",C1)]}

checks.append(("cloud RTT > edge RTT for a small task",
               PHYS["light"]["cloud"] > PHYS["light"]["edge"]))
checks.append(("cloud still beats local on a compute-heavy task",
               PHYS["heavy"]["cloud"] < PHYS["heavy"]["local"]))
checks.append(("local beats cloud on a small task (edge computing premise)",
               PHYS["light"]["local"] < PHYS["light"]["cloud"]))
e_off = probe.evaluate(bigdata, E1)[1]; e_loc = bigdata["cpu_cycles"]*KAPPA_DEVICE
checks.append(("radio energy dominates for a large-payload/light-compute task", e_off > e_loc))
_g = run_policy(eval_env, test_stream, greedy_latency_action)
checks.append(("deadline misses are non-trivial (5%-40%) under a strong heuristic",
               0.03 <= _g["miss"] <= 0.40))
_r = run_policy(eval_env, test_stream, lambda e,t: np.random.randint(N_NODES))
checks.append(("a sensible heuristic beats random", _g["reward"] > _r["reward"]))
before = probe.backlog[E1]
for _ in range(60): probe.step(heavy, E1)
checks.append(("loading a node degrades it (queueing is live)", probe.backlog[E1] > before))

print("GATE 1 - PHYSICS SANITY\n" + "-"*62)
for name, ok in checks: print("  [%s] %s" % ("PASS" if ok else "FAIL", name))
if STRICT_GATES:
    assert all(ok for _, ok in checks), "GATE 1 FAILED - fix the environment before continuing."
elif not all(ok for _, ok in checks):
    print("  (smoke profile: gate failure reported, not enforced)")
print("-"*62 + "\nGATE 1 PASSED\n")
print("latency (ms) by target:")
print(pd.DataFrame(PHYS).T.round(2))
RESULTS["gate1"] = {n: bool(o) for n, o in checks}; RESULTS["physics_table"] = PHYS

GATE 1 - PHYSICS SANITY
--------------------------------------------------------------
  [PASS] cloud RTT > edge RTT for a small task
  [PASS] cloud still beats local on a compute-heavy task
  [PASS] local beats cloud on a small task (edge computing premise)
  [PASS] radio energy dominates for a large-payload/light-compute task
  [PASS] deadline misses are non-trivial (5%-40%) under a strong heuristic
  [PASS] a sensible heuristic beats random
  [PASS] loading a node degrades it (queueing is live)
--------------------------------------------------------------
GATE 1 PASSED

latency (ms) by target:
           local    edge     fog   cloud
light        4.0    4.97   20.01   57.17
heavy     1000.0  191.13   63.05   70.85
big-data    15.0  616.81  642.27  775.06


## §3 — Oracle upper bound and **Gate 2**

The *myopic oracle* brute-forces all 19 actions at every step and takes the reward-maximising
one given the current network state. It is not a globally optimal policy (it cannot reason
about future congestion), but it bounds what any per-step-greedy method can achieve and is
the right reference for how much a learned policy can add.

**The gap between the best heuristic and the oracle is the available headroom.**

In [8]:
ref_policies = {
    "oracle":     myopic_oracle_action,
    "greedy":     greedy_latency_action,
    "random":     lambda e,t: np.random.randint(N_NODES),
    "local_only": lambda e,t: 0,
    "all_cloud":  lambda e,t: 1+NUM_EDGE+NUM_FOG,
    "maxmin":     maxmin_fair_action,          # V5 fairness-aware baseline (runs last: RNG order unchanged)
}
np.random.seed(0)
REF = {k: run_policy(eval_env, test_stream, f) for k, f in ref_policies.items()}
tbl = pd.DataFrame(REF).T[["reward","latency","energy","miss","cost_ratio","offload_rate","decision_ms"]].astype(float)
tbl.columns = ["Reward","Latency (ms)","Device energy (J)","Deadline miss","Cost ratio","Offload rate","Decision (ms)"]
display(tbl.round(4))

# Gate 2 keeps the V4 definition (greedy/random/local/cloud) so the reported headroom is
# comparable; V5 also reports headroom against the new max-min baseline.
best_heur = max((v["reward"], k) for k, v in REF.items() if k not in ("oracle", "maxmin"))
if REF["maxmin"]["reward"] > best_heur[0]:
    print("NOTE: max-min fair (%.4f) beats %s (%.4f) - report it as the strongest rule."
          % (REF["maxmin"]["reward"], best_heur[1], best_heur[0]))
HEADROOM = (REF["oracle"]["reward"] - best_heur[0]) / best_heur[0] * 100
print("\nGATE 2 - HEADROOM\n" + "-"*62)
print("  best heuristic : %-10s reward %.4f" % (best_heur[1], best_heur[0]))
print("  myopic oracle  : %-10s reward %.4f" % ("oracle", REF["oracle"]["reward"]))
print("  HEADROOM       : %.2f%%   (gate requires >= 5%%)" % HEADROOM)
assert (not STRICT_GATES) or HEADROOM >= 5.0, ("GATE 2 FAILED - headroom %.2f%%. A heuristic is already near-optimal, "
                         "so no learning result here would be meaningful. Revisit §2." % HEADROOM)
print("-"*62 + "\nGATE 2 PASSED - there is a real learning problem.\n")
RESULTS["reference"] = {k: {m: x for m, x in v.items() if isinstance(x, (int, float))}
                        for k, v in REF.items()}
RESULTS["headroom_pct"] = float(HEADROOM)
RESULTS["headroom_vs_maxmin_pct"] = float((REF["oracle"]["reward"]-REF["maxmin"]["reward"])/REF["maxmin"]["reward"]*100)

,Reward,Latency (ms),Device energy (J),Deadline miss,Cost ratio,Offload rate,Decision (ms)
oracle,0.8812,115.1329,0.0517,0.060,0.1227,0.9895,0.0607
greedy,0.8196,55.6673,0.0340,0.062,0.3213,0.9305,0.0220
random,0.7283,204.9696,0.1259,0.194,0.2994,0.9520,0.0049
local_only,0.6365,555.8301,1.1117,0.384,0.0000,0.0000,0.0002
all_cloud,0.5102,100.4318,0.0446,0.133,1.0000,1.0000,0.0003
maxmin,0.8386,72.6325,0.0352,0.061,0.2678,0.9910,0.0392


NOTE: max-min fair (0.8386) beats greedy (0.8196) - report it as the strongest rule.

GATE 2 - HEADROOM
--------------------------------------------------------------
  best heuristic : greedy     reward 0.8196
  myopic oracle  : oracle     reward 0.8812
  HEADROOM       : 7.52%   (gate requires >= 5%)
--------------------------------------------------------------
GATE 2 PASSED - there is a real learning problem.



## §4 — LSH engine · **Objective 1**

Proposal §8.2 specifies LSH for *similarity-based clustering to lower the complexity of
searching for optimal offloading targets*. So Objective 1 is evaluated on what LSH is
actually for: **retrieval quality and query-time scaling**, measured against exhaustive
nearest-neighbour search as ground truth.

Three defects in the original implementation are fixed here:
1. reuse was gated on `avg_reward > 0`, always true, so it memorised random exploration;
2. the LSH **verification** step was missing — buckets were used without checking similarity;
3. features were all-positive, so projection signs and cosine similarity both saturated
   (81 distinct buckets over the whole task space). Standardisation fixes this.

In [9]:
class LSHCache:
    # Proper LSH: hash -> CANDIDATES -> exact similarity verification -> reuse.
    # The original implementation skipped verification entirely and gated reuse on
    # "avg_reward > 0", which was always true, so it memorised random exploration.
    def __init__(self, dim=TASK_FEAT_DIM, tables=4, bits=12, seed=0,
                 sim_thresh=0.99, min_count=2, bucket_cap=40):
        rng = np.random.RandomState(seed)
        self.proj = [rng.randn(bits, dim) for _ in range(tables)]
        self.sim_thresh = sim_thresh; self.min_count = min_count
        self.bucket_cap = bucket_cap
        self.store = {}                      # key -> list of (zfeat, action, reward)
        self.mu = np.zeros(dim, dtype=np.float32); self.sd = np.ones(dim, dtype=np.float32)
        self.hits = 0; self.queries = 0
        self.r_ema = None; self.ema_a = 0.01
        self.eval_conf = 99.0   # inference uses the Q-network; LSH does pruning only

    def fit(self, feats):
        # Standardise: raw task features are all-positive, so random-projection signs
        # and cosine similarity both saturate. Centering makes the hash discriminative.
        F = np.asarray(feats, dtype=np.float32)
        self.mu = F.mean(0); self.sd = F.std(0) + 1e-6

    def _z(self, tf):
        return (tf - self.mu) / self.sd

    def _keys(self, z, lb):
        return [(t, lb, tuple((P @ z > 0).astype(np.int8))) for t, P in enumerate(self.proj)]

    @staticmethod
    def load_bucket(env):
        u = float(np.mean(env.backlog[1:] / env.cap[1:]))
        return int(np.clip(u * 4, 0, 3))

    def lookup(self, tf, env):
        self.queries += 1
        z = self._z(tf); nq = np.linalg.norm(z) + 1e-9
        agg = {}
        for k in self._keys(z, self.load_bucket(env)):
            for feat, a, r in self.store.get(k, []):
                sim = float(np.dot(feat, z) / ((np.linalg.norm(feat) + 1e-9) * nq))
                if sim >= self.sim_thresh:                      # VERIFICATION step
                    s, n = agg.get(a, (0.0, 0)); agg[a] = (s + r, n + 1)
        if not agg: return None, 0.0
        a_best, (s, n) = max(agg.items(), key=lambda kv: kv[1][0] / kv[1][1])
        if n < self.min_count: return None, 0.0
        if self.r_ema is not None and s / n < self.r_ema: return None, 0.0
        self.hits += 1
        return int(a_best), float(min(n / 8.0, 1.0))

    def update(self, tf, env, action, reward, allow_write):
        self.r_ema = reward if self.r_ema is None else (1-self.ema_a)*self.r_ema + self.ema_a*reward
        if not allow_write: return
        if self.r_ema is not None and reward < self.r_ema: return
        z = self._z(tf)
        for k in self._keys(z, self.load_bucket(env)):
            b = self.store.setdefault(k, [])
            b.append((z, action, reward))
            if len(b) > self.bucket_cap: b.pop(0)

    def stats(self):
        return dict(hit_rate=self.hits / max(self.queries, 1),
                    buckets=len(self.store),
                    entries=sum(len(v) for v in self.store.values()),
                    queries=self.queries)

In [10]:
# ---- Objective 1 experiment: LSH retrieval vs exhaustive search ----
feat_bank = np.array([task_features(t, env) for t in train_stream], dtype=np.float32)
lsh_probe = LSHCache(seed=0); lsh_probe.fit(feat_bank)
Z = (feat_bank - lsh_probe.mu) / lsh_probe.sd
Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)

def exhaustive_topk(qi, k=10, upto=None):
    S = Zn[:upto] @ Zn[qi]; S[qi] = -2
    return set(np.argpartition(-S, k)[:k])

def lsh_candidates(qi, upto, index):
    z = Z[qi]; out = set()
    for key in lsh_probe._keys(z, 0): out |= index.get(key, set())
    out.discard(qi); return out

# LSH's advantage is asymptotic. Extend the index with jittered replicas so the
# O(N) vs O(1) crossover is actually reached. This is an index-scaling study;
# it does not claim 140k real traces.
_base = Z.copy()
Z = np.vstack([_base] + [_base + np.random.RandomState(s).normal(0, 0.8, _base.shape)
                         for s in range(35)]).astype(np.float32)
Zn = Z / (np.linalg.norm(Z, axis=1, keepdims=True) + 1e-9)
sizes = [s for s in [1000, 5000, 20000, 60000, 140000] if s <= len(Z)]
lsh_rows = []
for N in sizes:
    # Hash width scales with log N so bucket occupancy stays bounded; with a fixed
    # width, candidate lists grow linearly with N and the O(1) property is lost.
    # More hash bits sharpen selectivity but cost recall; more tables restore it.
    lsh_probe = LSHCache(seed=0, tables=10, bits=max(10, int(np.log2(N)) - 1))
    lsh_probe.fit(feat_bank)
    index = {}
    for i in range(N):
        for key in lsh_probe._keys(Z[i], 0): index.setdefault(key, set()).add(i)
    qs = np.random.RandomState(0).choice(N, min(120, N), replace=False)
    rec, t_l, t_e, cand = [], [], [], []
    for qi in qs:
        t0 = time.perf_counter()
        for _ in range(5): c = lsh_candidates(qi, N, index)
        t_l.append((time.perf_counter()-t0)/5)
        t0 = time.perf_counter()
        for _ in range(5): true = exhaustive_topk(qi, 10, N)
        t_e.append((time.perf_counter()-t0)/5)
        rec.append(len(c & true)/10.0); cand.append(len(c))
    lsh_rows.append(dict(N=N, recall_at_10=float(np.mean(rec)),
                         lsh_ms=float(np.mean(t_l)*1000), exhaustive_ms=float(np.mean(t_e)*1000),
                         speedup=float(np.mean(t_e)/max(np.mean(t_l),1e-9)),
                         candidates=float(np.mean(cand)),
                         search_space_reduction=100*(1-np.mean(cand)/N)))
LSH_SCALE = pd.DataFrame(lsh_rows)
display(LSH_SCALE.round(3))
print("\nOBJECTIVE 1 RESULT")
print("  recall@10 vs exhaustive search : %.1f%%" % (100*LSH_SCALE.recall_at_10.mean()))
print("  query speedup at N=%d          : %.1fx" % (LSH_SCALE.N.iloc[-1], LSH_SCALE.speedup.iloc[-1]))
print("  search-space reduction at N=%d : %.1f%%" % (LSH_SCALE.N.iloc[-1],
                                                     LSH_SCALE.search_space_reduction.iloc[-1]))
RESULTS["objective1_lsh"] = LSH_SCALE.to_dict("records")

,N,recall_at_10,lsh_ms,exhaustive_ms,speedup,candidates,search_space_reduction
0,1000,0.982,0.102,0.024,0.234,130.667,86.933
1,5000,0.998,0.165,0.046,0.280,586.475,88.270
2,20000,0.907,0.151,0.148,0.982,768.483,96.158
3,60000,0.918,0.313,0.441,1.409,1475.933,97.540
4,140000,0.932,0.429,1.097,2.559,2342.783,98.327



OBJECTIVE 1 RESULT
  recall@10 vs exhaustive search : 94.8%
  query speedup at N=140000          : 2.6x
  search-space reduction at N=140000 : 98.3%


## §5 — Graph-RL agent · **Objective 2**

Four defects fixed relative to the original code:

| Original | Consequence | Fix here |
|---|---|---|
| GAT had no backward pass | "graph learning" was a fixed random projection | GAT trained end-to-end with the Q-head |
| Graph embedding mean-pooled over all nodes | state could not distinguish nodes, so node selection was unlearnable | **per-node scoring**: each candidate scored separately |
| `next_state` recomputed from the *same* task, `done` always False | degenerate self-bootstrapping target | `next_state` is the **next** task; `done` at episode end |
| Training-phase averages reported as results | reported performance included random exploration | frozen weights, **ε = 0**, held-out tasks |

**How the parts combine (stated exactly as implemented).** During training, an exploratory
step follows an LSH-retrieved action with probability 0.7 when one is available. At decision time the
Q-network scores all nodes, the top `k` form the shortlist, and the LSH memory may add **one** further
candidate. In the full method the refiner (HPFOO) then chooses within this shortlist by fusing the
normalised Q-score and its learned utility with weight α (selected on validation). In LSH-GRL without
HPFOO, the action is the Q-network's argmax, so LSH is not consulted at inference.

In [11]:
import math, time, random, copy
import numpy as np, torch
import torch.nn as nn, torch.nn.functional as F
from collections import deque

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NODE_FEAT_DIM = 8
EMB_DIM, HEADS, HIDDEN = 32, 4, 128


# ---------------- trainable GAT (the original code's GAT had NO backward pass) ----------
class GATLayer(nn.Module):
    def __init__(self, fin, fout, heads):
        super().__init__()
        assert fout % heads == 0
        self.h, self.d = heads, fout // heads
        self.W = nn.Linear(fin, fout, bias=False)
        self.a_src = nn.Parameter(torch.empty(heads, self.d)); nn.init.xavier_uniform_(self.a_src)
        self.a_dst = nn.Parameter(torch.empty(heads, self.d)); nn.init.xavier_uniform_(self.a_dst)

    def forward(self, x, adj):
        B, N, _ = x.shape
        h = self.W(x).view(B, N, self.h, self.d)
        es = (h * self.a_src).sum(-1)                       # (B,N,H)
        ed = (h * self.a_dst).sum(-1)
        e = F.leaky_relu(es.unsqueeze(2) + ed.unsqueeze(1), 0.2)   # (B,N,N,H)
        mask = adj.view(1, N, N, 1) > 0
        e = e.masked_fill(~mask, float("-inf"))
        alpha = torch.softmax(e, dim=2)
        out = torch.einsum("bijh,bjhd->bihd", alpha, h)
        return out.reshape(B, N, self.h * self.d)


class GraphEncoder(nn.Module):
    def __init__(self, fin=NODE_FEAT_DIM, emb=EMB_DIM, heads=HEADS):
        super().__init__()
        self.g1 = GATLayer(fin, emb, heads)
        self.g2 = GATLayer(emb, emb, heads)

    def forward(self, node_raw, adj):
        h = F.elu(self.g1(node_raw, adj))
        return F.elu(self.g2(h, adj))


class QNet(nn.Module):
    # Scores EACH candidate node separately (the original mean-pooled the graph,
    # destroying the per-node information needed to choose a node).
    def __init__(self, use_graph=True):
        super().__init__()
        self.use_graph = use_graph
        self.enc = GraphEncoder() if use_graph else None
        din = TASK_FEAT_DIM + NODE_FEAT_DIM + (EMB_DIM if use_graph else 0)
        self.mlp = nn.Sequential(nn.Linear(din, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, 1))

    def forward(self, task, node_raw, adj):
        B, N, _ = node_raw.shape
        parts = [task.unsqueeze(1).expand(B, N, TASK_FEAT_DIM), node_raw]
        if self.use_graph:
            parts.append(self.enc(node_raw, adj))
        return self.mlp(torch.cat(parts, -1)).squeeze(-1)          # (B,N)


class FlatQNet(nn.Module):
    # Literature baseline "Standard DQN": task features only -> 19 Q-values, no node state.
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(nn.Linear(TASK_FEAT_DIM, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, HIDDEN), nn.ReLU(),
                                 nn.Linear(HIDDEN, N_NODES))

    def forward(self, task, node_raw, adj):
        return self.mlp(task)


# ---------------- LSH policy cache (Objective 1) ----------------
class Replay:
    def __init__(self, cap=20000):
        self.b = deque(maxlen=cap)
    def push(self, *x): self.b.append(x)
    def __len__(self): return len(self.b)
    def sample(self, n):
        batch = random.sample(self.b, min(n, len(self.b)))
        t, nr, a, r, t2, nr2, d = zip(*batch)
        T = lambda z, dt=torch.float32: torch.tensor(np.array(z), dtype=dt, device=DEV)
        return (T(t), T(nr), torch.tensor(a, dtype=torch.long, device=DEV),
                T(r), T(t2), T(nr2), T(d))


class Agent:
    def __init__(self, adj, use_graph=True, use_lsh=True, flat=False, seed=0, lr=3e-4,
                 use_hpfoo=False, topk=6, hpfoo_schedule=None, refiner="hpfoo"):
        torch.manual_seed(seed)
        self.flat = flat
        self.q  = (FlatQNet() if flat else QNet(use_graph)).to(DEV)
        self.qt = (FlatQNet() if flat else QNet(use_graph)).to(DEV)
        self.qt.load_state_dict(self.q.state_dict())
        self.opt = torch.optim.Adam(self.q.parameters(), lr=lr)
        self.adj = torch.tensor(adj, dtype=torch.float32, device=DEV)
        self.lsh = LSHCache(seed=seed) if use_lsh else None
        # V5: the refinement stage is pluggable (HPFOO, GA, PSO, GWO - see §5b). The
        # attribute keeps its V4 name "hpfoo" so the rest of the pipeline is unchanged.
        self.refiner = refiner if use_hpfoo else None
        self.hpfoo = (REFINERS[refiner](seed=seed, schedule=hpfoo_schedule)
                      if use_hpfoo else None)
        self.topk = topk; self.n_cand = []
        self.mem = Replay(); self.eps = 1.0; self.steps = 0
        self.gamma = 0.95; self.tau_update = 200; self.learn_every = 2; self.batch = 128

    def q_values(self, tf, node_raw):
        with torch.no_grad():
            t = torch.tensor(tf, dtype=torch.float32, device=DEV).unsqueeze(0)
            n = torch.tensor(node_raw, dtype=torch.float32, device=DEV).unsqueeze(0)
            return self.q(t, n, self.adj)[0].cpu().numpy()

    def candidates(self, tf, env, q):
        # LSH prunes the search space the optimiser must evaluate (proposal 8.2).
        cand = np.argsort(-q)[:self.topk]
        if self.lsh is not None:
            a, conf = self.lsh.lookup(tf, env)
            if a is not None and a not in cand: cand = np.append(cand, a)
        self.n_cand.append(len(cand))
        return cand

    def act(self, tf, node_raw, env, task=None, greedy=False):
        if not greedy and np.random.random() < self.eps:
            if self.lsh is not None:
                a, conf = self.lsh.lookup(tf, env)
                if a is not None and np.random.random() < 0.7:
                    return int(a), True
            return int(np.random.randint(N_NODES)), False
        q = self.q_values(tf, node_raw)
        if self.hpfoo is None or not self.hpfoo.fitted or task is None:
            return int(np.argmax(q)), False
        cand = self.candidates(tf, env, q)
        F = self.hpfoo.features(env, task, cand)
        U = self.hpfoo.utility(F); Q = q[cand]
        nz = lambda v: (v - v.min())/(np.ptp(v) + 1e-9)
        al = self.hpfoo.alpha()
        return int(cand[int(np.argmax(al*nz(Q) + (1-al)*nz(U)))]), False

    def learn(self, batch=64):
        if len(self.mem) < 500: return None
        t, nr, a, r, t2, nr2, d = self.mem.sample(batch)
        q = self.q(t, nr, self.adj).gather(1, a.unsqueeze(1)).squeeze(1)
        with torch.no_grad():
            a2 = self.q(t2, nr2, self.adj).argmax(1, keepdim=True)      # Double DQN
            q2 = self.qt(t2, nr2, self.adj).gather(1, a2).squeeze(1)
            y = r + self.gamma * q2 * (1 - d)
        loss = F.smooth_l1_loss(q, y)
        self.opt.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(self.q.parameters(), 5.0); self.opt.step()
        self.steps += 1
        if self.steps % self.tau_update == 0: self.qt.load_state_dict(self.q.state_dict())
        return float(loss)


def evaluate(agent, env, stream):
    # Held-out evaluation: frozen weights, epsilon = 0. The original code reported
    # training-phase averages (including random exploration) as final performance.
    env.reset(); R=[];L=[];E=[];OK=[];A=[];DT=[]
    for task in stream:
        env.advance_to(task["arrival_ms"])
        tf = task_features(task, env); nr = env.node_features()
        t0 = time.perf_counter(); a,_ = agent.act(tf, nr, env, task=task, greedy=True)
        DT.append((time.perf_counter()-t0)*1000)
        r,lat,e,ok = env.step(task, a)
        R.append(r);L.append(lat);E.append(e);OK.append(ok);A.append(a)
    out = summarize(stream, env, R, L, E, OK, A, DT)     # V5: adds cost + fairness measures
    out["actions"] = A
    return out


def train(agent, env, train_stream, val_env, val_stream, episodes, ep_len,
          eval_every=5, verbose=True, tag="", early_stop=True):
    # Model selection uses the VALIDATION stream only. The test stream is never
    # touched during training, so checkpoint selection cannot leak into the result.
    curve = []
    best_r, best_q, best_gb = -1e9, None, None
    total = episodes * ep_len
    step_i = 0
    for ep in range(episodes):
        s = (ep * ep_len) % max(1, len(train_stream) - ep_len)
        chunk = train_stream[s:s+ep_len]
        env.reset()
        prev = None
        for i, task in enumerate(chunk):
            env.advance_to(task["arrival_ms"])
            tf = task_features(task, env); nr = env.node_features()
            a, from_cache = agent.act(tf, nr, env, task=task)
            r, lat, e, ok = env.step(task, a)
            nr2 = env.node_features()
            nxt = chunk[i+1] if i+1 < len(chunk) else task
            tf2 = task_features(nxt, env)
            done = 1.0 if i == len(chunk)-1 else 0.0
            agent.mem.push(tf, nr, a, r, tf2, nr2, done)     # next state = NEXT task
            if agent.lsh is not None:
                agent.lsh.update(tf, env, a, r, allow_write=(agent.eps < 0.85))
            if agent.hpfoo is not None:
                agent.hpfoo.observe(agent.hpfoo.features(env, task, np.array([a]))[0], r)
                if step_i % 250 == 0: agent.hpfoo.optimise()
            if step_i % agent.learn_every == 0: agent.learn(agent.batch)
            step_i += 1
            agent.eps = max(0.05, 1.0 - step_i / (0.6 * total))
        if (ep+1) % eval_every == 0 or ep == episodes-1:
            m = evaluate(agent, val_env, val_stream)
            curve.append((ep+1, m["reward"], m["miss"]))
            if m["reward"] > best_r:
                best_r = m["reward"]
                best_q = copy.deepcopy(agent.q.state_dict())
                best_gb = None if agent.hpfoo is None else agent.hpfoo.gb.copy()
            if verbose:
                print("    [%s] ep %3d/%d  val reward %.4f  miss %5.2f%%  eps %.2f"
                      % (tag, ep+1, episodes, m["reward"], m["miss"]*100, agent.eps))
    if early_stop and best_q is not None:
        agent.q.load_state_dict(best_q)
        if best_gb is not None: agent.hpfoo.gb = best_gb
        if verbose: print("    [%s] restored best checkpoint (val reward %.4f)" % (tag, best_r))
    agent.best_val = best_r
    return curve


# ================= HPFOO: Hybrid Pufferfish-Osprey Optimisation =========
# Proposal Objective 3. Learns a linear multi-objective utility over OBSERVABLE node
# features from REALISED rewards (no simulator/oracle access), and fuses it with the
# GRL Q-values. LSH prunes the candidate set the optimiser searches (proposal 8.2).
HPFOO_DIM = 7          # 6 feature weights + 1 fusion coefficient alpha

class HPFOO:
    # Hybrid Pufferfish-Osprey Optimisation (HPFOO).
    #   Phase 1 - Osprey exploration      : position identification & hunting (OOA)
    #   Phase 2 - Pufferfish exploitation : defensive-inflation local search (POA)
    # Both phases use greedy selection: a candidate replaces the incumbent only if
    # its fitness improves, so the population fitness is monotonically non-decreasing.
    # Fitness ranks the six utility weights against realised rewards; it cannot score
    # the fusion coefficient alpha (index 6), which is selected on validation instead.
    def __init__(self, n_particles=14, seed=0, iters=6, lb=-3.0, ub=3.0,
                 schedule=None):
        schedule = PUFFERFISH_SCHEDULE if schedule is None else schedule
        rng = np.random.RandomState(seed); self.rng = rng
        self.lb, self.ub = float(lb), float(ub)
        self.X = rng.uniform(lb, ub, (n_particles, HPFOO_DIM))
        self.F = np.full(n_particles, -1e9)
        self.gb = self.X[0].copy(); self.gbf = -1e9
        self.iters = iters; self.t = 1
        self.buf_F, self.buf_r = [], []
        self._alpha_override = None
        self.fitted = False
        # ---- Requirement 2: step-size schedule state ----
        self.schedule = schedule
        self.n_calls = 0           # how many times optimise() has been invoked
        self.radius_log = []       # every radius actually applied, for reporting
        # ---- V5: overhead accounting (identical for every refiner) ----
        self.opt_time = 0.0        # wall-clock seconds spent inside optimise()
        self.n_evals = 0           # number of candidate fitness evaluations

    def step_radius(self, tau):
        """Pufferfish defensive-inflation radius for local iteration `tau`.

        legacy   : (ub-lb)/t,  t monotonic over the entire training run.
        reset    : (ub-lb)/tau, tau = 1..iters, fresh contraction each invocation.
        annealed : (ub-lb)/tau * 1/sqrt(1+k), k = invocation index. Coarse-to-fine
                   within an invocation, plus a global anneal across the run so the
                   population is not re-scattered late in training.
        """
        span = self.ub - self.lb
        if self.schedule == "legacy":
            rad = span / self.t
        elif self.schedule == "reset":
            rad = span / tau
        else:
            rad = span / tau / math.sqrt(1.0 + self.n_calls)
        self.radius_log.append(float(rad))
        return rad

    @staticmethod
    def features(env, task, cand):
        cap = env.cap[cand]
        q    = env.backlog[cand]/cap*1000.0
        comp = task["cpu_cycles"]/cap*1000.0
        prop = env.prop[cand]
        price = _PC[cand]*task["cpu_cycles"] + _PB[cand]*task["data_size_bytes"]*8.0
        D = max(task["deadline_ms"], 1e-6)
        est = q + comp + prop
        return np.stack([np.clip(q/100,0,5), np.clip(comp/100,0,5), prop/70.0,
                         price/(price.max()+1e-12), np.clip(est/D,0,3)/3.0,
                         (TIER[cand] == LOCAL).astype(float)], axis=1)

    def utility(self, F, w=None):
        w = self.gb if w is None else w
        return F @ w[:6]

    def alpha(self):
        if self._alpha_override is not None: return float(self._alpha_override)
        return float(1.0/(1.0+np.exp(-self.gb[6])))

    def set_alpha(self, a):
        self._alpha_override = None if a is None else float(np.clip(a, 0.0, 1.0))

    def observe(self, f_row, r):
        self.buf_F.append(f_row); self.buf_r.append(r)
        if len(self.buf_F) > 2000: self.buf_F.pop(0); self.buf_r.pop(0)

    def _fitness(self, W):
        # Pearson correlation between predicted utility and realised reward
        F = np.asarray(self.buf_F); r = np.asarray(self.buf_r)
        U = F @ W[:, :6].T
        Uc = U - U.mean(0); rc = r - r.mean()
        d = (np.linalg.norm(Uc, axis=0)*np.linalg.norm(rc) + 1e-9)
        self.n_evals += len(W)
        return (Uc*rc[:, None]).sum(0)/d

    def _update_gb(self, X, F):
        # shared global-best bookkeeping (same rule as the inline HPFOO code below)
        j = int(np.argmax(F))
        if F[j] > self.gbf:
            self.gbf = float(F[j]); self.gb = X[j].copy()

    def optimise(self):
        # V5: the search itself lives in _search() so GA/PSO/GWO can replace it with the
        # same trigger, window, fitness, bounds and evaluation budget.
        if len(self.buf_F) < 200: return
        t0 = time.perf_counter()
        self._search()
        self.opt_time += time.perf_counter() - t0
        self.n_calls += 1
        self.fitted = True

    def _search(self):
        # Budget per invocation: 1 initial + iters x (osprey + pufferfish) population
        # evaluations = (1 + 2*iters) * n_particles fitness evaluations.
        n = len(self.X)
        F = self._fitness(self.X)
        for _tau in range(1, self.iters + 1):
            self.t += 1

            # ---- Phase 1: Osprey exploration ----
            # x_new = x + r * (SF - I*x),  SF drawn from members with better fitness,
            # r ~ U(0,1),  I in {1,2}
            Xn = self.X.copy()
            for i in range(n):
                better = np.where(F > F[i])[0]
                SF = self.gb if len(better) == 0 else self.X[self.rng.choice(better)]
                r = self.rng.random(HPFOO_DIM)
                I = self.rng.randint(1, 3, HPFOO_DIM)
                Xn[i] = self.X[i] + r*(SF - I*self.X[i])
            Xn = np.clip(Xn, self.lb, self.ub)
            Fn = self._fitness(Xn)
            imp = Fn > F
            self.X[imp] = Xn[imp]; F[imp] = Fn[imp]

            # ---- Phase 2: Pufferfish exploitation ----
            # x_new = x + (1 - 2r) * R(tau, k)   : R is the scheduled radius, see
            # HPFOO.step_radius. The neighbourhood contracts as tau grows within an
            # invocation and as k grows across the training run.
            r = self.rng.random((n, HPFOO_DIM))
            Xn = np.clip(self.X + (1.0 - 2.0*r)*self.step_radius(_tau),
                         self.lb, self.ub)
            Fn = self._fitness(Xn)
            imp = Fn > F
            self.X[imp] = Xn[imp]; F[imp] = Fn[imp]

            j = int(np.argmax(F))
            if F[j] > self.gbf:
                self.gbf = float(F[j]); self.gb = self.X[j].copy()
        self.F = F

### §5b — Alternative refiners: GA, PSO, GWO (reviewer item 03)

Each one replaces **only** HPFOO's search step. Everything else is identical: the same six-weight utility, the
same fitness (correlation with the rewards received over the last 2,000 decisions), the same bounds $[-3, 3]$,
the same population of 14, the same trigger (every 250 training steps), the same Q-ranked shortlist with the
LSH-retrieved candidate, and the same validation-selected fusion weight $\alpha$.

**Equal budget.** HPFOO evaluates $(1 + 2\cdot 6)\times 14 = 182$ candidates per invocation (initial population,
then an Osprey and a Pufferfish move in each of 6 iterations). GA, PSO and GWO get the same 182 evaluations per
invocation: the initial population plus 12 generations. The notebook asserts this budget equality in §15.

| Refiner | Update rule (standard settings) |
|---|---|
| GA | elitism (1), binary tournament selection, arithmetic crossover $p_c = 0.9$, Gaussian mutation $\sigma = 0.1\,(ub-lb)$ with per-gene rate $1/7$ |
| PSO | inertia $w = 0.729$, $c_1 = c_2 = 1.494$, velocity clamp $0.2\,(ub-lb)$ |
| GWO | $\alpha$, $\beta$, $\delta$ leaders, $a$ decreasing linearly from 2 to 0 within each invocation |

In [12]:
# ===== §5b: GA / PSO / GWO refiners — drop-in replacements for HPFOO's search step =====
class GARefiner(HPFOO):
    """Real-coded genetic algorithm with the same budget as HPFOO."""
    def __init__(self, n_particles=14, seed=0, iters=6, lb=-3.0, ub=3.0, schedule=None,
                 pc=0.9, pm=None, sigma=0.1):
        super().__init__(n_particles=n_particles, seed=seed, iters=iters, lb=lb, ub=ub,
                         schedule=schedule)
        self.pc = pc; self.pm = (1.0/HPFOO_DIM) if pm is None else pm
        self.sigma = sigma*(self.ub - self.lb)

    def _search(self):
        n = len(self.X); F = self._fitness(self.X)
        for _g in range(2*self.iters):
            def tournament():
                i, j = self.rng.randint(0, n, 2)
                return i if F[i] >= F[j] else j
            kids = [self.X[int(np.argmax(F))].copy()]                 # elitism
            while len(kids) < n:
                p1, p2 = self.X[tournament()], self.X[tournament()]
                if self.rng.random() < self.pc:
                    b = self.rng.random(HPFOO_DIM)                    # arithmetic crossover
                    c1, c2 = b*p1 + (1-b)*p2, b*p2 + (1-b)*p1
                else:
                    c1, c2 = p1.copy(), p2.copy()
                for c in (c1, c2):
                    mask = self.rng.random(HPFOO_DIM) < self.pm         # Gaussian mutation
                    c[mask] += self.rng.normal(0.0, self.sigma, int(mask.sum()))
                    kids.append(np.clip(c, self.lb, self.ub))
            self.X = np.array(kids[:n]); F = self._fitness(self.X)
            self._update_gb(self.X, F)
        self.F = F


class PSORefiner(HPFOO):
    """Global-best particle swarm (constriction-equivalent constants)."""
    def __init__(self, n_particles=14, seed=0, iters=6, lb=-3.0, ub=3.0, schedule=None,
                 w=0.729, c1=1.49445, c2=1.49445, vmax=0.2):
        super().__init__(n_particles=n_particles, seed=seed, iters=iters, lb=lb, ub=ub,
                         schedule=schedule)
        self.w, self.c1, self.c2 = w, c1, c2
        self.vmax = vmax*(self.ub - self.lb)
        self.V = np.zeros_like(self.X)

    def _search(self):
        F = self._fitness(self.X)
        pb, pbf = self.X.copy(), F.copy()          # personal bests re-scored on the current window
        for _g in range(2*self.iters):
            g = int(np.argmax(pbf))
            r1 = self.rng.random(self.X.shape); r2 = self.rng.random(self.X.shape)
            self.V = (self.w*self.V + self.c1*r1*(pb - self.X) + self.c2*r2*(pb[g] - self.X))
            self.V = np.clip(self.V, -self.vmax, self.vmax)
            self.X = np.clip(self.X + self.V, self.lb, self.ub)
            F = self._fitness(self.X)
            imp = F > pbf
            pb[imp] = self.X[imp]; pbf[imp] = F[imp]
            self._update_gb(pb, pbf)
        self.F = pbf


class GWORefiner(HPFOO):
    """Grey wolf optimiser (alpha, beta, delta leaders; a: 2 -> 0 within an invocation)."""
    def _search(self):
        F = self._fitness(self.X); G = 2*self.iters
        for it in range(G):
            order = np.argsort(-F)
            leaders = [self.X[order[0]].copy(), self.X[order[1]].copy(), self.X[order[2]].copy()]
            a = 2.0 - 2.0*it/G
            Xn = np.zeros_like(self.X)
            for Lp in leaders:
                r1 = self.rng.random(self.X.shape); r2 = self.rng.random(self.X.shape)
                A = 2.0*a*r1 - a; C = 2.0*r2
                Xn += Lp - A*np.abs(C*Lp - self.X)
            self.X = np.clip(Xn/3.0, self.lb, self.ub)
            F = self._fitness(self.X)
            self._update_gb(self.X, F)
        self.F = F


REFINERS = {"hpfoo": HPFOO, "ga": GARefiner, "pso": PSORefiner, "gwo": GWORefiner}

# Budget check on a synthetic fitness window: every refiner must spend exactly the same
# number of fitness evaluations per invocation.
_bud = {}
for _nm, _cls in REFINERS.items():
    _r = _cls(seed=0)
    _rng = np.random.RandomState(0)
    for _ in range(300): _r.observe(_rng.random(6), _rng.random())
    _r.optimise(); _bud[_nm] = _r.n_evals
print("fitness evaluations per invocation:", _bud)
assert len(set(_bud.values())) == 1, "refiners must share the same evaluation budget"

fitness evaluations per invocation: {'hpfoo': 182, 'ga': 182, 'pso': 182, 'gwo': 182}


### §5c — GAT-PPO graph-RL baseline (reviewer item 03)

A representative modern graph-based RL offloader, in the style of GAPO (graph attention + proximal policy
optimisation; Zhao et al., *Sensors* 2025, ref. [29]). It is **re-implemented in this environment**, not the
authors' code. It uses the same graph-attention encoder, the same task and node features, the same reward,
the same number of training decisions (episodes × episode length), and the same validation-checkpoint protocol
as the proposed method. It has no LSH memory and no metaheuristic refiner.

Policy: per-node logits from a shared MLP on [task, node, graph-embedding] features; value: node-averaged MLP.
Training: one rollout per episode, GAE (γ = 0.95, λ = 0.95), 4 PPO epochs, minibatch 64, clip 0.2,
entropy coefficient 0.01, value coefficient 0.5, Adam with learning rate 3×10⁻⁴. Evaluation uses argmax actions.

In [13]:
# ===== §5c: GAT-PPO baseline (GAPO-style graph reinforcement learning) =====
class PPONet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = GraphEncoder()
        din = TASK_FEAT_DIM + NODE_FEAT_DIM + EMB_DIM
        self.pi = nn.Sequential(nn.Linear(din, HIDDEN), nn.Tanh(),
                                nn.Linear(HIDDEN, HIDDEN), nn.Tanh(), nn.Linear(HIDDEN, 1))
        self.v  = nn.Sequential(nn.Linear(din, HIDDEN), nn.Tanh(), nn.Linear(HIDDEN, 1))

    def forward(self, task, node_raw, adj):
        B, N, _ = node_raw.shape
        x = torch.cat([task.unsqueeze(1).expand(B, N, TASK_FEAT_DIM), node_raw,
                       self.enc(node_raw, adj)], -1)
        return self.pi(x).squeeze(-1), self.v(x).squeeze(-1).mean(1)     # logits (B,N), value (B,)


class PPOAgent:
    def __init__(self, adj, seed=0, lr=3e-4, clip=0.2, epochs=4, minibatch=64,
                 gamma=0.95, lam=0.95, ent_coef=0.01, vf_coef=0.5):
        torch.manual_seed(seed)
        self.net = PPONet().to(DEV)
        self.q = self.net                       # alias: checkpointing / parameter counting
        self.opt = torch.optim.Adam(self.net.parameters(), lr=lr)
        self.adj = torch.tensor(adj, dtype=torch.float32, device=DEV)
        self.clip, self.epochs, self.minibatch = clip, epochs, minibatch
        self.gamma, self.lam, self.ent_coef, self.vf_coef = gamma, lam, ent_coef, vf_coef
        self.hpfoo = None; self.lsh = None; self.n_cand = []; self.eps = 0.0
        self._last = (0.0, 0.0)

    def _forward(self, tf, node_raw):
        t = torch.tensor(tf, dtype=torch.float32, device=DEV).unsqueeze(0)
        n = torch.tensor(node_raw, dtype=torch.float32, device=DEV).unsqueeze(0)
        return self.net(t, n, self.adj)

    def act(self, tf, node_raw, env, task=None, greedy=False):
        with torch.no_grad():
            logits, v = self._forward(tf, node_raw)
        if greedy:
            return int(torch.argmax(logits[0]).item()), False
        dist = torch.distributions.Categorical(logits=logits[0])
        a = dist.sample()
        self._last = (float(dist.log_prob(a).item()), float(v[0].item()))
        return int(a.item()), False

    def update(self, buf):
        T = len(buf["r"])
        r = np.asarray(buf["r"], np.float32); v = np.asarray(buf["v"], np.float32)
        adv = np.zeros(T, np.float32); last = 0.0
        for t in reversed(range(T)):                     # GAE; episode ends at T-1
            nv = 0.0 if t == T-1 else v[t+1]
            nonterm = 0.0 if t == T-1 else 1.0
            delta = r[t] + self.gamma*nv*nonterm - v[t]
            last = delta + self.gamma*self.lam*nonterm*last
            adv[t] = last
        ret = adv + v
        adv = (adv - adv.mean())/(adv.std() + 1e-8)
        tt = lambda x, dt=torch.float32: torch.tensor(np.asarray(x), dtype=dt, device=DEV)
        TF, NR = tt(buf["tf"]), tt(buf["nr"])
        A = tt(buf["a"], torch.long); OLD = tt(buf["logp"])
        ADV, RET = tt(adv), tt(ret)
        idx = np.arange(T)
        for _ in range(self.epochs):
            np.random.shuffle(idx)
            for s in range(0, T, self.minibatch):
                b = torch.tensor(idx[s:s+self.minibatch], dtype=torch.long, device=DEV)
                logits, val = self.net(TF[b], NR[b], self.adj)
                dist = torch.distributions.Categorical(logits=logits)
                lp = dist.log_prob(A[b])
                ratio = torch.exp(lp - OLD[b])
                s1 = ratio*ADV[b]
                s2 = torch.clamp(ratio, 1.0-self.clip, 1.0+self.clip)*ADV[b]
                loss = (-torch.min(s1, s2).mean() + self.vf_coef*F.mse_loss(val, RET[b])
                        - self.ent_coef*dist.entropy().mean())
                self.opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(self.net.parameters(), 0.5); self.opt.step()


def train_ppo(agent, env, train_stream, val_env, val_stream, episodes, ep_len,
              eval_every=5, verbose=True, tag="gat_ppo"):
    # Same data chunks, same number of decisions and same validation-checkpoint rule as train().
    curve = []; best_r, best_state = -1e9, None
    for ep in range(episodes):
        s = (ep * ep_len) % max(1, len(train_stream) - ep_len)
        chunk = train_stream[s:s+ep_len]
        env.reset()
        buf = {k: [] for k in ["tf", "nr", "a", "logp", "v", "r"]}
        for task in chunk:
            env.advance_to(task["arrival_ms"])
            tf = task_features(task, env); nr = env.node_features()
            a, _ = agent.act(tf, nr, env, task=task)
            r, lat, e, ok = env.step(task, a)
            lp, v = agent._last
            for k, x in zip(["tf", "nr", "a", "logp", "v", "r"], [tf, nr, a, lp, v, r]):
                buf[k].append(x)
        agent.update(buf)
        if (ep+1) % eval_every == 0 or ep == episodes-1:
            m = evaluate(agent, val_env, val_stream)
            curve.append((ep+1, m["reward"], m["miss"]))
            if m["reward"] > best_r:
                best_r = m["reward"]; best_state = copy.deepcopy(agent.net.state_dict())
            if verbose:
                print("    [%s] ep %3d/%d  val reward %.4f  miss %5.2f%%"
                      % (tag, ep+1, episodes, m["reward"], m["miss"]*100))
    if best_state is not None:
        agent.net.load_state_dict(best_state)
        if verbose: print("    [%s] restored best checkpoint (val reward %.4f)" % (tag, best_r))
    agent.best_val = best_r
    return curve

## §6–§7 — Training and held-out evaluation · **Objective 4**

Three learned variants (proposed, and two ablations) plus the reference policies from §3.

In [14]:
SPEC = {"lsh_grl_hpfoo": dict(use_graph=True,  use_lsh=True,  use_hpfoo=True),
        "lsh_grl":       dict(use_graph=True,  use_lsh=True,  use_hpfoo=False),
        "grl":           dict(use_graph=True,  use_lsh=False, use_hpfoo=False),
        "no_graph":      dict(use_graph=False, use_lsh=False, use_hpfoo=False),
        "dqn":           dict(flat=True,       use_lsh=False, use_hpfoo=False),
        # V5: alternative refiners in the identical pipeline (§15)
        "lsh_grl_ga":    dict(use_graph=True,  use_lsh=True,  use_hpfoo=True, refiner="ga"),
        "lsh_grl_pso":   dict(use_graph=True,  use_lsh=True,  use_hpfoo=True, refiner="pso"),
        "lsh_grl_gwo":   dict(use_graph=True,  use_lsh=True,  use_hpfoo=True, refiner="gwo")}

def build_agent(kind, seed, topk=6, hpfoo_schedule=None, adj=None, fb=None):
    # V4 always passed env.adj (the seed-0 graph). V5 passes the graph of the seed's own
    # environment unless ADJ_FROM_SEED_ENV is False.
    ag = Agent(env.adj if adj is None else adj, seed=seed, topk=topk,
               hpfoo_schedule=hpfoo_schedule, **SPEC[kind])
    if ag.lsh is not None: ag.lsh.fit(feat_bank if fb is None else fb)
    return ag

ALPHA_GRID = [0.0, 0.25, 0.5, 0.75, 1.0]

def train_one(kind, seed, verbose=True, topk=6, tune_alpha=True,
              hpfoo_schedule=None, data=None):
    # `data` (V5) lets §16/§17 train on other streams; default = the main split.
    D = data or dict(train=train_stream, val=val_stream, test=test_stream,
                     dls=dl_scale, fb=feat_bank, sites=sites)
    set_seed(seed)
    e  = OffloadingEnv(seed=seed, sites=D["sites"], deadline_scale=D["dls"])
    ve = OffloadingEnv(seed=seed, sites=D["sites"], deadline_scale=D["dls"])
    te = OffloadingEnv(seed=seed, sites=D["sites"], deadline_scale=D["dls"])
    adj = e.adj if (ADJ_FROM_SEED_ENV or e.adj.shape != env.adj.shape) else env.adj
    t0 = time.perf_counter()
    if kind == "gat_ppo":
        ag = PPOAgent(adj, seed=seed)
        curve = train_ppo(ag, e, D["train"], ve, D["val"], P["EPISODES"], P["EP_LEN"],
                          eval_every=P["EVAL_EVERY"], verbose=verbose, tag=kind)
    else:
        ag = build_agent(kind, seed, topk, hpfoo_schedule, adj=adj, fb=D["fb"])
        curve = train(ag, e, D["train"], ve, D["val"], P["EPISODES"], P["EP_LEN"],
                      eval_every=P["EVAL_EVERY"], verbose=verbose, tag=kind)
    train_s = time.perf_counter() - t0
    # Fusion weight alpha is selected on VALIDATION. The metaheuristic cannot score it
    # (its fitness only ranks the six utility weights), so it is chosen here instead.
    # alpha = 1.0 (pure Q-network) is in the grid, so this can never select a policy
    # worse than the no-HPFOO ablation on validation.
    if tune_alpha and ag.hpfoo is not None and ag.hpfoo.fitted:
        best = (-1e9, None)
        for al in ALPHA_GRID:
            ag.hpfoo.set_alpha(al)
            r = evaluate(ag, ve, D["val"])["reward"]
            if r > best[0]: best = (r, al)
        ag.hpfoo.set_alpha(best[1]); ag.alpha_selected = best[1]
    m = evaluate(ag, te, D["test"])
    m["candidates"] = float(np.mean(ag.n_cand)) if ag.n_cand else float(N_NODES)
    m["alpha"] = getattr(ag, "alpha_selected", None)
    m["train_s"] = train_s
    m["params"] = int(sum(p.numel() for p in ag.q.parameters()))
    if ag.hpfoo is not None:
        m["opt_time_s"] = ag.hpfoo.opt_time; m["opt_calls"] = ag.hpfoo.n_calls
        m["fitness_evals"] = ag.hpfoo.n_evals
    if ag.lsh is not None:
        m.update({"lsh_" + k: v for k, v in ag.lsh.stats().items()})
    return ag, curve, m

### §7a — Sensitivity analyses (run BEFORE the main comparison)

`topk` (how many candidates HPFOO evaluates) and `alpha` (Q-network / HPFOO-utility fusion)
are both selected on the **validation** stream. Test figures are shown for transparency and
play no part in selection. Because `alpha = 1.0` (pure Q-network) and `topk = 19` (no pruning)
are both in the grids, this can never select a configuration worse than the ablated variants
on validation.

The HPFOO fitness ranks only the six utility weights, so it cannot score `alpha`; selecting
it here is the correct treatment.

In [15]:
# ---- topk sweep: selection on VALIDATION ----
TOPK_GRID = [3, 6, 10, N_NODES]
tk_rows = []
for tk in TOPK_GRID:
    _ag, _, m = train_one('lsh_grl_hpfoo', 0, verbose=False, topk=tk)
    _ve = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale)
    v = evaluate(_ag, _ve, val_stream)['reward']
    tk_rows.append(dict(topk=tk, val_reward=v, reward=m['reward'], latency=m['latency'],
                        miss_pct=m['miss']*100, candidates=m['candidates'],
                        reduction_pct=100*(1-m['candidates']/N_NODES),
                        decision_ms=m['decision_ms'], alpha=m['alpha']))
    print('  topk=%-3d val %.4f  test %.4f  miss %5.2f%%  cand %.1f'
          % (tk, v, m['reward'], m['miss']*100, m['candidates']))
TOPK = pd.DataFrame(tk_rows); display(TOPK.round(4))
TOPK_BEST = int(TOPK.loc[TOPK.val_reward.idxmax(), 'topk'])
print('\ntopk selected on validation : %d  (test reward at that setting: %.4f)'
      % (TOPK_BEST, float(TOPK.loc[TOPK.val_reward.idxmax(), 'reward'])))
RESULTS['sensitivity_topk'] = TOPK.to_dict('records'); RESULTS['topk_selected'] = TOPK_BEST

  topk=3   val 0.8741  test 0.8757  miss  6.90%  cand 3.5
  topk=6   val 0.8715  test 0.8738  miss  6.90%  cand 6.3
  topk=10  val 0.8707  test 0.8741  miss  7.40%  cand 10.2
  topk=19  val 0.8686  test 0.8728  miss  7.70%  cand 19.0


,topk,val_reward,reward,latency,miss_pct,candidates,reduction_pct,decision_ms,alpha
0,3,0.8741,0.8757,111.0212,6.9,3.4769,81.7006,2.2926,0.25
1,6,0.8715,0.8738,104.3138,6.9,6.2934,66.8766,2.2716,0.25
2,10,0.8707,0.8741,109.9184,7.4,10.1881,46.3785,2.2993,0.50
3,19,0.8686,0.8728,116.0890,7.7,19.0000,0.0000,2.2906,0.50



topk selected on validation : 3  (test reward at that setting: 0.8757)


In [16]:
# ---- alpha sweep at the selected topk (no retraining per alpha) ----
set_seed(0)
_ag, _, _ = train_one('lsh_grl_hpfoo', 0, verbose=False, topk=TOPK_BEST, tune_alpha=False)
_ve = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale)
_te = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale)
al_rows = []
for al in ALPHA_GRID:
    _ag.hpfoo.set_alpha(al)
    v = evaluate(_ag, _ve, val_stream); t = evaluate(_ag, _te, test_stream)
    al_rows.append(dict(alpha=al, val_reward=v['reward'], test_reward=t['reward'],
                        test_latency=t['latency'], test_miss_pct=t['miss']*100))
ALPHA = pd.DataFrame(al_rows)
_best_al = float(ALPHA.loc[ALPHA.val_reward.idxmax(), 'alpha'])
display(ALPHA.round(4))
print('\nalpha selected on validation : %.2f   (metaheuristic drift value: %.2f)'
      % (_best_al, 1/(1+np.exp(-_ag.hpfoo.gb[6]))))
RESULTS['sensitivity_alpha'] = ALPHA.to_dict('records')
RESULTS['alpha_selected'] = _best_al

,alpha,val_reward,test_reward,test_latency,test_miss_pct
0,0.00,0.8722,0.8760,112.6069,6.90
1,0.25,0.8741,0.8757,111.0212,6.90
2,0.50,0.8717,0.8727,98.1368,7.45
3,0.75,0.8593,0.8646,82.3237,8.40
4,1.00,0.8571,0.8627,79.7018,8.65



alpha selected on validation : 0.25   (metaheuristic drift value: 0.50)


### §7b — Main comparison at the validation-selected settings

Trained with `topk = TOPK_BEST`; `alpha` is re-selected on validation inside `train_one`.

In [17]:
LEARNED = {}; CURVES = {}
for kind in ["lsh_grl_hpfoo", "lsh_grl", "grl", "dqn", "gat_ppo"]:
    t0 = time.time(); print("training %s ..." % LABELS.get(kind, kind))
    ag, cur, m = train_one(kind, 0, topk=TOPK_BEST)
    LEARNED[kind] = m; CURVES[kind] = cur
    if ag.lsh is not None: RESULTS["lsh_"+kind] = ag.lsh.stats()
    if ag.n_cand: RESULTS["cand_"+kind] = dict(mean_candidates=float(np.mean(ag.n_cand)),
        reduction_pct=100*(1-float(np.mean(ag.n_cand))/N_NODES))
    print("  -> held-out reward %.4f | miss %.2f%% | %.0fs\n" % (m["reward"], m["miss"]*100, time.time()-t0))

training LSH-GRL-HPFOO (Proposed) ...
    [lsh_grl_hpfoo] ep   5/25  val reward 0.8673  miss  9.50%  eps 0.67
    [lsh_grl_hpfoo] ep  10/25  val reward 0.8717  miss  7.67%  eps 0.33
    [lsh_grl_hpfoo] ep  15/25  val reward 0.8693  miss  8.33%  eps 0.05
    [lsh_grl_hpfoo] ep  20/25  val reward 0.8708  miss  7.83%  eps 0.05
    [lsh_grl_hpfoo] ep  25/25  val reward 0.8703  miss  7.75%  eps 0.05
    [lsh_grl_hpfoo] restored best checkpoint (val reward 0.8717)
  -> held-out reward 0.8757 | miss 6.90% | 70s

training LSH-GRL (no HPFOO) ...
    [lsh_grl] ep   5/25  val reward 0.8691  miss  9.00%  eps 0.67
    [lsh_grl] ep  10/25  val reward 0.8561  miss 10.75%  eps 0.33
    [lsh_grl] ep  15/25  val reward 0.8653  miss  9.67%  eps 0.05
    [lsh_grl] ep  20/25  val reward 0.8651  miss  9.25%  eps 0.05
    [lsh_grl] ep  25/25  val reward 0.8621  miss  8.33%  eps 0.05
    [lsh_grl] restored best checkpoint (val reward 0.8691)
  -> held-out reward 0.8699 | miss 9.05% | 44s

training GRL only ..

In [18]:
ALL = {}
_scalar = lambda d: {m: x for m, x in d.items() if isinstance(x, (int, float)) and not isinstance(x, bool)}
for k in ["lsh_grl_hpfoo","lsh_grl","grl","dqn","gat_ppo"]: ALL[k] = _scalar(LEARNED[k])
for k in ["greedy","maxmin","random","local_only","all_cloud","oracle"]: ALL[k] = RESULTS["reference"][k]

MAIN = pd.DataFrame(ALL).T.loc[["lsh_grl_hpfoo","lsh_grl","grl","gat_ppo","dqn","greedy","maxmin",
                                 "random","local_only","all_cloud","oracle"]]
MAIN.index = [LABELS[i] for i in MAIN.index]
MAIN = MAIN[["reward","latency","energy","miss","cost_ratio","offload_rate","decision_ms"]].astype(float)
MAIN.columns = ["Reward","Latency (ms)","Energy (J)","Deadline miss","Cost ratio","Offload rate","Decision (ms)"]
display(MAIN.round(4))

prop, orc = ALL["lsh_grl_hpfoo"]["reward"], ALL["oracle"]["reward"]
print("\nHELD-OUT RESULT (frozen weights, epsilon = 0, unseen tasks)\n" + "="*66)
for k in ["lsh_grl","grl","gat_ppo","dqn","greedy","maxmin","random","local_only","all_cloud"]:
    print("  Proposed vs %-24s : %+7.2f%%" % (LABELS[k], (prop-ALL[k]["reward"])/ALL[k]["reward"]*100))
gap_closed = (prop-ALL["greedy"]["reward"])/(orc-ALL["greedy"]["reward"])*100
print("  oracle headroom closed         : %6.1f%%" % gap_closed)
print("="*66)
RESULTS["main_comparison"] = ALL; RESULTS["gap_closed_pct"] = float(gap_closed)
RESULTS["convergence"] = {k: [[int(a), float(b), float(c)] for a,b,c in v] for k,v in CURVES.items()}

,Reward,Latency (ms),Energy (J),Deadline miss,Cost ratio,Offload rate,Decision (ms)
LSH-GRL-HPFOO (Proposed),0.8757,111.0212,0.0489,0.0690,0.1333,1.0000,2.3134
LSH-GRL (no HPFOO),0.8699,95.0696,0.0425,0.0905,0.1349,1.0000,1.3685
GRL only,0.8727,94.6309,0.0423,0.0785,0.1410,1.0000,1.3552
GAT-PPO (graph RL),0.8389,155.0145,0.1201,0.1535,0.1047,0.9280,1.5048
Standard DQN,0.6888,4524.7338,1.8168,0.3575,0.1768,0.9500,0.3465
Greedy (min-latency),0.8196,55.6673,0.0340,0.0620,0.3213,0.9305,0.0220
Max-min fair,0.8386,72.6325,0.0352,0.0610,0.2678,0.9910,0.0392
Random Offloading,0.7283,204.9696,0.1259,0.1940,0.2994,0.9520,0.0049
Local-Only,0.6365,555.8301,1.1117,0.3840,0.0000,0.0000,0.0002
All-to-Cloud,0.5102,100.4318,0.0446,0.1330,1.0000,1.0000,0.0003



HELD-OUT RESULT (frozen weights, epsilon = 0, unseen tasks)
  Proposed vs LSH-GRL (no HPFOO)       :   +0.66%
  Proposed vs GRL only                 :   +0.35%
  Proposed vs GAT-PPO (graph RL)       :   +4.39%
  Proposed vs Standard DQN             :  +27.13%
  Proposed vs Greedy (min-latency)     :   +6.85%
  Proposed vs Max-min fair             :   +4.42%
  Proposed vs Random Offloading        :  +20.23%
  Proposed vs Local-Only               :  +37.57%
  Proposed vs All-to-Cloud             :  +71.63%
  oracle headroom closed         :   91.0%


## §8–§9 — Unified multi-seed run (replaces V4 §8 ablation, §9 multi-seed, and the single-seed mobility / perturbation runs)

For each of the 10 seeds, one pass:

1. trains **all** learned methods once (proposed, LSH-GRL, GRL only, no-graph, Standard DQN, GAT-PPO) with the
   same validation-selected `topk`;
2. runs every reference policy (greedy, max-min fair, random, local-only, all-to-cloud, myopic oracle);
3. replays the **same trained policies** under 4 mobility levels, 15 perturbed operating conditions and 4
   arrival-time distributions (reviewer item 05);
4. times the LSH lookup against an exhaustive scan of the same memory, and evaluates the proposed method with
   the LSH candidate switched off at inference (reviewer item 04).

V4 trained the same models twice (ablation loop, then multi-seed loop). V5 trains them once, so the extra
experiments cost little extra time. Every seed's record is cached, so a disconnect loses at most one seed.

**Statistics.** Each seed varies capacities, distances, connectivity, initialisation and exploration, while
the job stream is fixed. Seeds are therefore the paired experimental unit, and every comparison uses a paired
t-test over seeds. The tables report the mean difference with its 95% t-interval, Cohen's $d_z$
(mean paired difference / s.d. of paired differences), the number of seeds won, and Holm-corrected $p$ within
each family of comparisons.

In [19]:
# ===== §9 helpers: statistics, per-seed experiment =====
from scipy import stats as sps

def ci95(x):
    x = np.asarray(x, float); n = len(x)
    if n < 2: return float(x.mean()), float("nan"), float("nan")
    h = sps.t.ppf(0.975, n-1)*x.std(ddof=1)/math.sqrt(n)
    return float(x.mean()), float(x.mean()-h), float(x.mean()+h)

def paired(a, b):
    """Paired comparison a - b over seeds."""
    a, b = np.asarray(a, float), np.asarray(b, float); d = a - b; n = len(d)
    m, lo, hi = ci95(d)
    sd = d.std(ddof=1) if n > 1 else 0.0
    if n < 2 or sd == 0:
        t, p = float("nan"), (0.0 if (n > 1 and m != 0) else float("nan"))
    else:
        t, p = sps.ttest_rel(a, b)
    return dict(diff=m, ci_lo=lo, ci_hi=hi, t=float(t), p=float(p),
                d=float(m/sd) if sd > 0 else float("nan"), wins=int((d > 0).sum()), n=n)

def holm(pvals):
    p = np.nan_to_num(np.asarray(pvals, float), nan=1.0); n = len(p)
    order = np.argsort(p); adj = np.empty(n); run = 0.0
    for r, i in enumerate(order):
        run = max(run, min(1.0, p[i]*(n-r))); adj[i] = run
    return adj

def fmt_ms(x):
    x = np.asarray(x, float)
    return "%.4f ± %.4f" % (x.mean(), x.std(ddof=1) if len(x) > 1 else 0.0)

def _slim(m, keep_actions=False):
    return {k: v for k, v in m.items() if (k != "actions" or keep_actions) and k != "_raw"}

SEEDS = P["SEEDS"]
LEARNER_KINDS = ["lsh_grl_hpfoo", "lsh_grl", "grl", "no_graph", "dqn", "gat_ppo"]
REF_KINDS = ["greedy", "random", "local_only", "all_cloud", "oracle", "maxmin"]   # V4 order + maxmin last
MOB_LEVELS = [("static", 0.05), ("low", 0.2), ("medium", 0.45), ("high", 0.8)]
OFAT_GRID = [("deadline_scale", [0.6, 0.8, 1.0, 1.3, 1.6]),
             ("interarrival",   [0.4, 0.5, 0.75, 1.0, 1.5, 2.0]),
             ("snr_offset_db",  [-8.0, -4.0, 0.0, 4.0])]
ARRIVAL_PATTERNS = ["poisson", "periodic", "bursty", "heavy_tail"]
newenv = lambda sd, dls=None: OffloadingEnv(seed=sd, sites=sites,
                                            deadline_scale=dl_scale if dls is None else dls)


def lsh_lookup_study(ag, sd):
    """Reviewer 04: does LSH speed up retrieval at the memory size actually used?
    Times LSHCache.lookup against an exhaustive cosine scan over the SAME stored entries
    (same load bucket, same 0.99 verification, same aggregation), on the test stream."""
    lsh = ag.lsh
    by_lb, seen = {}, set()
    for (tb, lb, bits), lst in lsh.store.items():
        for (z, a, r) in lst:
            if (lb, id(z)) in seen: continue
            seen.add((lb, id(z))); by_lb.setdefault(lb, []).append((z, a, r))
    mats = {lb: (np.stack([z for z, _, _ in L]), np.array([a for _, a, _ in L]),
                 np.array([r for _, _, r in L])) for lb, L in by_lb.items()}

    def exhaustive(tf, e):
        z = lsh._z(tf); lb = lsh.load_bucket(e)
        if lb not in mats: return None
        Z, Aa, Rr = mats[lb]
        sims = Z @ z / ((np.linalg.norm(Z, axis=1)+1e-9)*(np.linalg.norm(z)+1e-9))
        ok = sims >= lsh.sim_thresh
        if not ok.any(): return None
        best, bmean, bn = None, -1e9, 0
        for a in np.unique(Aa[ok]):
            sel = ok & (Aa == a); mu = Rr[sel].mean()
            if mu > bmean: best, bmean, bn = int(a), mu, int(sel.sum())
        if bn < lsh.min_count: return None
        if lsh.r_ema is not None and bmean < lsh.r_ema: return None
        return best

    e = newenv(sd); e.reset(); t_l, t_x, agree, hit_l, hit_x = [], [], 0, 0, 0
    q0, h0 = lsh.queries, lsh.hits
    for task in test_stream:
        e.advance_to(task["arrival_ms"])
        tf = task_features(task, e); nr = e.node_features()
        t0 = time.perf_counter(); al, _ = lsh.lookup(tf, e); t_l.append((time.perf_counter()-t0)*1000)
        t0 = time.perf_counter(); ax = exhaustive(tf, e);    t_x.append((time.perf_counter()-t0)*1000)
        agree += int(al == ax); hit_l += int(al is not None); hit_x += int(ax is not None)
        a, _ = ag.act(tf, nr, e, task=task, greedy=True)
        e.step(task, a)
    lsh.queries, lsh.hits = q0, h0
    n = len(test_stream)
    return dict(memory_entries=len(seen), lsh_lookup_ms=float(np.mean(t_l)),
                exhaustive_lookup_ms=float(np.mean(t_x)),
                lookup_speedup=float(np.mean(t_x)/max(np.mean(t_l), 1e-12)),
                agreement=agree/n, lsh_hit_rate=hit_l/n, exhaustive_hit_rate=hit_x/n)


def run_seed(sd):
    rec = dict(seed=sd, learned={}, ref={}, mobility={}, ofat=[], arrival={}, lsh={})
    agents = {}
    for kind in LEARNER_KINDS:
        ag, _, m = train_one(kind, sd, verbose=False, topk=TOPK_BEST)
        agents[kind] = ag; rec["learned"][kind] = _slim(m, keep_actions=(sd == SEEDS[0]))
    # reference policies: identical RNG handling to V4 §9
    set_seed(sd); ee = newenv(sd)
    for k in REF_KINDS:
        np.random.seed(sd); rec["ref"][k] = _slim(run_policy(ee, test_stream, ref_policies[k]))
    prop, grl_ag = agents["lsh_grl_hpfoo"], agents["grl"]

    # ---- mobility (V4 Fig. 9 / §3.8, now per seed and with the validated topk) ----
    for lvl, mi in MOB_LEVELS:
        sub = [dict(t, mobility_index=mi) for t in test_stream]; row = {}
        for k in ["greedy", "random", "local_only", "all_cloud", "maxmin", "oracle"]:
            np.random.seed(sd); row[k] = run_policy(newenv(sd), sub, ref_policies[k])["reward"]
        for k, a in [("lsh_grl_hpfoo", prop), ("grl", grl_ag)]:
            row[k] = evaluate(a, newenv(sd), sub)["reward"]
        rec["mobility"][lvl] = row

    # ---- perturbed operating conditions (V4 Table 10, now per seed) ----
    for pname, mults in OFAT_GRID:
        for mv in mults:
            st = [dict(t) for t in test_stream]
            _sv = dict(SNR_DB_BY_MOBILITY)
            try:
                if pname == "deadline_scale":
                    for t in st: t["deadline_ms"] *= mv
                elif pname == "interarrival":
                    for t in st: t["arrival_ms"] *= mv
                else:
                    for kk in SNR_DB_BY_MOBILITY: SNR_DB_BY_MOBILITY[kk] = _sv[kk] + mv
                pm = evaluate(prop, newenv(sd), st)
                np.random.seed(sd); gm = run_policy(newenv(sd), st, greedy_latency_action)
                om = run_policy(newenv(sd), st, myopic_oracle_action)
                mm = run_policy(newenv(sd), st, maxmin_fair_action)
            finally:
                SNR_DB_BY_MOBILITY.update(_sv)
            rec["ofat"].append(dict(parameter=pname, value=mv, proposed=pm["reward"],
                                    greedy=gm["reward"], oracle=om["reward"], maxmin=mm["reward"],
                                    miss_pct=pm["miss"]*100))

    # ---- arrival-time distributions at the same mean load (reviewer 05) ----
    for pat in ARRIVAL_PATTERNS:
        st = make_stream(TEST_DF, env, ia_ms, pattern=pat)
        pm = evaluate(prop, newenv(sd), st); row = dict(proposed=pm["reward"], proposed_miss=pm["miss"])
        for k in ["greedy", "oracle", "maxmin"]:
            np.random.seed(sd); row[k] = run_policy(newenv(sd), st, ref_policies[k])["reward"]
        row["grl"] = evaluate(grl_ag, newenv(sd), st)["reward"]
        rec["arrival"][pat] = row

    # ---- LSH at inference (reviewer 04) ----
    saved = prop.lsh; prop.lsh = None; n0 = len(prop.n_cand)
    try:
        wo = evaluate(prop, newenv(sd), test_stream)
        cand_wo = float(np.mean(prop.n_cand[n0:])) if len(prop.n_cand) > n0 else float("nan")
    finally:
        prop.lsh = saved
    rec["lsh"] = dict(reward_with=rec["learned"]["lsh_grl_hpfoo"]["reward"], reward_without=wo["reward"],
                      decision_ms_with=rec["learned"]["lsh_grl_hpfoo"]["decision_ms"],
                      decision_ms_without=wo["decision_ms"], candidates_without=cand_wo,
                      **lsh_lookup_study(prop, sd))
    return rec

print("helpers ready | learners:", LEARNER_KINDS, "| references:", REF_KINDS)

helpers ready | learners: ['lsh_grl_hpfoo', 'lsh_grl', 'grl', 'no_graph', 'dqn', 'gat_ppo'] | references: ['greedy', 'random', 'local_only', 'all_cloud', 'oracle', 'maxmin']


In [20]:
# ===== §9a: run all seeds (cached), then the headline multi-seed table =====
SEED_RECS = {}
_t_all = time.time()
for sd in SEEDS:
    t0 = time.time()
    SEED_RECS[sd] = cached("seed_%d" % sd, lambda sd=sd: run_seed(sd))
    print("seed %d done  (%.0fs)  proposed %.4f | greedy %.4f | oracle %.4f"
          % (sd, time.time()-t0, SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"]["reward"],
             SEED_RECS[sd]["ref"]["greedy"]["reward"], SEED_RECS[sd]["ref"]["oracle"]["reward"]))
print("all seeds: %.1f min" % ((time.time()-_t_all)/60))

def V(kind, metric="reward"):
    """Per-seed values of a metric for a method, in seed order."""
    out = []
    for sd in SEEDS:
        r = SEED_RECS[sd]; src = r["learned"] if kind in r["learned"] else r["ref"]
        out.append(src[kind][metric])
    return np.array(out, float)

ALL_KINDS = LEARNER_KINDS + REF_KINDS
multi = {k: list(V(k)) for k in ALL_KINDS}
_rows = {}
for k in ALL_KINDS:
    m, lo, hi = ci95(multi[k])
    _rows[LABELS[k]] = dict(mean=m, std=float(np.std(multi[k], ddof=1)) if len(SEEDS) > 1 else 0.0,
                            ci_lo=lo, ci_hi=hi)
MULTI = pd.DataFrame(_rows).T
MULTI["reported"] = [("%.4f ± %.4f" % (r["mean"], r["std"])) for _, r in MULTI.iterrows()]
display(MULTI.round(5))
assert len(SEEDS) < 2 or MULTI["std"].drop(index=LABELS["oracle"], errors="ignore").max() > 0, \
       "Zero variance across seeds - the seed is not propagating."

# ---- Table 4 of the manuscript: proposed vs every other method, Holm-corrected ----
_cmp = [k for k in ALL_KINDS if k not in ("lsh_grl_hpfoo", "oracle")]
_res = [paired(multi["lsh_grl_hpfoo"], multi[k]) for k in _cmp]
_adj = holm([r["p"] for r in _res])
T4 = pd.DataFrame([dict(method=LABELS[k], mean=np.mean(multi[k]), sd=np.std(multi[k], ddof=1) if len(SEEDS) > 1 else 0,
                        proposed_ahead_pct=(np.mean(multi["lsh_grl_hpfoo"])/np.mean(multi[k])-1)*100,
                        diff=r["diff"], diff_ci_lo=r["ci_lo"], diff_ci_hi=r["ci_hi"],
                        p_raw=r["p"], p_holm=a, cohen_dz=r["d"], wins="%d/%d" % (r["wins"], r["n"]))
                   for k, r, a in zip(_cmp, _res, _adj)])
_o = paired(multi["lsh_grl_hpfoo"], multi["oracle"])
print("\nTABLE 4 - proposed vs each method (paired over %d seeds, Holm-corrected over %d tests)"
      % (len(SEEDS), len(_cmp)))
display(T4.round(5))
print("oracle minus proposed: %.4f (seeds where oracle is ahead: %d/%d)"
      % (-_o["diff"], _o["n"]-_o["wins"], _o["n"]))
gap_ms = (np.mean(multi["lsh_grl_hpfoo"])-np.mean(multi["greedy"]))/(np.mean(multi["oracle"])-np.mean(multi["greedy"]))*100
print("gap closed (multi-seed means): %.1f%%" % gap_ms)
RESULTS["multiseed"] = {k: [float(x) for x in v] for k, v in multi.items()}
RESULTS["table4"] = T4.to_dict("records"); RESULTS["gap_closed_multiseed_pct"] = float(gap_ms)

# ---- Ablation table: 10-seed means of every measure (greedy now also a 10-seed mean) ----
LEARNED = LEARNED if "LEARNED" in globals() else {}
_abl_rows = []
_flags = {"lsh_grl_hpfoo": (True, True, True), "lsh_grl": (True, True, False), "grl": (False, True, False),
          "no_graph": (False, False, False), "dqn": (False, False, False), "gat_ppo": (False, True, False),
          "greedy": (False, False, False), "maxmin": (False, False, False)}
_names = {"lsh_grl_hpfoo": "LSH-GRL-HPFOO (proposed)", "lsh_grl": "LSH-GRL, no HPFOO", "grl": "GRL only",
          "no_graph": "Per-node scoring, no graph", "dqn": "Standard DQN (task feats only)",
          "gat_ppo": "GAT-PPO (graph RL)", "greedy": "Greedy heuristic (no learning)",
          "maxmin": "Max-min fair heuristic"}
for k, nm in _names.items():
    Lf, G, H = _flags[k]
    row = [nm, Lf, G, H] + [float(np.mean(V(k, m))) for m in
                            ["reward", "latency", "energy", "miss", "cost_ratio", "decision_ms"]]
    cand = float(np.mean(V(k, "candidates"))) if k in SEED_RECS[SEEDS[0]]["learned"] else float(N_NODES)
    row.insert(9, cand); _abl_rows.append(row)
    if k in LEARNER_KINDS:
        LEARNED[k] = {m: float(np.mean(V(k, m))) for m in
                      ["reward", "latency", "energy", "miss", "cost_ratio", "candidates", "decision_ms"]}
        LEARNED[k]["actions"] = SEED_RECS[SEEDS[0]]["learned"][k].get("actions", [])
ABLATION = pd.DataFrame(_abl_rows, columns=["Configuration", "LSH", "Graph", "HPFOO", "Reward",
        "Latency (ms)", "Energy (J)", "Miss (%)", "Cost ratio", "Nodes searched", "Decision (ms)"])
ABLATION["Miss (%)"] *= 100
ABLATION["vs greedy (%)"] = (ABLATION["Reward"]/np.mean(multi["greedy"])-1)*100
display(ABLATION.round(4))

# ---- component contributions with paired 95% CIs (reviewer 04) ----
COMPONENTS = [("HPFOO refinement",        "lsh_grl_hpfoo", "lsh_grl"),
              ("LSH memory",              "lsh_grl",       "grl"),
              ("Graph encoder",           "grl",           "no_graph"),
              ("Node-state awareness",    "no_graph",      "dqn"),
              ("Learning vs best rule",   "grl",           "greedy"),
              ("Proposed vs GAT-PPO",     "lsh_grl_hpfoo", "gat_ppo"),
              ("Proposed vs max-min fair","lsh_grl_hpfoo", "maxmin")]
_cr = []
for nm, a, b in COMPONENTS:
    r = paired(V(a), V(b))
    _cr.append(dict(comparison=nm, a=LABELS[a], b=LABELS[b],
                    change_pct=(np.mean(V(a))/np.mean(V(b))-1)*100, diff=r["diff"],
                    ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], p_raw=r["p"], cohen_dz=r["d"],
                    wins="%d/%d" % (r["wins"], r["n"]),
                    d_miss_pp=(np.mean(V(a, "miss"))-np.mean(V(b, "miss")))*100,
                    d_latency_ms=np.mean(V(a, "latency"))-np.mean(V(b, "latency")),
                    d_cost_ratio=np.mean(V(a, "cost_ratio"))-np.mean(V(b, "cost_ratio"))))
COMPONENT_TBL = pd.DataFrame(_cr); COMPONENT_TBL["p_holm"] = holm(COMPONENT_TBL["p_raw"])
print("\nCOMPONENT CONTRIBUTIONS (paired over seeds; reward diff with 95%% CI; Holm over %d tests)" % len(_cr))
display(COMPONENT_TBL.round(5))
RESULTS["ablation"] = ABLATION.to_dict("records"); RESULTS["components"] = COMPONENT_TBL.to_dict("records")

seed 0 done  (376s)  proposed 0.8757 | greedy 0.8196 | oracle 0.8812
seed 1 done  (377s)  proposed 0.8687 | greedy 0.8163 | oracle 0.8751
seed 2 done  (376s)  proposed 0.8687 | greedy 0.8187 | oracle 0.8809
seed 3 done  (375s)  proposed 0.8730 | greedy 0.8237 | oracle 0.8793
seed 4 done  (378s)  proposed 0.8772 | greedy 0.8236 | oracle 0.8842
seed 5 done  (377s)  proposed 0.8702 | greedy 0.8433 | oracle 0.8844
seed 6 done  (378s)  proposed 0.8772 | greedy 0.8235 | oracle 0.8838
seed 7 done  (380s)  proposed 0.8795 | greedy 0.8344 | oracle 0.8862
seed 8 done  (377s)  proposed 0.8675 | greedy 0.8143 | oracle 0.8766
seed 9 done  (376s)  proposed 0.8780 | greedy 0.8300 | oracle 0.8865
all seeds: 62.8 min


,mean,std,ci_lo,ci_hi,reported
LSH-GRL-HPFOO (Proposed),0.87359,0.00449,0.87038,0.87681,0.8736 ± 0.0045
LSH-GRL (no HPFOO),0.86993,0.00504,0.86632,0.87354,0.8699 ± 0.0050
GRL only,0.86939,0.00444,0.86622,0.87257,0.8694 ± 0.0044
No-graph scoring,0.86924,0.00704,0.86421,0.87428,0.8692 ± 0.0070
Standard DQN,0.78240,0.05128,0.74571,0.81908,0.7824 ± 0.0513
GAT-PPO (graph RL),0.83431,0.01202,0.82571,0.84290,0.8343 ± 0.0120
Greedy (min-latency),0.82475,0.00890,0.81839,0.83112,0.8248 ± 0.0089
Random Offloading,0.72161,0.00955,0.71478,0.72844,0.7216 ± 0.0095
Local-Only,0.63652,0.00000,0.63652,0.63652,0.6365 ± 0.0000
All-to-Cloud,0.51450,0.01354,0.50481,0.52418,0.5145 ± 0.0135



TABLE 4 - proposed vs each method (paired over 10 seeds, Holm-corrected over 10 tests)


,method,mean,sd,proposed_ahead_pct,diff,diff_ci_lo,diff_ci_hi,p_raw,p_holm,cohen_dz,wins
0,LSH-GRL (no HPFOO),0.86993,0.00504,0.42054,0.00366,0.00152,0.00580,0.00380,0.00760,1.22328,8/10
1,GRL only,0.86939,0.00444,0.48312,0.00420,0.00213,0.00627,0.00132,0.00395,1.45046,9/10
2,No-graph scoring,0.86924,0.00704,0.49998,0.00435,0.00029,0.00840,0.03820,0.03820,0.76735,8/10
3,Standard DQN,0.78240,0.05128,11.65573,0.09119,0.05425,0.12814,0.00034,0.00136,1.76580,10/10
4,GAT-PPO (graph RL),0.83431,0.01202,4.70868,0.03928,0.03163,0.04694,0.00000,0.00001,3.67107,10/10
5,Greedy (min-latency),0.82475,0.00890,5.92137,0.04884,0.04285,0.05482,0.00000,0.00000,5.83804,10/10
6,Random Offloading,0.72161,0.00955,21.06078,0.15198,0.14555,0.15840,0.00000,0.00000,16.91482,10/10
7,Local-Only,0.63652,0.00000,37.24404,0.23707,0.23385,0.24028,0.00000,0.00000,52.74404,10/10
8,All-to-Cloud,0.51450,0.01354,69.79562,0.35910,0.35013,0.36806,0.00000,0.00000,28.66731,10/10
9,Max-min fair,0.84543,0.00833,3.33142,0.02816,0.02235,0.03398,0.00000,0.00001,3.46739,10/10


oracle minus proposed: 0.0082 (seeds where oracle is ahead: 10/10)
gap closed (multi-seed means): 85.6%


,Configuration,LSH,Graph,HPFOO,Reward,Latency (ms),Energy (J),Miss (%),Cost ratio,Nodes searched,Decision (ms),vs greedy (%)
0,LSH-GRL-HPFOO (proposed),True,True,True,0.8736,106.7177,0.0471,6.760,0.1454,3.4849,2.3067,5.9214
1,"LSH-GRL, no HPFOO",True,True,False,0.8699,105.7160,0.0467,7.470,0.1480,19.0000,1.3777,5.4778
2,GRL only,False,True,False,0.8694,111.5812,0.0492,7.510,0.1491,19.0000,1.3838,5.4121
3,"Per-node scoring, no graph",False,False,False,0.8692,101.3108,0.0450,7.540,0.1501,19.0000,0.4525,5.3944
4,Standard DQN (task feats only),False,False,False,0.7824,1233.3325,0.5031,17.490,0.1911,19.0000,0.3466,-5.1358
5,GAT-PPO (graph RL),False,True,False,0.8343,112.6031,0.0549,11.920,0.1829,19.0000,1.5127,1.1582
6,Greedy heuristic (no learning),False,False,False,0.8248,52.9818,0.0329,5.555,0.3159,19.0000,0.0227,0.0000
7,Max-min fair heuristic,False,False,False,0.8454,74.7029,0.0366,5.430,0.2518,19.0000,0.0398,2.5064



COMPONENT CONTRIBUTIONS (paired over seeds; reward diff with 95% CI; Holm over 7 tests)


,comparison,a,b,change_pct,diff,ci_lo,ci_hi,p_raw,cohen_dz,wins,d_miss_pp,d_latency_ms,d_cost_ratio,p_holm
0,HPFOO refinement,LSH-GRL-HPFOO (Proposed),LSH-GRL (no HPFOO),0.42054,0.00366,0.00152,0.00580,0.00380,1.22328,8/10,-0.710,1.00172,-0.00259,0.01139
1,LSH memory,LSH-GRL (no HPFOO),GRL only,0.06232,0.00054,-0.00087,0.00195,0.40775,0.27460,7/10,-0.040,-5.86516,-0.00113,0.81549
2,Graph encoder,GRL only,No-graph scoring,0.01678,0.00015,-0.00447,0.00476,0.94452,0.02263,4/10,-0.030,10.27041,-0.00105,0.94452
3,Node-state awareness,No-graph scoring,Standard DQN,11.10025,0.08685,0.05227,0.12142,0.00030,1.79691,10/10,-9.950,-1132.02178,-0.04101,0.00120
4,Learning vs best rule,GRL only,Greedy (min-latency),5.41211,0.04464,0.03877,0.05050,0.00000,5.44581,10/10,1.955,58.59941,-0.16686,0.00000
5,Proposed vs GAT-PPO,LSH-GRL-HPFOO (Proposed),GAT-PPO (graph RL),4.70868,0.03928,0.03163,0.04694,0.00000,3.67107,10/10,-5.160,-5.88541,-0.03751,0.00001
6,Proposed vs max-min fair,LSH-GRL-HPFOO (Proposed),Max-min fair,3.33142,0.02816,0.02235,0.03398,0.00000,3.46739,10/10,1.330,32.01482,-0.10645,0.00001


In [21]:
# ===== §9b: fairness / equity (reviewer 02) =====
FAIR_KINDS = ["lsh_grl_hpfoo", "lsh_grl", "grl", "gat_ppo", "dqn", "greedy", "maxmin", "oracle", "random"]
FAIR_METRICS = [("jain_tenant_success", "Jain (tenant success)"), ("miss_gap", "Miss gap (max-min)"),
                ("worst_tenant_miss", "Worst-tenant miss"), ("lat_p95", "Latency P95 (ms)"),
                ("lat_p99", "Latency P99 (ms)"), ("worst_tenant_p95", "Worst-tenant P95 (ms)"),
                ("jain_server_util", "Jain (server load)"), ("jain_edge_util", "Jain (edge load)"),
                ("miss_urgent", "URLLC miss"), ("miss_highprio", "High-priority miss"),
                ("cost_ratio", "Cost ratio")]
FAIRNESS = pd.DataFrame({LABELS[k]: {lab: fmt_ms(V(k, m)) for m, lab in FAIR_METRICS} for k in FAIR_KINDS}).T
print("FAIRNESS MEASURES (mean ± s.d. over %d seeds)" % len(SEEDS))
display(FAIRNESS)
FAIRNESS_MEAN = pd.DataFrame({LABELS[k]: {lab: float(np.mean(V(k, m))) for m, lab in FAIR_METRICS}
                              for k in FAIR_KINDS}).T

# Proposed vs greedy and vs max-min on each fairness measure (sign: for Jain higher is better,
# for everything else lower is better). Holm over all tests in this table.
_fr = []
for base in ["greedy", "maxmin"]:
    for m, lab in FAIR_METRICS:
        r = paired(V("lsh_grl_hpfoo", m), V(base, m))
        better = (r["diff"] > 0) if m.startswith("jain") else (r["diff"] < 0)
        _fr.append(dict(vs=LABELS[base], measure=lab, proposed=np.mean(V("lsh_grl_hpfoo", m)),
                        baseline=np.mean(V(base, m)), diff=r["diff"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                        p_raw=r["p"], proposed_better=bool(better)))
FAIR_TESTS = pd.DataFrame(_fr); FAIR_TESTS["p_holm"] = holm(FAIR_TESTS["p_raw"])
display(FAIR_TESTS.round(5))

# Does the aggregate gain come at the expense of a particular tenant or priority group?
_tr = []
for g in SOURCES:
    pv = np.array([SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"]["miss_by_source"].get(g, np.nan) for sd in SEEDS])
    gv = np.array([SEED_RECS[sd]["ref"]["greedy"]["miss_by_source"].get(g, np.nan) for sd in SEEDS])
    pr = np.array([SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"]["reward_by_source"].get(g, np.nan) for sd in SEEDS])
    gr = np.array([SEED_RECS[sd]["ref"]["greedy"]["reward_by_source"].get(g, np.nan) for sd in SEEDS])
    rr, rm = paired(pr, gr), paired(pv, gv)
    _tr.append(dict(tenant=g, reward_proposed=pr.mean(), reward_greedy=gr.mean(), reward_diff=rr["diff"],
                    reward_p=rr["p"], miss_proposed_pct=pv.mean()*100, miss_greedy_pct=gv.mean()*100,
                    miss_diff_pp=rm["diff"]*100, miss_p=rm["p"]))
for m, lab in [("miss_urgent", "URLLC jobs"), ("miss_highprio", "top-quartile priority")]:
    rm = paired(V("lsh_grl_hpfoo", m), V("greedy", m))
    _tr.append(dict(tenant=lab, miss_proposed_pct=np.mean(V("lsh_grl_hpfoo", m))*100,
                    miss_greedy_pct=np.mean(V("greedy", m))*100, miss_diff_pp=rm["diff"]*100, miss_p=rm["p"]))
TENANT_TBL = pd.DataFrame(_tr)
_pp = list(TENANT_TBL["reward_p"].fillna(1.0)) + list(TENANT_TBL["miss_p"].fillna(1.0))
_h = holm(_pp); n_t = len(TENANT_TBL)
TENANT_TBL["reward_p_holm"] = [(_h[i] if not np.isnan(TENANT_TBL["reward_p"].iloc[i]) else np.nan) for i in range(n_t)]
TENANT_TBL["miss_p_holm"] = _h[n_t:]
print("\nPER-TENANT AND PRIORITY-GROUP OUTCOMES: proposed vs greedy")
display(TENANT_TBL.round(5))
_worse = TENANT_TBL[(TENANT_TBL.miss_diff_pp > 0) & (TENANT_TBL.miss_p_holm < 0.05)]
print("groups whose deadline-miss rate is significantly WORSE under the proposed method:",
      list(_worse.tenant) if len(_worse) else "none")
RESULTS["fairness"] = FAIRNESS_MEAN.to_dict(); RESULTS["fairness_tests"] = FAIR_TESTS.to_dict("records")
RESULTS["tenant_outcomes"] = TENANT_TBL.to_dict("records")

FAIRNESS MEASURES (mean ± s.d. over 10 seeds)


,Jain (tenant success),Miss gap (max-min),Worst-tenant miss,Latency P95 (ms),Latency P99 (ms),Worst-tenant P95 (ms),Jain (server load),Jain (edge load),URLLC miss,High-priority miss,Cost ratio
LSH-GRL-HPFOO (Proposed),0.9975 ± 0.0010,0.1263 ± 0.0235,0.1263 ± 0.0235,372.0592 ± 30.5153,648.9370 ± 106.3804,554.5473 ± 70.5113,0.5671 ± 0.0586,0.8140 ± 0.0947,0.2174 ± 0.0253,0.0318 ± 0.0059,0.1454 ± 0.0128
LSH-GRL (no HPFOO),0.9971 ± 0.0011,0.1368 ± 0.0227,0.1368 ± 0.0227,403.8535 ± 140.6228,661.8535 ± 218.4574,580.3004 ± 126.2668,0.5155 ± 0.0384,0.7220 ± 0.0778,0.2400 ± 0.0322,0.0356 ± 0.0099,0.1480 ± 0.0081
GRL only,0.9971 ± 0.0007,0.1370 ± 0.0174,0.1370 ± 0.0174,413.4661 ± 122.6803,841.8909 ± 457.5173,678.5146 ± 294.0647,0.5134 ± 0.0513,0.7125 ± 0.0702,0.2410 ± 0.0233,0.0345 ± 0.0054,0.1491 ± 0.0096
GAT-PPO (graph RL),0.9892 ± 0.0057,0.2388 ± 0.0658,0.2390 ± 0.0661,347.1597 ± 142.8029,652.6317 ± 322.0228,535.6331 ± 34.3571,0.4740 ± 0.1800,0.9787 ± 0.0470,0.3771 ± 0.0985,0.0621 ± 0.0203,0.1829 ± 0.0833
Standard DQN,0.9735 ± 0.0303,0.2929 ± 0.1478,0.3242 ± 0.1537,8652.6127 ± 15112.7938,12355.0227 ± 20279.4371,10950.6110 ± 18156.0031,0.2672 ± 0.0855,0.2886 ± 0.1178,0.3704 ± 0.0783,0.1275 ± 0.1095,0.1911 ± 0.0142
Greedy (min-latency),0.9986 ± 0.0003,0.0988 ± 0.0091,0.0988 ± 0.0091,130.1235 ± 4.9950,480.3692 ± 4.3261,377.7160 ± 0.0000,0.2401 ± 0.0455,0.1773 ± 0.0588,0.1811 ± 0.0103,0.0219 ± 0.0035,0.3159 ± 0.0233
Max-min fair,0.9986 ± 0.0002,0.0982 ± 0.0085,0.0982 ± 0.0085,256.1173 ± 31.1435,514.0175 ± 4.8590,487.7794 ± 10.1277,0.3614 ± 0.0167,0.4605 ± 0.0330,0.1770 ± 0.0120,0.0218 ± 0.0033,0.2518 ± 0.0215
Myopic Oracle (bound),0.9986 ± 0.0002,0.0961 ± 0.0082,0.0961 ± 0.0082,369.7329 ± 33.0711,578.2272 ± 27.7274,535.8505 ± 9.7277,0.5017 ± 0.0228,0.8100 ± 0.0332,0.1748 ± 0.0116,0.0203 ± 0.0031,0.1280 ± 0.0044
Random Offloading,0.9883 ± 0.0021,0.2367 ± 0.0184,0.3287 ± 0.0239,1029.3331 ± 216.1163,1979.4800 ± 374.0287,1515.2117 ± 178.0232,0.6231 ± 0.0261,0.9271 ± 0.0271,0.5242 ± 0.0138,0.1503 ± 0.0174,0.2908 ± 0.0080


,vs,measure,proposed,baseline,diff,ci_lo,ci_hi,p_raw,proposed_better,p_holm
0,Greedy (min-latency),Jain (tenant success),0.99755,0.99856,-0.00101,-0.00166,-0.00036,0.00675,False,0.02521
1,Greedy (min-latency),Miss gap (max-min),0.12628,0.09877,0.02752,0.01130,0.04373,0.00397,False,0.02521
2,Greedy (min-latency),Worst-tenant miss,0.12628,0.09877,0.02752,0.01130,0.04373,0.00397,False,0.02521
3,Greedy (min-latency),Latency P95 (ms),372.05919,130.12348,241.93572,221.09648,262.77495,0.00000,False,0.00000
4,Greedy (min-latency),Latency P99 (ms),648.93695,480.36922,168.56774,94.03671,243.09876,0.00063,False,0.00694
5,Greedy (min-latency),Worst-tenant P95 (ms),554.54728,377.71601,176.83127,126.39054,227.27201,0.00002,False,0.00033
6,Greedy (min-latency),Jain (server load),0.56706,0.24006,0.32700,0.28895,0.36505,0.00000,True,0.00000
7,Greedy (min-latency),Jain (edge load),0.81398,0.17735,0.63663,0.57533,0.69793,0.00000,True,0.00000
8,Greedy (min-latency),URLLC miss,0.21743,0.18109,0.03635,0.02103,0.05166,0.00045,False,0.00541
9,Greedy (min-latency),High-priority miss,0.03178,0.02189,0.00989,0.00501,0.01477,0.00132,False,0.01191



PER-TENANT AND PRIORITY-GROUP OUTCOMES: proposed vs greedy


,tenant,reward_proposed,reward_greedy,reward_diff,reward_p,miss_proposed_pct,miss_greedy_pct,miss_diff_pp,miss_p,reward_p_holm,miss_p_holm
0,task_offloading,0.86191,0.84586,0.01605,0.00000,7.85263,6.23158,1.62105,0.00000,0.00001,0.00001
1,vec_edge,0.86526,0.83881,0.02645,0.00000,6.98656,6.41075,0.57582,0.01338,0.00000,0.06688
2,azure_llm,0.94577,0.94788,-0.00211,0.04669,0.00000,0.00000,0.00000,NaN,0.18675,1.00000
3,borg,0.81727,0.65842,0.15885,0.00000,12.62834,9.87680,2.75154,0.00397,0.00000,0.02383
4,URLLC jobs,NaN,NaN,NaN,NaN,21.74342,18.10855,3.63487,0.00045,NaN,0.00361
5,top-quartile priority,NaN,NaN,NaN,NaN,3.17797,2.18927,0.98870,0.00132,NaN,0.00926


groups whose deadline-miss rate is significantly WORSE under the proposed method: ['task_offloading', 'borg', 'URLLC jobs', 'top-quartile priority']


In [22]:
# ===== §9c: robustness over seeds (reviewer 05) =====
# ---- mobility ----
_mk = ["lsh_grl_hpfoo", "grl", "greedy", "maxmin", "random", "local_only", "all_cloud", "oracle"]
MOB_SEEDS = {lvl: {k: np.array([SEED_RECS[sd]["mobility"][lvl][k] for sd in SEEDS]) for k in _mk}
             for lvl, _ in MOB_LEVELS}
MOBILITY = pd.DataFrame({lvl: {k: v.mean() for k, v in d.items()} for lvl, d in MOB_SEEDS.items()}).T
MOBILITY_SD = pd.DataFrame({lvl: {k: (v.std(ddof=1) if len(v) > 1 else 0.0) for k, v in d.items()}
                            for lvl, d in MOB_SEEDS.items()}).T
_mr = []
for base in ["greedy", "grl"]:
    for lvl, _ in MOB_LEVELS:
        r = paired(MOB_SEEDS[lvl]["lsh_grl_hpfoo"], MOB_SEEDS[lvl][base])
        _mr.append(dict(level=lvl, vs=LABELS[base], diff=r["diff"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"],
                        p_raw=r["p"], wins="%d/%d" % (r["wins"], r["n"])))
MOB_TESTS = pd.DataFrame(_mr); MOB_TESTS["p_holm"] = holm(MOB_TESTS["p_raw"])
print("MOBILITY (mean over %d seeds)" % len(SEEDS)); display(MOBILITY.round(4))
display(MOB_TESTS.round(5))

# ---- perturbed operating conditions ----
_of = pd.DataFrame([dict(r, seed=sd) for sd in SEEDS for r in SEED_RECS[sd]["ofat"]])
_of["gap_closed"] = 100*(_of.proposed-_of.greedy)/(_of.oracle-_of.greedy).where(lambda x: x.abs() > 1e-9)
_orows = []
for (pn, val), g in _of.groupby(["parameter", "value"], sort=False):
    rg, ro = paired(g.proposed, g.greedy), paired(g.proposed, g.oracle)
    _orows.append(dict(parameter=pn, value=val, proposed=g.proposed.mean(),
                       proposed_sd=g.proposed.std(ddof=1) if len(g) > 1 else 0.0,
                       greedy=g.greedy.mean(), maxmin=g.maxmin.mean(), oracle=g.oracle.mean(),
                       miss_pct=g.miss_pct.mean(),
                       gap_closed=100*(g.proposed.mean()-g.greedy.mean())/(g.oracle.mean()-g.greedy.mean()),
                       gap_closed_seed_median=g.gap_closed.median(),
                       p_vs_greedy=rg["p"], wins_vs_greedy="%d/%d" % (rg["wins"], rg["n"]),
                       diff_vs_oracle=ro["diff"], p_vs_oracle=ro["p"],
                       seeds_above_oracle="%d/%d" % (ro["wins"], ro["n"])))
OFAT_DF = pd.DataFrame(_orows)
OFAT_DF["p_vs_greedy_holm"] = holm(OFAT_DF.p_vs_greedy)
OFAT_DF["p_vs_oracle_holm"] = holm(OFAT_DF.p_vs_oracle)
print("\nPERTURBED CONDITIONS (trained policy fixed; mean over %d seeds)" % len(SEEDS))
display(OFAT_DF.round(4))
print("proposed beats greedy (Holm p<0.05 and ahead on average) in %d of %d conditions"
      % (int(((OFAT_DF.proposed > OFAT_DF.greedy) & (OFAT_DF.p_vs_greedy_holm < 0.05)).sum()), len(OFAT_DF)))
_heavy = OFAT_DF[(OFAT_DF.parameter == "interarrival") & (OFAT_DF.value <= 0.5)]
print("heavy-load check (does the proposed method exceed the oracle?):")
display(_heavy[["value", "proposed", "oracle", "diff_vs_oracle", "p_vs_oracle", "p_vs_oracle_holm",
                "seeds_above_oracle"]].round(5))

# ---- arrival-time distributions ----
_ak = ["proposed", "grl", "greedy", "maxmin", "oracle"]
ARR_SEEDS = {pat: {k: np.array([SEED_RECS[sd]["arrival"][pat][k] for sd in SEEDS]) for k in _ak}
             for pat in ARRIVAL_PATTERNS}
_ar = []
for pat in ARRIVAL_PATTERNS:
    d = ARR_SEEDS[pat]; rg = paired(d["proposed"], d["greedy"])
    _ar.append(dict(pattern=pat, **{k: d[k].mean() for k in _ak},
                    proposed_sd=d["proposed"].std(ddof=1) if len(SEEDS) > 1 else 0.0,
                    gap_closed=100*(d["proposed"].mean()-d["greedy"].mean())/(d["oracle"].mean()-d["greedy"].mean()),
                    p_vs_greedy=rg["p"], wins_vs_greedy="%d/%d" % (rg["wins"], rg["n"])))
ARRIVAL_TBL = pd.DataFrame(_ar); ARRIVAL_TBL["p_vs_greedy_holm"] = holm(ARRIVAL_TBL.p_vs_greedy)
print("\nARRIVAL-TIME DISTRIBUTIONS (same mean load; trained policy fixed)")
display(ARRIVAL_TBL.round(4))
RESULTS["mobility"] = MOBILITY.to_dict(); RESULTS["mobility_tests"] = MOB_TESTS.to_dict("records")
RESULTS["param_sensitivity_ofat"] = OFAT_DF.to_dict("records")
RESULTS["arrival_patterns"] = ARRIVAL_TBL.to_dict("records")

MOBILITY (mean over 10 seeds)


,lsh_grl_hpfoo,grl,greedy,maxmin,random,local_only,all_cloud,oracle
static,0.8860,0.8824,0.8387,0.8537,0.7298,0.6365,0.5243,0.8926
low,0.8814,0.8776,0.8328,0.8490,0.7267,0.6365,0.5202,0.8884
medium,0.8696,0.8641,0.8236,0.8417,0.7203,0.6365,0.5119,0.8806
high,0.8450,0.8398,0.8084,0.8273,0.7068,0.6365,0.4936,0.8651


,level,vs,diff,ci_lo,ci_hi,p_raw,wins,p_holm
0,static,Greedy (min-latency),0.04732,0.04102,0.05362,0.00000,10/10,0.00000
1,low,Greedy (min-latency),0.04859,0.04295,0.05424,0.00000,10/10,0.00000
2,medium,Greedy (min-latency),0.04593,0.03767,0.05419,0.00000,10/10,0.00000
3,high,Greedy (min-latency),0.03664,0.02525,0.04802,0.00005,9/10,0.00023
4,static,GRL only,0.00365,0.00116,0.00614,0.00904,8/10,0.02264
5,low,GRL only,0.00379,0.00162,0.00597,0.00337,9/10,0.01348
6,medium,GRL only,0.00549,0.00187,0.00912,0.00755,9/10,0.02264
7,high,GRL only,0.00516,0.00032,0.01001,0.03920,9/10,0.03920



PERTURBED CONDITIONS (trained policy fixed; mean over 10 seeds)


,parameter,value,proposed,proposed_sd,greedy,maxmin,oracle,miss_pct,gap_closed,gap_closed_seed_median,p_vs_greedy,wins_vs_greedy,diff_vs_oracle,p_vs_oracle,seeds_above_oracle,p_vs_greedy_holm,p_vs_oracle_holm
0,deadline_scale,0.60,0.8357,0.0042,0.7924,0.8099,0.8487,13.485,76.9197,77.5511,0.0000,10/10,-0.0130,0.0000,0/10,0.0000,0.0000
1,deadline_scale,0.80,0.8588,0.0042,0.8122,0.8312,0.8681,9.290,83.2626,84.4008,0.0000,10/10,-0.0094,0.0000,0/10,0.0000,0.0000
2,deadline_scale,1.00,0.8736,0.0045,0.8248,0.8454,0.8818,6.760,85.5728,87.8180,0.0000,10/10,-0.0082,0.0000,0/10,0.0000,0.0001
3,deadline_scale,1.30,0.8876,0.0040,0.8384,0.8679,0.8954,4.380,86.2444,86.1227,0.0000,10/10,-0.0078,0.0000,0/10,0.0000,0.0002
4,deadline_scale,1.60,0.8939,0.0070,0.8457,0.8834,0.9040,2.840,82.7524,86.0434,0.0000,10/10,-0.0101,0.0004,0/10,0.0000,0.0018
5,interarrival,0.40,0.8401,0.0164,0.8217,0.8299,0.8330,7.780,162.9053,166.1921,0.0080,9/10,0.0071,0.1835,7/10,0.0080,0.3670
6,interarrival,0.50,0.8544,0.0142,0.8227,0.8382,0.8415,7.330,168.6526,170.6329,0.0002,9/10,0.0129,0.0171,9/10,0.0005,0.0514
7,interarrival,0.75,0.8696,0.0045,0.8240,0.8437,0.8710,6.905,96.9875,97.3877,0.0000,10/10,-0.0014,0.3039,3/10,0.0000,0.3670
8,interarrival,1.00,0.8736,0.0045,0.8248,0.8454,0.8818,6.760,85.5728,87.8180,0.0000,10/10,-0.0082,0.0000,0/10,0.0000,0.0001
9,interarrival,1.50,0.8755,0.0043,0.8256,0.8471,0.8861,6.690,82.4834,83.4393,0.0000,10/10,-0.0106,0.0000,0/10,0.0000,0.0000


proposed beats greedy (Holm p<0.05 and ahead on average) in 15 of 15 conditions
heavy-load check (does the proposed method exceed the oracle?):


,value,proposed,oracle,diff_vs_oracle,p_vs_oracle,p_vs_oracle_holm,seeds_above_oracle
5,0.4,0.84015,0.83301,0.00713,0.18351,0.36703,7/10
6,0.5,0.85442,0.84151,0.01291,0.01712,0.05136,9/10



ARRIVAL-TIME DISTRIBUTIONS (same mean load; trained policy fixed)


,pattern,proposed,grl,greedy,maxmin,oracle,proposed_sd,gap_closed,p_vs_greedy,wins_vs_greedy,p_vs_greedy_holm
0,poisson,0.8736,0.8694,0.8248,0.8454,0.8818,0.0045,85.5728,0.0,10/10,0.0
1,periodic,0.8741,0.8701,0.8242,0.8450,0.8831,0.0045,84.7790,0.0,10/10,0.0
2,bursty,0.8717,0.8681,0.8242,0.8445,0.8780,0.0045,88.2545,0.0,10/10,0.0
3,heavy_tail,0.8718,0.8686,0.8242,0.8442,0.8785,0.0040,87.6044,0.0,10/10,0.0


In [23]:
# ===== §9d: LSH at the workload size actually used (reviewer 04) =====
LSH_INF = pd.DataFrame([SEED_RECS[sd]["lsh"] for sd in SEEDS], index=SEEDS)
display(LSH_INF.round(5))
_rw = paired(LSH_INF.reward_with, LSH_INF.reward_without)
_dt = paired(LSH_INF.decision_ms_with, LSH_INF.decision_ms_without)
_lk = paired(LSH_INF.exhaustive_lookup_ms, LSH_INF.lsh_lookup_ms)
LSH_SUMMARY = pd.DataFrame([
    dict(measure="stored memory entries", value=LSH_INF.memory_entries.mean()),
    dict(measure="LSH lookup (ms/query)", value=LSH_INF.lsh_lookup_ms.mean()),
    dict(measure="exhaustive scan of the same memory (ms/query)", value=LSH_INF.exhaustive_lookup_ms.mean()),
    dict(measure="lookup speed-up (exhaustive / LSH)", value=LSH_INF.lookup_speedup.mean(), p=_lk["p"]),
    dict(measure="LSH / exhaustive answer agreement", value=LSH_INF.agreement.mean()),
    dict(measure="decision time with LSH candidate (ms)", value=LSH_INF.decision_ms_with.mean()),
    dict(measure="decision time without LSH candidate (ms)", value=LSH_INF.decision_ms_without.mean(), p=_dt["p"]),
    dict(measure="reward with LSH candidate", value=LSH_INF.reward_with.mean()),
    dict(measure="reward without LSH candidate", value=LSH_INF.reward_without.mean(), p=_rw["p"]),
    dict(measure="LSH-GRL decision time (LSH unused at inference)", value=float(np.mean(V("lsh_grl", "decision_ms")))),
    dict(measure="GRL-only decision time", value=float(np.mean(V("grl", "decision_ms")))),
])
display(LSH_SUMMARY.round(5))
print("Interpretation guide: speed-up < 1 means the hash lookup is SLOWER than scanning the whole memory at this size.")
RESULTS["lsh_inference"] = LSH_SUMMARY.to_dict("records")

,reward_with,reward_without,decision_ms_with,decision_ms_without,candidates_without,memory_entries,lsh_lookup_ms,exhaustive_lookup_ms,lookup_speedup,agreement,lsh_hit_rate,exhaustive_hit_rate
0,0.87568,0.87353,2.29038,1.57789,3.0,3433,0.62879,0.33886,0.53891,0.6660,0.7030,0.5070
1,0.86873,0.86814,2.30272,1.60267,3.0,3580,0.63768,0.34060,0.53412,0.6860,0.7055,0.5200
2,0.86874,0.86874,2.29887,1.57797,3.0,3363,0.63217,0.33896,0.53619,0.5810,0.7180,0.4355
3,0.87300,0.87266,2.30343,1.58203,3.0,3401,0.62632,0.32789,0.52352,0.6885,0.7160,0.4880
4,0.87723,0.87538,2.35171,1.59899,3.0,3470,0.65534,0.34063,0.51977,0.6860,0.6835,0.4845
5,0.87022,0.87190,2.31324,1.57424,3.0,3380,0.65630,0.33268,0.50690,0.6515,0.7250,0.5230
6,0.87725,0.87680,2.31209,1.59137,3.0,3474,0.64520,0.34300,0.53161,0.5790,0.7005,0.4560
7,0.87952,0.87910,2.31350,1.59733,3.0,3453,0.64975,0.33892,0.52162,0.6045,0.7140,0.4310
8,0.86751,0.86657,2.29745,1.59382,3.0,3477,0.63917,0.34070,0.53303,0.6750,0.7060,0.4990
9,0.87804,0.87747,2.28369,1.61774,3.0,3654,0.63312,0.35845,0.56616,0.6555,0.7070,0.4930


,measure,value,p
0,stored memory entries,3468.50000,NaN
1,LSH lookup (ms/query),0.64039,NaN
2,exhaustive scan of the same memory (ms/query),0.34007,NaN
3,lookup speed-up (exhaustive / LSH),0.53118,0.00000
4,LSH / exhaustive answer agreement,0.64730,NaN
5,decision time with LSH candidate (ms),2.30671,NaN
6,decision time without LSH candidate (ms),1.59140,0.00000
7,reward with LSH candidate,0.87359,NaN
8,reward without LSH candidate,0.87303,0.12245
9,LSH-GRL decision time (LSH unused at inference),1.37767,NaN


Interpretation guide: speed-up < 1 means the hash lookup is SLOWER than scanning the whole memory at this size.


In [24]:
# ===== §9e: regression check against the V4 run (verification step) =====
# V4 per-seed rewards (results_V4/results_all.json). Reference policies do not use the
# graph, so they must match for every seed. Learned methods match V4 for every seed only
# when ADJ_FROM_SEED_ENV = False; with the fix on, only seed 0 (whose graph is unchanged)
# must match. Differences elsewhere are the effect of the graph fix, not an error.
V4_MULTISEED = {
 "lsh_grl_hpfoo": [0.8755419822, 0.8673367777, 0.8710413588, 0.8725825641, 0.8758366155, 0.8740355807, 0.8748203467, 0.8786044098, 0.8655802098, 0.8772259459],
 "lsh_grl": [0.8699006508, 0.8659376691, 0.8696562634, 0.8692611709, 0.8758368802, 0.8722812015, 0.8731414361, 0.8707750803, 0.8632654137, 0.8756921573],
 "grl": [0.8726574728, 0.8598295056, 0.867220862, 0.8690866714, 0.8710887426, 0.8591170005, 0.8716041818, 0.87048987, 0.8606490095, 0.8747345952],
 "dqn": [0.7941880941, 0.8093833935, 0.8265618329, 0.7098945525, 0.7935391227, 0.7986438704, 0.821041104, 0.7627774946, 0.7632211399, 0.7166047593],
 "greedy": [0.8195752095, 0.8163414535, 0.8187414674, 0.8237486816, 0.8236053142, 0.8433392777, 0.8234864748, 0.8343778122, 0.8142944299, 0.8300313011],
 "random": [0.7283293986, 0.7044575222, 0.7295140682, 0.7117287141, 0.7117221398, 0.7252800637, 0.728813613, 0.7338103934, 0.7191631627, 0.7233154516],
 "local_only": [0.6365237467]*10,
 "all_cloud": [0.5102168641, 0.5184507162, 0.5184654772, 0.5026190271, 0.537236126, 0.504309804, 0.5021582385, 0.5138467138, 0.5008119465, 0.5368405087],
 "oracle": [0.8812397183, 0.8751334789, 0.8809211676, 0.8792531241, 0.8842361443, 0.8844348186, 0.8837899951, 0.8861518195, 0.8766013906, 0.8864840276]}
V4_OFAT_SEED0 = {("deadline_scale", 0.6): 0.8374202579, ("deadline_scale", 0.8): 0.8595803871,
 ("deadline_scale", 1.0): 0.8755419822, ("deadline_scale", 1.3): 0.8885965838, ("deadline_scale", 1.6): 0.8951166699,
 ("interarrival", 0.5): 0.8552177785, ("interarrival", 0.75): 0.869881631, ("interarrival", 1.0): 0.8755419822,
 ("interarrival", 1.5): 0.8776601798, ("interarrival", 2.0): 0.8777082716, ("snr_offset_db", -8.0): 0.8504112527,
 ("snr_offset_db", -4.0): 0.8649353779, ("snr_offset_db", 0.0): 0.8755419822, ("snr_offset_db", 4.0): 0.8805913334}
TOL = 1e-6
REGRESSION = []
if PROFILE == "full":
    for k, v4 in V4_MULTISEED.items():
        for sd in SEEDS:
            must = (k in REF_KINDS) or (not ADJ_FROM_SEED_ENV) or sd == 0
            new = multi[k][SEEDS.index(sd)]
            REGRESSION.append(dict(method=k, seed=sd, v4=v4[sd], v5=new, abs_diff=abs(new-v4[sd]),
                                   must_match=must, ok=(abs(new-v4[sd]) < TOL) or not must))
    for r in SEED_RECS[0]["ofat"]:
        key = (r["parameter"], r["value"])
        if key in V4_OFAT_SEED0:
            REGRESSION.append(dict(method="ofat:%s=%g" % key, seed=0, v4=V4_OFAT_SEED0[key], v5=r["proposed"],
                                   abs_diff=abs(r["proposed"]-V4_OFAT_SEED0[key]), must_match=True,
                                   ok=abs(r["proposed"]-V4_OFAT_SEED0[key]) < TOL))
    REGRESSION = pd.DataFrame(REGRESSION)
    _bad = REGRESSION[~REGRESSION.ok]
    print("REGRESSION vs V4: %d of %d required values match within %g"
          % (int(REGRESSION[REGRESSION.must_match].ok.sum()), int(REGRESSION.must_match.sum()), TOL))
    if len(_bad):
        print("MISMATCHES (check runtime type / library versions - V4 ran on a Colab GPU runtime):")
        display(_bad)
    _chg = REGRESSION[~REGRESSION.must_match]
    if len(_chg):
        print("effect of the graph fix on seeds 1-9 (mean |change| in reward): %.5f" % _chg.abs_diff.mean())
    REGRESSION.to_csv(os.path.join(OUT_DIR, "check_regression_vs_V4.csv"), index=False)
else:
    print("regression check runs only with PROFILE = 'full' (V4 used the full settings)")

# Internal consistency: the Poisson arrival stream is the test stream, so it must reproduce
# the proposed method's test reward exactly on every seed.
_pc = [abs(SEED_RECS[sd]["arrival"]["poisson"]["proposed"]-SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"]["reward"])
       for sd in SEEDS]
print("Poisson-pattern replay reproduces the test reward: max |diff| = %.2e" % max(_pc))
assert max(_pc) < 1e-9, "evaluation is not deterministic"

REGRESSION vs V4: 51 of 68 required values match within 1e-06
MISMATCHES (check runtime type / library versions - V4 ran on a Colab GPU runtime):


,method,seed,v4,v5,abs_diff,must_match,ok
0,lsh_grl_hpfoo,0,0.875542,0.875684,0.000142,True,False
20,grl,0,0.872657,0.872666,0.000009,True,False
30,dqn,0,0.794188,0.688826,0.105362,True,False
90,ofat:deadline_scale=0.6,0,0.837420,0.836099,0.001321,True,False
91,ofat:deadline_scale=0.8,0,0.859580,0.860407,0.000827,True,False
92,ofat:deadline_scale=1,0,0.875542,0.875684,0.000142,True,False
93,ofat:deadline_scale=1.3,0,0.888597,0.886326,0.002271,True,False
94,ofat:deadline_scale=1.6,0,0.895117,0.885254,0.009863,True,False
95,ofat:interarrival=0.5,0,0.855218,0.852957,0.002260,True,False
96,ofat:interarrival=0.75,0,0.869882,0.870587,0.000705,True,False


effect of the graph fix on seeds 1-9 (mean |change| in reward): 0.00614
Poisson-pattern replay reproduces the test reward: max |diff| = 0.00e+00


---
## §12 — Client requirement 1: deriving the reward weights

The four objective weights are derived from the data by Shannon-entropy weighting, then stress-tested across seven weight scenarios. They are **not** tuned: reward is defined by the weights, so optimising them against reward would be circular.

In [25]:
# ============================================================================
# REQUIREMENT 1 - Deriving the reward weights instead of asserting them
# ============================================================================
# The four reward weights (0.35 / 0.15 / 0.15 / 0.35) were previously chosen by
# hand. This section derives them from the data by Shannon-entropy weighting and
# then shows that the ranking of methods does not depend on the choice.
#
# IMPORTANT: the weights cannot be optimised against reward, because reward is
# DEFINED by the weights - any vector is optimal under its own objective. So we
# (a) derive a data-driven vector, and (b) test robustness. We do not "tune" them.
#
# Note also that HPFOO already learns six weights, but those are NODE-UTILITY
# weights fitted to realised reward. They are a different object from these four
# objective weights and must not be conflated in the write-up.

def term_matrix(env_seed, stream, actions):
    """Replay a fixed action sequence and return the four raw reward TERMS per task."""
    e = OffloadingEnv(seed=env_seed, sites=sites, deadline_scale=dl_scale); e.reset()
    rows = []
    for task, a in zip(stream, actions):
        e.advance_to(task["arrival_ms"])
        lat, en = e.evaluate(task, a)
        D = max(task["deadline_ms"], 1e-6)
        e_loc = max(task["cpu_cycles"]*KAPPA_DEVICE, 1e-9)
        cst, cref = e.cost(task, a), max(e.cost_ref(task), 1e-12)
        rows.append([1.0 if lat <= D else 0.0,
                     float(np.clip(1.0 - lat/D, 0, 1)),
                     float(np.clip(1.0 - en/e_loc, 0, 1)),
                     float(np.clip(1.0 - cst/cref, 0, 1))])
        if a != 0: e.backlog[a] += task["cpu_cycles"]
    return np.asarray(rows)

def entropy_weights(M):
    """Shannon-entropy (objective) weighting over an n x m decision matrix."""
    P = M + 1e-12
    P = P / P.sum(0, keepdims=True)
    n = P.shape[0]
    E = -(P*np.log(P)).sum(0) / np.log(n)      # entropy of each criterion
    d = 1.0 - E                                 # degree of diversification
    return d / d.sum()

# Reference behaviour: the myopic oracle, i.e. the term distribution a per-step
# optimal scheduler actually produces on this workload.
_orc_actions = run_policy(eval_env, test_stream, myopic_oracle_action,
                          collect=True)["_raw"]["A"]
M_ref = term_matrix(0, test_stream, _orc_actions)
W_ENT = entropy_weights(M_ref)
TERMS = ["success", "latency", "energy", "cost"]
W_USED = np.array([W_SUCCESS, W_LAT, W_ENERGY, W_COST])

print("REWARD-WEIGHT DERIVATION (Shannon entropy over oracle term distribution)")
print("-"*68)
print("  %-10s %10s %10s %10s" % ("term", "entropy", "derived", "used (v3)"))
_Pm = M_ref + 1e-12; _Pm = _Pm/_Pm.sum(0, keepdims=True)
_E = -(_Pm*np.log(_Pm)).sum(0)/np.log(len(_Pm))
for i, t in enumerate(TERMS):
    print("  %-10s %10.4f %10.4f %10.2f" % (t, _E[i], W_ENT[i], W_USED[i]))
print("-"*68)
print("  cosine similarity derived vs used : %.4f"
      % float(W_ENT @ W_USED / (np.linalg.norm(W_ENT)*np.linalg.norm(W_USED))))
RESULTS["reward_weights"] = dict(
    terms=TERMS, entropy=[float(x) for x in _E],
    derived=[float(x) for x in W_ENT], used=[float(x) for x in W_USED])

REWARD-WEIGHT DERIVATION (Shannon entropy over oracle term distribution)
--------------------------------------------------------------------
  term          entropy    derived  used (v3)
  success        0.9919     0.2980       0.35
  latency        0.9840     0.5861       0.15
  energy         0.9977     0.0835       0.15
  cost           0.9991     0.0324       0.35
--------------------------------------------------------------------
  cosine similarity derived vs used : 0.6047


In [26]:
# ---- Weight sensitivity: does the CONCLUSION depend on the weights? ----
# Every policy's actions are fixed; only the scoring changes. No retraining.
WEIGHT_SCENARIOS = {
    "v3 (used)":        W_USED,
    "entropy-derived":  W_ENT,
    "equal":            np.array([0.25, 0.25, 0.25, 0.25]),
    "latency-first":    np.array([0.40, 0.35, 0.15, 0.10]),
    "energy-first":     np.array([0.30, 0.10, 0.45, 0.15]),
    "cost-first":       np.array([0.30, 0.10, 0.10, 0.50]),
    "deadline-first":   np.array([0.60, 0.20, 0.10, 0.10]),
}

set_seed(0)
_ag_ws, _, _m_ws = train_one("lsh_grl_hpfoo", 0, verbose=False, topk=TOPK_BEST)
ACTIONS = {"lsh_grl_hpfoo": _m_ws["actions"]}
for _k, _fn in [("greedy", greedy_latency_action), ("random", ref_policies["random"]),
                ("local_only", ref_policies["local_only"]),
                ("all_cloud", ref_policies["all_cloud"]),
                ("oracle", myopic_oracle_action)]:
    np.random.seed(0)
    ACTIONS[_k] = run_policy(eval_env, test_stream, _fn, collect=True)["_raw"]["A"]

TERM_M = {k: term_matrix(0, test_stream, a) for k, a in ACTIONS.items()}
ws_rows = []
for name, w in WEIGHT_SCENARIOS.items():
    w = np.asarray(w, dtype=float); w = w/w.sum()
    row = {"scenario": name}
    for k, M in TERM_M.items():
        row[LABELS[k]] = float((M @ w).mean())
    order = sorted([k for k in TERM_M if k != "oracle"],
                   key=lambda k: -(TERM_M[k] @ w).mean())
    row["winner"] = LABELS[order[0]]
    ws_rows.append(row)
WEIGHTSENS = pd.DataFrame(ws_rows)
display(WEIGHTSENS.round(4))

_wins = WEIGHTSENS["winner"].value_counts()
print("\nWEIGHT SENSITIVITY")
print("  proposed method ranks first in %d of %d weight scenarios"
      % (int((WEIGHTSENS["winner"] == LABELS["lsh_grl_hpfoo"]).sum()), len(WEIGHTSENS)))
print("  the ranking reverses only where it does not rank first - report these scenarios explicitly")
RESULTS["weight_sensitivity"] = WEIGHTSENS.to_dict("records")

,scenario,LSH-GRL-HPFOO (Proposed),Greedy (min-latency),Random Offloading,Local-Only,All-to-Cloud,Myopic Oracle (bound),winner
0,v3 (used),0.8757,0.8196,0.7283,0.6365,0.5102,0.8812,LSH-GRL-HPFOO (Proposed)
1,entropy-derived,0.8118,0.8466,0.6851,0.4931,0.7079,0.8143,Greedy (min-latency)
2,equal,0.8602,0.8271,0.7117,0.5222,0.5614,0.8630,LSH-GRL-HPFOO (Proposed)
3,latency-first,0.8524,0.8575,0.7169,0.5119,0.6866,0.8559,Greedy (min-latency)
4,energy-first,0.8918,0.8632,0.7338,0.3821,0.6475,0.8920,LSH-GRL-HPFOO (Proposed)
5,cost-first,0.8770,0.7899,0.7261,0.7321,0.3979,0.8841,LSH-GRL-HPFOO (Proposed)
6,deadline-first,0.8830,0.8802,0.7494,0.5642,0.7246,0.8887,LSH-GRL-HPFOO (Proposed)



WEIGHT SENSITIVITY
  proposed method ranks first in 5 of 7 weight scenarios
  the ranking reverses only where it does not rank first - report these scenarios explicitly


---
## §13 — Client requirement 2: Pufferfish step size

Three perturbation-radius schedules compared head to head on identical seeds. `legacy` reproduces the v3 behaviour, in which `t` never reset and the radius collapsed to ~0.04 on a span of 6.0 — so the defensive-inflation phase was inert.

In [27]:
# ============================================================================
# REQUIREMENT 2 - Pufferfish step size, in depth
# ============================================================================
# The v3 code used (ub-lb)/t with `t` monotonic across the WHOLE training run.
# Because optimise() fires every 250 steps and adds `iters` to t each time, the
# radius never took its coarse global value and had collapsed to ~0.04 on a
# search span of 6.0 by late training - i.e. the defensive-inflation phase was
# effectively inert. The three schedules below are compared head to head.

print("SCHEDULE SHAPE (span = %.1f, iters = %d)" % (6.0, 6))
print("-"*72)
_hdr = "  %-10s" % "invocation" + "".join("  tau=%d" % t for t in range(1, 7))
for sch in ["legacy", "reset", "annealed"]:
    print("\n  schedule: %s" % sch); print(_hdr)
    _probe = HPFOO(seed=0, schedule=sch)
    for k in [0, 4, 24]:
        _probe.n_calls = k; _probe.t = 1 + 6*k
        vals = []
        for tau in range(1, 7):
            if sch == "legacy": _probe.t += 1
            vals.append(_probe.step_radius(tau))
        print("  %-10d" % (k+1) + "".join("%7.3f" % v for v in vals))
print("-"*72)

# ---- head-to-head: same seeds, same everything, only the schedule changes ----
SCHED_SEEDS = P["SCHED_SEEDS"]
sched_rows, SCHED_RAW = [], {}
RADIUS_LOG = {}       # radius actually applied in one training run (seed SCHED_SEEDS[0]), for Fig. 13
for sch in ["legacy", "reset", "annealed"]:
    rs, ms_, lat_, miss_ = [], [], [], []
    for sd in SCHED_SEEDS:
        _ags, _, m = train_one("lsh_grl_hpfoo", sd, verbose=False, topk=TOPK_BEST,
                               hpfoo_schedule=sch)
        if sd == SCHED_SEEDS[0]: RADIUS_LOG[sch] = list(_ags.hpfoo.radius_log)
        rs.append(m["reward"]); lat_.append(m["latency"]); miss_.append(m["miss"]*100)
    SCHED_RAW[sch] = rs
    sched_rows.append(dict(schedule=sch, reward=float(np.mean(rs)),
                           std=float(np.std(rs, ddof=1)),
                           latency=float(np.mean(lat_)), miss_pct=float(np.mean(miss_))))
    print("  %-9s reward %.4f +- %.4f" % (sch, np.mean(rs), np.std(rs, ddof=1)))
SCHEDULE = pd.DataFrame(sched_rows); display(SCHEDULE.round(4))

print("\nPAIRED t-TESTS vs legacy (n=%d seeds)" % len(SCHED_SEEDS))
for sch in ["reset", "annealed"]:
    t, p = sps.ttest_rel(SCHED_RAW[sch], SCHED_RAW["legacy"])
    print("  %-9s t=%7.3f  p=%.4f  %s"
          % (sch, t, p, "significant" if p < 0.05 else "n.s."))
_best_sch = SCHEDULE.loc[SCHEDULE.reward.idxmax(), "schedule"]
print("\n  best schedule: %s   (notebook default: %s)" % (_best_sch, PUFFERFISH_SCHEDULE))
RESULTS["pufferfish_schedule"] = SCHEDULE.to_dict("records")
RESULTS["pufferfish_schedule_raw"] = {k: [float(x) for x in v] for k, v in SCHED_RAW.items()}
RESULTS["pufferfish_schedule_used"] = PUFFERFISH_SCHEDULE
RESULTS["pufferfish_radius_log_seed0"] = RADIUS_LOG

SCHEDULE SHAPE (span = 6.0, iters = 6)
------------------------------------------------------------------------

  schedule: legacy
  invocation  tau=1  tau=2  tau=3  tau=4  tau=5  tau=6
  1           3.000  2.000  1.500  1.200  1.000  0.857
  5           0.231  0.222  0.214  0.207  0.200  0.194
  25          0.041  0.041  0.041  0.040  0.040  0.040

  schedule: reset
  invocation  tau=1  tau=2  tau=3  tau=4  tau=5  tau=6
  1           6.000  3.000  2.000  1.500  1.200  1.000
  5           6.000  3.000  2.000  1.500  1.200  1.000
  25          6.000  3.000  2.000  1.500  1.200  1.000

  schedule: annealed
  invocation  tau=1  tau=2  tau=3  tau=4  tau=5  tau=6
  1           6.000  3.000  2.000  1.500  1.200  1.000
  5           2.683  1.342  0.894  0.671  0.537  0.447
  25          1.200  0.600  0.400  0.300  0.240  0.200
------------------------------------------------------------------------
  legacy    reward 0.8727 +- 0.0039
  reset     reward 0.8724 +- 0.0038
  annealed  reward 0.8

,schedule,reward,std,latency,miss_pct
0,legacy,0.8727,0.0039,104.9487,6.72
1,reset,0.8724,0.0038,113.5931,6.73
2,annealed,0.8715,0.0035,105.1226,6.79



PAIRED t-TESTS vs legacy (n=5 seeds)
  reset     t= -0.338  p=0.7526  n.s.
  annealed  t= -2.322  p=0.0809  n.s.

  best schedule: legacy   (notebook default: legacy)


---
## §14 — Client requirement 3: parameter sensitivity of the equations

Part A decomposes end-to-end latency term by term under one-factor-at-a-time sweeps. Part B gives the reward gradients analytically. Part C perturbs the world around a frozen trained policy.

In [28]:
# ============================================================================
# REQUIREMENT 3 - How the equation terms move with their parameters
# ============================================================================
# Part A: analytical decomposition. For a fixed task, sweep one parameter at a
# time and report each LATENCY term, so the write-up can state which term
# dominates in which regime rather than asserting it.
#   l = B/R_up + B/C_j + tau_j + Q_j/kappa_j + c/kappa_j + 0.05B/(2 R_up)
# R_up = W log2(1+SNR)  -> mobility enters LOGARITHMICALLY, payload LINEARLY.

_probe_env = OffloadingEnv(seed=0, sites=sites, deadline_scale=dl_scale); _probe_env.reset()
_E1 = 1
def latency_terms(cyc, byts, mob, node=_E1, backlog=0.0):
    e = _probe_env
    bits = byts*8.0
    r_up = e.uplink_rate(mob)
    return dict(uplink=bits/r_up*1000, backhaul=bits/e.backhaul[node]*1000,
                prop=e.prop[node], queue=backlog/e.cap[node]*1000,
                compute=cyc/e.cap[node]*1000,
                downlink=(bits*RESULT_RATIO)/(2*r_up)*1000)

sens_rows = []
_base = dict(cyc=5e8, byts=1e5, mob=0.05, backlog=0.0)
SWEEPS = [("data_size_bytes", "byts", [1e3, 1e4, 1e5, 1e6, 1e7, 2e7]),
          ("cpu_cycles",      "cyc",  [5e6, 5e7, 5e8, 2e9, 3e9]),
          ("mobility_index",  "mob",  [0.05, 0.2, 0.45, 0.8]),
          ("node_backlog",    "backlog", [0.0, 1e9, 5e9, 2e10, 5e10])]
for pname, key, vals in SWEEPS:
    for v in vals:
        kw = dict(_base); kw[key] = v
        T = latency_terms(**kw); tot = sum(T.values())
        dom = max(T, key=T.get)
        sens_rows.append(dict(parameter=pname, value=v, total_ms=tot,
                              dominant_term=dom, dominant_share=T[dom]/tot, **T))
PARAMSENS = pd.DataFrame(sens_rows)
display(PARAMSENS.round(3))
print("\nREGIME MAP (which latency term dominates)")
for pname, g in PARAMSENS.groupby("parameter", sort=False):
    print("  %-18s %s" % (pname, " -> ".join(
        "%s@%.3g" % (r.dominant_term, r.value) for r in g.itertuples())))
RESULTS["param_sensitivity_latency"] = PARAMSENS.to_dict("records")

,parameter,value,total_ms,dominant_term,dominant_share,uplink,backhaul,prop,queue,compute,downlink
0,data_size_bytes,1.000000e+03,50.402,compute,0.917,0.010,0.001,4.189,0.000,46.202,0.000
1,data_size_bytes,1.000000e+04,50.498,compute,0.915,0.096,0.008,4.189,0.000,46.202,0.002
2,data_size_bytes,1.000000e+05,51.458,compute,0.898,0.963,0.080,4.189,0.000,46.202,0.024
3,data_size_bytes,1.000000e+06,61.059,compute,0.757,9.628,0.800,4.189,0.000,46.202,0.241
4,data_size_bytes,1.000000e+07,157.075,uplink,0.613,96.277,8.000,4.189,0.000,46.202,2.407
5,data_size_bytes,2.000000e+07,263.758,uplink,0.730,192.554,16.000,4.189,0.000,46.202,4.814
6,cpu_cycles,5.000000e+06,5.718,prop,0.733,0.963,0.080,4.189,0.000,0.462,0.024
7,cpu_cycles,5.000000e+07,9.876,compute,0.468,0.963,0.080,4.189,0.000,4.620,0.024
8,cpu_cycles,5.000000e+08,51.458,compute,0.898,0.963,0.080,4.189,0.000,46.202,0.024
9,cpu_cycles,2.000000e+09,190.063,compute,0.972,0.963,0.080,4.189,0.000,184.807,0.024



REGIME MAP (which latency term dominates)
  data_size_bytes    compute@1e+03 -> compute@1e+04 -> compute@1e+05 -> compute@1e+06 -> uplink@1e+07 -> uplink@2e+07
  cpu_cycles         prop@5e+06 -> compute@5e+07 -> compute@5e+08 -> compute@2e+09 -> compute@3e+09
  mobility_index     compute@0.05 -> compute@0.2 -> compute@0.45 -> compute@0.8
  node_backlog       compute@0 -> queue@1e+09 -> queue@5e+09 -> queue@2e+10 -> queue@5e+10


In [29]:
# Part B: reward gradients. Away from the deadline boundary the reward is smooth
# and linear in each term; AT the boundary the success term contributes a step of
# W_SUCCESS. That discontinuity is what makes the policy deadline-SEEKING rather
# than latency-minimising, and it is the cleanest explanation of the greedy result.
print("REWARD RESPONSE")
print("-"*60)
print("  dr/dl   (l < D)      = -W_LAT / D          = %+.5f per ms at D=100ms"
      % (-W_LAT/100))
print("  dr/dl   at l = D     = -W_SUCCESS (step)   = %+.4f" % (-W_SUCCESS))
print("  dr/dE                = -W_ENERGY / E_local")
print("  dr/dcost             = -W_COST / cost_cloud")
print("  bound                : r in [0,1], weights sum to %.2f"
      % (W_SUCCESS+W_LAT+W_ENERGY+W_COST))

# Part C (V5): end-to-end robustness with the trained policy held FIXED is now run for every
# seed inside §9 (run_seed -> rec["ofat"]); the multi-seed table OFAT_DF is shown here again.
print("\nPERTURBED CONDITIONS - multi-seed (from §9c)")
display(OFAT_DF[["parameter", "value", "proposed", "proposed_sd", "greedy", "oracle", "miss_pct",
                 "gap_closed", "p_vs_greedy_holm", "seeds_above_oracle"]].round(4))

REWARD RESPONSE
------------------------------------------------------------
  dr/dl   (l < D)      = -W_LAT / D          = -0.00150 per ms at D=100ms
  dr/dl   at l = D     = -W_SUCCESS (step)   = -0.3500
  dr/dE                = -W_ENERGY / E_local
  dr/dcost             = -W_COST / cost_cloud
  bound                : r in [0,1], weights sum to 1.00

PERTURBED CONDITIONS - multi-seed (from §9c)


,parameter,value,proposed,proposed_sd,greedy,oracle,miss_pct,gap_closed,p_vs_greedy_holm,seeds_above_oracle
0,deadline_scale,0.60,0.8357,0.0042,0.7924,0.8487,13.485,76.9197,0.0000,0/10
1,deadline_scale,0.80,0.8588,0.0042,0.8122,0.8681,9.290,83.2626,0.0000,0/10
2,deadline_scale,1.00,0.8736,0.0045,0.8248,0.8818,6.760,85.5728,0.0000,0/10
3,deadline_scale,1.30,0.8876,0.0040,0.8384,0.8954,4.380,86.2444,0.0000,0/10
4,deadline_scale,1.60,0.8939,0.0070,0.8457,0.9040,2.840,82.7524,0.0000,0/10
5,interarrival,0.40,0.8401,0.0164,0.8217,0.8330,7.780,162.9053,0.0080,7/10
6,interarrival,0.50,0.8544,0.0142,0.8227,0.8415,7.330,168.6526,0.0005,9/10
7,interarrival,0.75,0.8696,0.0045,0.8240,0.8710,6.905,96.9875,0.0000,3/10
8,interarrival,1.00,0.8736,0.0045,0.8248,0.8818,6.760,85.5728,0.0000,0/10
9,interarrival,1.50,0.8755,0.0043,0.8256,0.8861,6.690,82.4834,0.0000,0/10


In [30]:
# ---- save the three new requirement tables ----
WEIGHTSENS.to_csv(os.path.join(OUT_DIR, "table_weight_sensitivity.csv"), index=False)
SCHEDULE.to_csv(os.path.join(OUT_DIR, "table_pufferfish_schedule.csv"), index=False)
PARAMSENS.to_csv(os.path.join(OUT_DIR, "table_param_sensitivity_latency.csv"), index=False)
OFAT_DF.to_csv(os.path.join(OUT_DIR, "table_param_sensitivity_ofat.csv"), index=False)
pd.DataFrame(RESULTS["reward_weights"]).to_csv(
    os.path.join(OUT_DIR, "table_reward_weights.csv"), index=False)
print("requirement tables written:")
for _f in ["table_reward_weights.csv", "table_weight_sensitivity.csv",
           "table_pufferfish_schedule.csv", "table_param_sensitivity_latency.csv",
           "table_param_sensitivity_ofat.csv"]:
    print("   ", _f)

requirement tables written:
    table_reward_weights.csv
    table_weight_sensitivity.csv
    table_pufferfish_schedule.csv
    table_param_sensitivity_latency.csv
    table_param_sensitivity_ofat.csv


## §15 — Refiner comparison: HPFOO vs GA, PSO, GWO (reviewer item 03)

The same seeds as §9; HPFOO and "no refiner" rows are taken from the §9 run (identical code path, so they are
not retrained). Overhead is reported as optimiser wall time per call and per training run, alongside decision time.

In [31]:
# ===== §15: HPFOO vs GA / PSO / GWO in the identical pipeline (reviewer 03) =====
REF_ALGOS = ["ga", "pso", "gwo"]
REFINER_RECS = {}
if RUN["refiners"]:
    def _run_refiners(sd):
        out = {}
        for alg in REF_ALGOS:
            _, _, m = train_one("lsh_grl_" + alg, sd, verbose=False, topk=TOPK_BEST)
            out[alg] = _slim(m)
        return out
    for sd in P["REFINER_SEEDS"]:
        t0 = time.time()
        REFINER_RECS[sd] = cached("refiners_seed_%d" % sd, lambda sd=sd: _run_refiners(sd))
        print("refiners seed %d done (%.0fs)" % (sd, time.time()-t0))

    _rs = [sd for sd in P["REFINER_SEEDS"] if sd in SEED_RECS]       # HPFOO comes from §9, same seeds
    def RV(alg, metric="reward"):
        if alg == "hpfoo": return np.array([SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"][metric] for sd in _rs], float)
        if alg == "none":  return np.array([SEED_RECS[sd]["learned"]["lsh_grl"][metric] for sd in _rs], float)
        return np.array([REFINER_RECS[sd][alg][metric] for sd in _rs], float)

    # equal-budget check on the real runs: evaluations per optimise() call must be identical
    _epc = {a: RV(a, "fitness_evals")/np.maximum(RV(a, "opt_calls"), 1) for a in ["hpfoo"] + REF_ALGOS}
    print("fitness evaluations per optimiser call:", {a: float(np.mean(v)) for a, v in _epc.items()})
    assert len({round(float(np.mean(v)), 6) for v in _epc.values()}) == 1, "unequal optimiser budgets"

    _rows, _ps = [], []
    for alg, nm in [("hpfoo", "HPFOO (proposed)"), ("ga", "GA"), ("pso", "PSO"), ("gwo", "GWO"),
                    ("none", "No refiner (LSH-GRL)")]:
        row = dict(refiner=nm, reward=RV(alg).mean(), reward_sd=RV(alg).std(ddof=1) if len(_rs) > 1 else 0.0,
                   latency=RV(alg, "latency").mean(), miss_pct=100*RV(alg, "miss").mean(),
                   cost_ratio=RV(alg, "cost_ratio").mean(), decision_ms=RV(alg, "decision_ms").mean(),
                   train_s=RV(alg, "train_s").mean())
        if alg != "none":
            row.update(opt_total_s=RV(alg, "opt_time_s").mean(),
                       opt_ms_per_call=1000*(RV(alg, "opt_time_s")/np.maximum(RV(alg, "opt_calls"), 1)).mean(),
                       evals_per_call=float(np.mean(_epc[alg])))
        if alg != "hpfoo":
            r = paired(RV("hpfoo"), RV(alg))
            row.update(hpfoo_minus_this=r["diff"], ci_lo=r["ci_lo"], ci_hi=r["ci_hi"], p_raw=r["p"],
                       hpfoo_wins="%d/%d" % (r["wins"], r["n"]))
            _ps.append(r["p"])
        _rows.append(row)
    REFINER_TBL = pd.DataFrame(_rows)
    _h = iter(holm(_ps))
    REFINER_TBL["p_holm"] = [np.nan if r == "HPFOO (proposed)" else next(_h) for r in REFINER_TBL.refiner]
    print("\nREFINER COMPARISON (%d seeds, same shortlist, reward, fitness and evaluation budget)" % len(_rs))
    display(REFINER_TBL.round(5))
    RESULTS["refiners"] = REFINER_TBL.to_dict("records")
    REFINER_TBL.to_csv(os.path.join(OUT_DIR, "table_refiners.csv"), index=False)
else:
    print("refiner comparison switched off (RUN['refiners'] = False)")

refiners seed 0 done (209s)
refiners seed 1 done (210s)
refiners seed 2 done (211s)
refiners seed 3 done (210s)
refiners seed 4 done (212s)
refiners seed 5 done (213s)
refiners seed 6 done (212s)
refiners seed 7 done (211s)
refiners seed 8 done (210s)
refiners seed 9 done (210s)
fitness evaluations per optimiser call: {'hpfoo': 182.0, 'ga': 182.0, 'pso': 182.0, 'gwo': 182.0}

REFINER COMPARISON (10 seeds, same shortlist, reward, fitness and evaluation budget)


,refiner,reward,reward_sd,latency,miss_pct,cost_ratio,decision_ms,train_s,opt_total_s,opt_ms_per_call,evals_per_call,hpfoo_minus_this,ci_lo,ci_hi,p_raw,hpfoo_wins,p_holm
0,HPFOO (proposed),0.87359,0.00449,106.71773,6.760,0.14536,2.30671,50.30420,0.46088,19.20313,182.0,NaN,NaN,NaN,NaN,NaN,NaN
1,GA,0.87353,0.00419,110.17293,6.885,0.14296,2.30322,51.10800,0.57231,23.84638,182.0,0.00006,-0.00247,0.00260,0.95522,5/10,1.00000
2,PSO,0.87277,0.00507,105.91114,7.025,0.14452,2.30548,50.86098,0.39975,16.65618,182.0,0.00082,-0.00182,0.00346,0.49979,7/10,1.00000
3,GWO,0.87269,0.00575,104.31505,6.830,0.14753,2.31025,50.86964,0.40848,17.01990,182.0,0.00090,-0.00189,0.00368,0.48528,6/10,1.00000
4,No refiner (LSH-GRL),0.86993,0.00504,105.71601,7.470,0.14795,1.37767,41.28597,NaN,NaN,NaN,0.00366,0.00152,0.00580,0.00380,8/10,0.01519


## §16 — Generalisation to an unseen workload source (reviewer item 06)

Four folds. Each fold trains and validates on three traces (80/20) and tests on the fourth, which is never seen
during training, checkpointing or tuning. The compute-intensity scale, arrival rate, deadline scale and LSH
feature standardisation are all refitted on the training traces only. The arrival rate calibrated on the
training traces is kept for the test trace, so the held-out source also shifts the load the policy sees (a genuine
out-of-distribution test). The shortlist size stays at the value selected in §7a; α is re-selected on each fold's
validation split.

In [32]:
# ===== §16: leave-one-source-out generalisation (reviewer 06) =====
LOSO_KINDS = ["lsh_grl_hpfoo", "grl", "gat_ppo", "dqn"]
LOSO_REFS = ["greedy", "maxmin", "random", "oracle"]

def loso_data(held):
    """Train/validate on the other three traces, test on `held`. Every calibration that
    looks at data (compute scale, arrival rate, deadline scale, LSH standardisation) is
    refitted on the training traces only."""
    tr = df[df.source != held].copy(); te = df[df.source == held].copy()
    assert len(te) > 0 and not set(tr.task_id) & set(te.task_id)
    med_ms = tr["cpu_cycles_raw"].median()/CAP_RANGE[LOCAL][0]*1000
    sc = TARGET_LOCAL_MEDIAN_MS/med_ms
    for d in (tr, te): d["cpu_cycles"] = (d["cpu_cycles_raw"]*sc).clip(5e6, 1e10)
    tr = tr.reset_index(drop=True); te = te.reset_index(drop=True)
    ia_f, dls_f = calibrate(tr, sites, verbose=False)
    env_f = OffloadingEnv(seed=0, sites=sites, deadline_scale=dls_f)
    k = int(len(tr)*0.8)
    trs = make_stream(tr.iloc[:k], env_f, ia_f)
    vas = make_stream(tr.iloc[k:].reset_index(drop=True), env_f, ia_f)
    tes = make_stream(te, env_f, ia_f)
    fb = np.array([task_features(t, env_f) for t in trs], dtype=np.float32)
    return dict(train=trs, val=vas, test=tes, dls=dls_f, fb=fb, sites=sites, ia=ia_f, cpu_scale=sc)

LOSO_RECS = {}
if RUN["loso"]:
    for held in SOURCES:
        D = loso_data(held)
        print("held-out %-16s train %d | val %d | test %d | cpu scale %.2f | ia %.2f ms | deadline scale %.3f"
              % (held, len(D["train"]), len(D["val"]), len(D["test"]), D["cpu_scale"], D["ia"], D["dls"]))
        for sd in P["LOSO_SEEDS"]:
            def _run(D=D, sd=sd):
                res = {}
                for k in LOSO_KINDS:
                    _, _, m = train_one(k, sd, verbose=False, topk=TOPK_BEST, data=D); res[k] = _slim(m)
                for k in LOSO_REFS:
                    np.random.seed(sd)
                    res[k] = _slim(run_policy(OffloadingEnv(seed=sd, sites=sites, deadline_scale=D["dls"]),
                                              D["test"], ref_policies[k]))
                return res
            t0 = time.time()
            LOSO_RECS[(held, sd)] = cached("loso_%s_seed_%d" % (held, sd), _run)
            print("   seed %d (%.0fs)  proposed %.4f | greedy %.4f | oracle %.4f"
                  % (sd, time.time()-t0, LOSO_RECS[(held, sd)]["lsh_grl_hpfoo"]["reward"],
                     LOSO_RECS[(held, sd)]["greedy"]["reward"], LOSO_RECS[(held, sd)]["oracle"]["reward"]))
    _lr = []
    for held in SOURCES:
        g = {k: np.array([LOSO_RECS[(held, sd)][k]["reward"] for sd in P["LOSO_SEEDS"]])
             for k in LOSO_KINDS + LOSO_REFS}
        r = paired(g["lsh_grl_hpfoo"], g["greedy"]); r2 = paired(g["lsh_grl_hpfoo"], g["gat_ppo"])
        row = dict(held_out=held, **{LABELS[k]: g[k].mean() for k in LOSO_KINDS + LOSO_REFS})
        row.update(headroom_pct=100*(g["oracle"].mean()-g["greedy"].mean())/g["greedy"].mean(),
                   gap_closed=100*(g["lsh_grl_hpfoo"].mean()-g["greedy"].mean())/(g["oracle"].mean()-g["greedy"].mean()),
                   p_vs_greedy=r["p"], wins_vs_greedy="%d/%d" % (r["wins"], r["n"]),
                   p_vs_gat_ppo=r2["p"],
                   proposed_miss_pct=100*np.mean([LOSO_RECS[(held, sd)]["lsh_grl_hpfoo"]["miss"] for sd in P["LOSO_SEEDS"]]),
                   greedy_miss_pct=100*np.mean([LOSO_RECS[(held, sd)]["greedy"]["miss"] for sd in P["LOSO_SEEDS"]]))
        _lr.append(row)
    LOSO_TBL = pd.DataFrame(_lr)
    LOSO_TBL["p_vs_greedy_holm"] = holm(LOSO_TBL.p_vs_greedy)
    print("\nLEAVE-ONE-SOURCE-OUT (mean test reward on the unseen trace, %d seeds per fold)" % len(P["LOSO_SEEDS"]))
    display(LOSO_TBL.round(4))
    print("in-distribution reference (§9, same seeds): proposed gap closed %.1f%%" % RESULTS["gap_closed_multiseed_pct"])
    RESULTS["loso"] = LOSO_TBL.to_dict("records")
    LOSO_TBL.to_csv(os.path.join(OUT_DIR, "table_loso_generalisation.csv"), index=False)
else:
    print("leave-one-source-out switched off (RUN['loso'] = False)")

held-out task_offloading  train 4800 | val 1200 | test 2000 | cpu scale 0.59 | ia 7.69 ms | deadline scale 1.960
   seed 0 (162s)  proposed 0.8375 | greedy 0.8413 | oracle 0.8614
   seed 1 (162s)  proposed 0.8262 | greedy 0.8355 | oracle 0.8573
   seed 2 (162s)  proposed 0.8374 | greedy 0.8391 | oracle 0.8612
   seed 3 (161s)  proposed 0.8321 | greedy 0.8385 | oracle 0.8598
   seed 4 (162s)  proposed 0.8311 | greedy 0.8384 | oracle 0.8609
held-out vec_edge         train 4800 | val 1200 | test 2000 | cpu scale 3.44 | ia 36.81 ms | deadline scale 3.581
   seed 0 (161s)  proposed 0.8142 | greedy 0.6914 | oracle 0.8467
   seed 1 (161s)  proposed 0.8264 | greedy 0.6843 | oracle 0.8404
   seed 2 (162s)  proposed 0.8112 | greedy 0.6822 | oracle 0.8447
   seed 3 (161s)  proposed 0.8342 | greedy 0.6958 | oracle 0.8460
   seed 4 (161s)  proposed 0.8345 | greedy 0.6892 | oracle 0.8520
held-out azure_llm        train 4800 | val 1200 | test 2000 | cpu scale 0.50 | ia 7.10 ms | deadline scale 2.079


,held_out,LSH-GRL-HPFOO (Proposed),GRL only,GAT-PPO (graph RL),Standard DQN,Greedy (min-latency),Max-min fair,Random Offloading,Myopic Oracle (bound),headroom_pct,gap_closed,p_vs_greedy,wins_vs_greedy,p_vs_gat_ppo,proposed_miss_pct,greedy_miss_pct,p_vs_greedy_holm
0,task_offloading,0.8329,0.8295,0.8149,0.7422,0.8386,0.8510,0.7237,0.8601,2.5682,-26.5619,0.0130,0/5,0.2883,9.98,6.57,0.0261
1,vec_edge,0.8241,0.8148,0.8001,0.6613,0.6886,0.8220,0.6898,0.8460,22.8541,86.1110,0.0000,5/5,0.0174,12.64,8.02,0.0000
2,azure_llm,0.9409,0.9361,0.8858,0.8932,0.9458,0.9407,0.8284,0.9475,0.1810,-285.3074,0.0247,0/5,0.0688,0.00,0.00,0.0261
3,borg,0.6766,0.6704,0.6196,0.3386,0.5834,0.7174,0.4849,0.7352,26.0243,61.4148,0.0002,5/5,0.0085,26.98,12.35,0.0006


in-distribution reference (§9, same seeds): proposed gap closed 85.6%


## §17 — Computational complexity and scalability (reviewer item 07)

Notation: $N$ = number of machines (19 in the main study), $H$ = attention heads (4), $d$ = embedding size (32),
$h$ = MLP width (128), $k$ = shortlist size (3), $T$ = LSH tables (4), $b$ = hash bits (12),
$c$ = bucket cap (40), $P$ = refiner population (14), $I$ = refiner iterations (6), $W$ = fitness window (2,000),
$B$ = training batch (128), $C$ = replay capacity (20,000).

| Stage | Time per decision | Notes |
|---|---|---|
| Graph-attention encoder (2 layers, dense mask) | $O(N^2 H d)$ | dominant term for large $N$ |
| Per-node Q scoring (MLP) | $O(N h^2)$ | |
| Shortlist (sort of $N$ scores) | $O(N \log N)$ | |
| LSH lookup | $O(T b D + T c D)$ | independent of the stored history size |
| Refiner utility on the shortlist | $O(6k)$ | |
| **Total inference** | $O(N^2 H d + N h^2)$ | |
| Refiner search (every 250 decisions) | $O((1+2I)\,P\,W\cdot 6)$ | amortised $O((1+2I)P W \cdot 6/250)$ per decision |
| One Double-DQN update | $O(B (N^2 H d + N h^2))$ | every 2nd decision |

Memory: replay $O(C\,N\,8)$ node-feature floats (two states per transition), LSH $O(T\cdot\text{entries}\cdot D)$,
network parameters independent of $N$ (the scoring MLP is shared across nodes).

The cells below measure the same quantities. They time decisions, training and memory as the network grows from
19 to 145 machines (edge:fog:cloud kept at 10:5:3, arrival rate rescaled so that edge utilisation stays at 90%).
They also measure throughput as the number of tasks grows, and training time against the training budget.
Peak memory is the peak resident-set growth of the Python process during the run, sampled every 20 ms, plus
peak CUDA memory when a GPU is used.

In [33]:
# ===== §17: scalability measurements =====
import threading
try:
    import psutil
except ImportError:
    psutil = None

class PeakRSS:
    """Peak growth of resident memory (MB) while the block runs, sampled every 20 ms."""
    def __enter__(self):
        self.peak = self.base = (psutil.Process().memory_info().rss if psutil else 0)
        self._stop = False
        if psutil:
            self._t = threading.Thread(target=self._run, daemon=True); self._t.start()
        if torch.cuda.is_available(): torch.cuda.reset_peak_memory_stats()
        return self
    def _run(self):
        p = psutil.Process()
        while not self._stop:
            self.peak = max(self.peak, p.memory_info().rss); time.sleep(0.02)
    def __exit__(self, *exc):
        self._stop = True
        if psutil:
            self._t.join(); self.peak = max(self.peak, psutil.Process().memory_info().rss)
        self.cuda_mb = (torch.cuda.max_memory_allocated()/2**20) if torch.cuda.is_available() else 0.0
    @property
    def mb(self): return (self.peak - self.base)/2**20

_DEFAULT_TOPO = (NUM_EDGE, NUM_FOG, NUM_CLOUD)

def set_topology(ne, nf, nc):
    """Rebind the topology globals used by the environment, features, prices and policies."""
    global NUM_EDGE, NUM_FOG, NUM_CLOUD, N_NODES, TIER, _PC, _PB
    NUM_EDGE, NUM_FOG, NUM_CLOUD = ne, nf, nc
    N_NODES = 1 + ne + nf + nc
    TIER = np.array([LOCAL] + [EDGE]*ne + [FOG]*nf + [CLOUD]*nc)
    _PC = np.array([PRICE_CYCLE[t] for t in TIER]); _PB = np.array([PRICE_BIT[t] for t in TIER])

def run_scale(topo, sd):
    set_topology(*topo)
    try:
        st = sites if topo == _DEFAULT_TOPO else load_sites_n(max(3*topo[0], 30))
        probe = OffloadingEnv(seed=0, sites=st)
        ia_t = (df["cpu_cycles"].mean()/probe.cap[1:1+NUM_EDGE].sum())*1000.0/TARGET_EDGE_UTIL
        env_t = OffloadingEnv(seed=0, sites=st, deadline_scale=dl_scale)
        trs = make_stream(TRAIN_DF, env_t, ia_t); vas = make_stream(VAL_DF, env_t, ia_t)
        tes = make_stream(TEST_DF, env_t, ia_t)
        fb = np.array([task_features(t, env_t) for t in trs], dtype=np.float32)
        D = dict(train=trs, val=vas, test=tes, dls=dl_scale, fb=fb, sites=st)
        with PeakRSS() as mem:
            ag, _, m = train_one("lsh_grl_hpfoo", sd, verbose=False, topk=TOPK_BEST, data=D)
        e = OffloadingEnv(seed=sd, sites=st, deadline_scale=dl_scale)
        t0 = time.perf_counter(); mm = evaluate(ag, e, tes); wall = time.perf_counter()-t0
        np.random.seed(sd)
        g = run_policy(OffloadingEnv(seed=sd, sites=st, deadline_scale=dl_scale), tes, greedy_latency_action)
        o = run_policy(OffloadingEnv(seed=sd, sites=st, deadline_scale=dl_scale), tes, myopic_oracle_action)
        return dict(topology="%d/%d/%d" % topo, nodes=N_NODES, seed=sd, interarrival_ms=ia_t,
                    train_s=m["train_s"], decision_ms=mm["decision_ms"], decision_p95_ms=mm["decision_p95_ms"],
                    decisions_per_s=1000.0/mm["decision_ms"], sim_tasks_per_s=len(tes)/wall,
                    peak_rss_mb=mem.mb, peak_cuda_mb=mem.cuda_mb, params=m["params"],
                    lsh_entries=m.get("lsh_entries", 0), reward=mm["reward"], miss=mm["miss"],
                    greedy=g["reward"], oracle=o["reward"], greedy_decision_ms=g["decision_ms"],
                    oracle_decision_ms=o["decision_ms"])
    finally:
        set_topology(*_DEFAULT_TOPO)

def run_taskcount():
    ag, _, _ = train_one("lsh_grl_hpfoo", SEEDS[0], verbose=False, topk=TOPK_BEST)
    rows = []
    for n in P["TASK_COUNTS"]:
        # n jobs re-sampled from the held-out test jobs with fresh Poisson arrivals at the same load
        sub = TEST_DF.sample(n=n, replace=True, random_state=n).reset_index(drop=True)
        st = make_stream(sub, env, ia_ms)
        for name, fn in [("proposed", None), ("greedy", greedy_latency_action), ("oracle", myopic_oracle_action)]:
            e = OffloadingEnv(seed=SEEDS[0], sites=sites, deadline_scale=dl_scale)
            with PeakRSS() as mem:
                t0 = time.perf_counter()
                m = evaluate(ag, e, st) if fn is None else run_policy(e, st, fn)
                wall = time.perf_counter()-t0
            rows.append(dict(method=name, tasks=n, wall_s=wall, tasks_per_s=n/wall,
                             decision_ms=m["decision_ms"], reward=m["reward"], peak_rss_mb=mem.mb))
    return rows

def run_budget():
    rows = []
    for E in P["BUDGET_EPISODES"]:
        old = P["EPISODES"]; P["EPISODES"] = E
        try:
            with PeakRSS() as mem:
                _, _, m = train_one("lsh_grl_hpfoo", SEEDS[0], verbose=False, topk=TOPK_BEST)
        finally:
            P["EPISODES"] = old
        rows.append(dict(episodes=E, training_decisions=E*P["EP_LEN"], train_s=m["train_s"],
                         reward=m["reward"], peak_rss_mb=mem.mb, peak_cuda_mb=mem.cuda_mb))
    return rows

if RUN["scalability"]:
    _sc = []
    for topo in P["TOPOLOGIES"]:
        for sd in P["SCALE_SEEDS"]:
            t0 = time.time()
            _sc.append(cached("scale_%d_%d_%d_seed_%d" % (topo + (sd,)), lambda topo=topo, sd=sd: run_scale(topo, sd)))
            print("topology %s seed %d: %.0fs  decision %.3f ms  train %.0f s"
                  % (topo, sd, time.time()-t0, _sc[-1]["decision_ms"], _sc[-1]["train_s"]))
    assert N_NODES == 1 + sum(_DEFAULT_TOPO), "topology was not restored"
    SCALE_RAW = pd.DataFrame(_sc)
    SCALE_TBL = SCALE_RAW.groupby("nodes").mean(numeric_only=True).reset_index()
    SCALE_TBL["gap_closed"] = 100*(SCALE_TBL.reward-SCALE_TBL.greedy)/(SCALE_TBL.oracle-SCALE_TBL.greedy)
    print("\nSCALING WITH NETWORK SIZE (mean over %d seeds)" % len(P["SCALE_SEEDS"]))
    display(SCALE_TBL.round(4))
    TASKCOUNT_TBL = pd.DataFrame(cached("taskcount", run_taskcount))
    print("\nTHROUGHPUT WITH TASK COUNT"); display(TASKCOUNT_TBL.round(4))
    BUDGET_TBL = pd.DataFrame(cached("budget", run_budget))
    print("\nTRAINING TIME WITH TRAINING BUDGET"); display(BUDGET_TBL.round(4))

    def slope(x, y):
        x, y = np.log(np.asarray(x, float)), np.log(np.asarray(y, float))
        return float(np.polyfit(x, y, 1)[0]) if len(x) > 1 else float("nan")
    EXPONENTS = dict(
        decision_time_vs_nodes=slope(SCALE_TBL.nodes, SCALE_TBL.decision_ms),
        training_time_vs_nodes=slope(SCALE_TBL.nodes, SCALE_TBL.train_s),
        training_time_vs_decisions=slope(BUDGET_TBL.training_decisions, BUDGET_TBL.train_s),
        wall_time_vs_tasks=slope(TASKCOUNT_TBL[TASKCOUNT_TBL.method == "proposed"].tasks,
                                 TASKCOUNT_TBL[TASKCOUNT_TBL.method == "proposed"].wall_s))
    print("\nEMPIRICAL COMPLEXITY EXPONENTS (slope of log-log fit; 1 = linear, 2 = quadratic)")
    for k, v in EXPONENTS.items(): print("  %-28s %.2f" % (k, v))
    RESULTS["scalability"] = dict(nodes=SCALE_TBL.to_dict("records"), tasks=TASKCOUNT_TBL.to_dict("records"),
                                  budget=BUDGET_TBL.to_dict("records"), exponents=EXPONENTS,
                                  device=str(DEV), torch=torch.__version__)
    SCALE_RAW.to_csv(os.path.join(OUT_DIR, "table_scalability_nodes_per_seed.csv"), index=False)
    SCALE_TBL.to_csv(os.path.join(OUT_DIR, "table_scalability_nodes.csv"), index=False)
    TASKCOUNT_TBL.to_csv(os.path.join(OUT_DIR, "table_scalability_tasks.csv"), index=False)
    BUDGET_TBL.to_csv(os.path.join(OUT_DIR, "table_training_budget.csv"), index=False)
else:
    print("scalability switched off (RUN['scalability'] = False)")

topology (10, 5, 3) seed 0: 75s  decision 2.284 ms  train 51 s
topology (10, 5, 3) seed 1: 75s  decision 2.320 ms  train 51 s
topology (10, 5, 3) seed 2: 75s  decision 2.282 ms  train 51 s
topology (20, 10, 6) seed 0: 76s  decision 2.288 ms  train 51 s
topology (20, 10, 6) seed 1: 76s  decision 2.318 ms  train 51 s
topology (20, 10, 6) seed 2: 76s  decision 2.314 ms  train 52 s
topology (40, 20, 12) seed 0: 78s  decision 2.314 ms  train 53 s
topology (40, 20, 12) seed 1: 78s  decision 2.315 ms  train 53 s
topology (40, 20, 12) seed 2: 77s  decision 2.318 ms  train 52 s
topology (80, 40, 24) seed 0: 124s  decision 2.320 ms  train 99 s
topology (80, 40, 24) seed 1: 125s  decision 2.323 ms  train 99 s
topology (80, 40, 24) seed 2: 125s  decision 2.298 ms  train 99 s

SCALING WITH NETWORK SIZE (mean over 3 seeds)


,nodes,seed,interarrival_ms,train_s,decision_ms,decision_p95_ms,decisions_per_s,sim_tasks_per_s,peak_rss_mb,peak_cuda_mb,params,lsh_entries,reward,miss,greedy,oracle,greedy_decision_ms,oracle_decision_ms,gap_closed
0,19,1.0,11.8877,50.6756,2.2953,2.5520,435.6916,416.6091,0.7630,32.7363,24321.0,9949.0000,0.8711,0.0712,0.8182,0.8791,0.0218,0.0556,86.7815
1,37,1.0,6.3643,51.2691,2.3068,2.5584,433.5138,413.0588,0.2630,57.1885,24321.0,9992.0000,0.8780,0.0650,0.8238,0.8861,0.0220,0.0563,86.9554
2,73,1.0,3.0876,52.6367,2.3159,2.5850,431.7988,409.2043,0.5729,141.2290,24321.0,9932.6667,0.8791,0.0663,0.8261,0.8876,0.0226,0.0570,86.2060
3,145,1.0,1.5416,99.1604,2.3136,2.5778,432.2293,404.7022,8.6432,448.2954,24321.0,9834.0000,0.8801,0.0618,0.8236,0.8876,0.0254,0.0634,88.3191



THROUGHPUT WITH TASK COUNT


,method,tasks,wall_s,tasks_per_s,decision_ms,reward,peak_rss_mb
0,proposed,1000,2.4271,412.0111,2.3204,0.8807,0.0039
1,greedy,1000,0.0615,16265.9538,0.0268,0.8265,0.0039
2,oracle,1000,0.0984,10165.4805,0.0616,0.8875,0.0039
3,proposed,2000,4.9052,407.7321,2.3463,0.8719,0.0039
4,greedy,2000,0.1206,16585.6738,0.0267,0.8152,0.0039
5,oracle,2000,0.1909,10474.0349,0.0604,0.8764,0.0039
6,proposed,4000,9.7150,411.7365,2.3232,0.8775,0.0039
7,greedy,4000,0.2347,17043.9341,0.0261,0.8207,0.0156
8,oracle,4000,0.3796,10537.5860,0.0604,0.8828,0.0039
9,proposed,8000,19.3161,414.1625,2.3101,0.8747,0.0156



TRAINING TIME WITH TRAINING BUDGET


,episodes,training_decisions,train_s,reward,peak_rss_mb,peak_cuda_mb
0,6,1500,11.9864,0.8759,0.0039,32.7363
1,12,3000,24.6855,0.8756,0.0039,32.7363
2,25,6250,51.7944,0.8757,0.0195,32.7363
3,50,12500,106.6103,0.8751,0.0195,32.7363



EMPIRICAL COMPLEXITY EXPONENTS (slope of log-log fit; 1 = linear, 2 = quadratic)
  decision_time_vs_nodes       0.00
  training_time_vs_nodes       0.30
  training_time_vs_decisions   1.03
  wall_time_vs_tasks           1.00


## §10 — Figures (1000 DPI)

Every figure is drawn from a variable computed above.

In [34]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size":9,"axes.titlesize":11,"axes.labelsize":10,
                     "legend.fontsize":8,"figure.dpi":110,"savefig.bbox":"tight"})
SAVED = []
def save(fig, name):
    p = os.path.join(PLOTS_DIR, name)
    fig.savefig(p, dpi=PLOT_DPI, format="png"); plt.close(fig); SAVED.append(p)
    print("  saved %s (%d dpi)" % (name, PLOT_DPI))

# 1 - physics validation (Gate 1)
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
pv = pd.DataFrame(RESULTS["physics_table"]).T[["local","edge","fog","cloud"]]
pv.plot(kind="bar", ax=ax, logy=True, width=0.75,
        color=["#5D4037","#1565C0","#EF6C00","#455A64"], edgecolor="black", linewidth=0.4)
ax.set_ylabel("End-to-end latency (ms, log)"); ax.set_xlabel("Task archetype")
ax.set_title("Physics validation: no single tier dominates all task types")
ax.legend(title="Target tier"); ax.grid(axis="y", alpha=0.3); plt.xticks(rotation=0)
save(fig, "output1_physics_validation.png")

# 2 - oracle headroom
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
order = ["all_cloud","local_only","random","dqn","greedy","grl","lsh_grl","lsh_grl_hpfoo","oracle"]
vals = [ALL[k]["reward"] for k in order]
bars = ax.bar([LABELS[k] for k in order], vals,
              color=[COLORS.get(k,"#888") for k in order], edgecolor="black", linewidth=0.5)
ax.axhline(ALL["oracle"]["reward"], ls="--", c="black", lw=1)
ax.axhline(ALL["greedy"]["reward"], ls=":", c="#6A1B9A", lw=1)
ax.annotate("", xy=(7.4, ALL["oracle"]["reward"]), xytext=(6.4, ALL["greedy"]["reward"]),
            arrowprops=dict(arrowstyle="<->", color="red", lw=1.4))
ax.text(7.5, (ALL["oracle"]["reward"]+ALL["greedy"]["reward"])/2,
        "headroom\n%.1f%%" % RESULTS["headroom_pct"], color="red", fontsize=8, va="center")
for b, v in zip(bars, vals): ax.text(b.get_x()+b.get_width()/2, v+.006, "%.3f"%v, ha="center", fontsize=7)
ax.set_ylabel("Mean reward (held-out)"); ax.set_ylim(0, max(vals)*1.15)
ax.set_title("Oracle headroom: LSH-GRL-HPFOO closes %.0f%% of the greedy-to-oracle gap" % RESULTS["gap_closed_pct"])
plt.xticks(rotation=22, ha="right"); ax.grid(axis="y", alpha=0.3)
save(fig, "output2_headroom.png")

# 3 - convergence (held-out evaluation, epsilon = 0)
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
for k, cur in CURVES.items():
    ax.plot([c[0] for c in cur], [c[1] for c in cur], marker="o", ms=3.5,
            lw=1.6, color=COLORS.get(k), label=LABELS[k])
ax.axhline(ALL["greedy"]["reward"], ls=":", c=COLORS["greedy"], lw=1.2, label=LABELS["greedy"])
ax.axhline(ALL["oracle"]["reward"], ls="--", c="black", lw=1.2, label=LABELS["oracle"])
ax.set_xlabel("Training epoch"); ax.set_ylabel("Held-out reward (frozen, $\\epsilon=0$)")
ax.set_title("Convergence measured on held-out tasks, not on training-phase averages")
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
save(fig, "output3_convergence.png")

# 4 - Pareto frontier
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
for k in order:
    ax.scatter(ALL[k]["latency"], ALL[k]["energy"], s=180 if k=="lsh_grl_hpfoo" else 80,
               marker="*" if k=="lsh_grl_hpfoo" else "o", color=COLORS.get(k,"#888"),
               edgecolor="black", linewidth=0.6, zorder=3, label=LABELS[k])
ax.set_xscale("log"); ax.set_xlabel("Mean end-to-end latency (ms, log)")
ax.set_ylabel("Mean device energy (J)")
ax.set_title("Latency-energy trade-off (lower-left is better)")
ax.legend(fontsize=7); ax.grid(alpha=0.3)
save(fig, "output4_pareto.png")

# 5 - mobility impact
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
for k in ["lsh_grl_hpfoo","grl","greedy","random","local_only","all_cloud"]:
    if k in MOBILITY.columns:
        ax.plot(MOBILITY.index, MOBILITY[k], marker="s", ms=4, lw=1.6,
                color=COLORS.get(k), label=LABELS[k])
ax.set_xlabel("Device mobility / channel quality"); ax.set_ylabel("Mean reward")
ax.set_title("Robustness to mobility (Objective 4)"); ax.legend(fontsize=7); ax.grid(alpha=0.3)
save(fig, "output5_mobility.png")

# 6 - LSH scalability (Objective 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(PLOT_FIGSIZE[0], PLOT_FIGSIZE[1]*0.85))
a1.plot(LSH_SCALE.N, LSH_SCALE.exhaustive_ms, marker="o", ms=4, color="#C62828", label="Exhaustive search")
a1.plot(LSH_SCALE.N, LSH_SCALE.lsh_ms, marker="s", ms=4, color="#1565C0", label="LSH retrieval")
a1.set_xscale("log"); a1.set_yscale("log"); a1.set_xlabel("Indexed tasks $N$")
a1.set_ylabel("Query time (ms, log)"); a1.set_title("Query cost"); a1.legend(fontsize=7); a1.grid(alpha=0.3)
a2.plot(LSH_SCALE.N, LSH_SCALE.recall_at_10*100, marker="o", ms=4, color="#2E7D32")
a2.set_xscale("log"); a2.set_xlabel("Indexed tasks $N$"); a2.set_ylabel("Recall@10 vs exhaustive (%)")
a2.set_ylim(0, 105); a2.set_title("Retrieval quality"); a2.grid(alpha=0.3)
fig.suptitle("Objective 1: LSH clustering - %.1fx speedup at N=%d, %.0f%% recall"
             % (LSH_SCALE.speedup.iloc[-1], LSH_SCALE.N.iloc[-1], 100*LSH_SCALE.recall_at_10.mean()))
save(fig, "output6_lsh_scalability.png")

# 7 - ablation across all objectives
_metrics = [("Reward","higher is better"), ("Latency (ms)","lower is better"),
            ("Energy (J)","lower is better"), ("Miss (%)","lower is better")]
_short = [c.replace(" (proposed)","*").replace(" (task feats only)","")
           .replace(" heuristic (no learning)","") for c in ABLATION["Configuration"]]
fig, axes = plt.subplots(2, 2, figsize=(PLOT_FIGSIZE[0], PLOT_FIGSIZE[1]*1.3))
for axi, (col, note) in zip(axes.ravel(), _metrics):
    d = ABLATION.assign(_s=_short).sort_values(col, ascending=(col == "Reward"))
    cols = ["#0D47A1" if "proposed" in c else "#90A4AE" for c in d["Configuration"]]
    bars = axi.barh(d["_s"], d[col], color=cols, edgecolor="black", linewidth=0.4)
    for bb, v in zip(bars, d[col]):
        axi.text(v, bb.get_y()+bb.get_height()/2, " %.3f" % v, va="center", fontsize=6)
    if col == "Reward": axi.set_xlim(d[col].min()*0.92, d[col].max()*1.06)
    else: axi.set_xlim(0, d[col].max()*1.30)
    axi.set_title("%s (%s)" % (col, note), fontsize=9)
    axi.tick_params(labelsize=6); axi.grid(axis="x", alpha=0.3)
fig.suptitle("Ablation across all objectives (* = proposed)")
fig.tight_layout()
save(fig, "output7_ablation.png")

# 8 - multi-seed with real error bars
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
order8 = ["all_cloud","local_only","random","dqn","gat_ppo","greedy","maxmin","no_graph","grl","lsh_grl","lsh_grl_hpfoo","oracle"]
ms = MULTI.loc[[LABELS[k] for k in order8]]
ax.bar(ms.index, ms["mean"], yerr=ms["std"], capsize=4,
       color=[COLORS.get(k,"#888") for k in order8], edgecolor="black", linewidth=0.5)
ax.set_ylabel("Mean reward"); ax.set_ylim(0, ms["mean"].max()*1.15)
ax.set_title("Multi-seed validation (n=%d seeds, mean $\\pm$ s.d.)" % len(SEEDS))
plt.xticks(rotation=22, ha="right"); ax.grid(axis="y", alpha=0.3)
save(fig, "output8_multiseed.png")

# 9 - topology heat map on real sites
fig, ax = plt.subplots(figsize=PLOT_FIGSIZE)
acts = np.array(LEARNED["lsh_grl_hpfoo"]["actions"])
counts = np.array([(acts==i).sum() for i in range(N_NODES)])
if env.site_coords is not None:
    sc = ax.scatter(env.site_coords[:,1], env.site_coords[:,0],
                    s=40+520*counts[1:1+NUM_EDGE]/max(counts[1:1+NUM_EDGE].max(),1),
                    c=counts[1:1+NUM_EDGE], cmap="YlOrRd", edgecolor="black", linewidth=0.6)
    plt.colorbar(sc, ax=ax, label="Tasks assigned")
    ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
    ax.set_title("LSH-GRL-HPFOO offloading load across real edge sites (site.csv)")
else:
    ax.bar(range(N_NODES), counts, color="#1565C0"); ax.set_xlabel("Node index")
ax.grid(alpha=0.3)
save(fig, "output9_topology.png")

# 10 - summary table
fig, ax = plt.subplots(figsize=(PLOT_FIGSIZE[0]*1.45, 0.42*len(MAIN)+1.4))
ax.axis("off")
disp = MAIN.round(4).reset_index(); disp.columns = ["Method"] + list(MAIN.columns)
_w = [0.30] + [0.70/(len(disp.columns)-1)]*(len(disp.columns)-1)
tb = ax.table(cellText=disp.values, colLabels=disp.columns, cellLoc="center",
              loc="center", colWidths=_w, bbox=[0, 0, 1, 0.93])
tb.auto_set_font_size(False); tb.set_fontsize(8)
for (r, c), cell in tb.get_celld().items():
    cell.set_linewidth(0.5)
    if r == 0:
        cell.set_facecolor("#1565C0"); cell.set_text_props(color="white", weight="bold")
    else:
        if c == 0:
            cell.set_text_props(weight="bold", ha="left"); cell.PAD = 0.03
        if disp.iloc[r-1, 0] == LABELS["lsh_grl_hpfoo"]:
            cell.set_facecolor("#E3F2FD")
ax.set_title("Comparative Analysis: Performance Summary (held-out, $\\epsilon=0$)",
             pad=14, fontsize=11)
save(fig, "output10_summary_table.png")

# 11 - sensitivity
fig, (a1, a2) = plt.subplots(1, 2, figsize=(PLOT_FIGSIZE[0]*1.3, PLOT_FIGSIZE[1]),
                             constrained_layout=True)
a1b = a1.twinx()
l1, = a1.plot(TOPK.topk, TOPK.val_reward, marker="o", ms=5, color="#2E7D32", label="Validation reward")
l2, = a1.plot(TOPK.topk, TOPK.reward, marker="s", ms=5, ls="--", color="#0D47A1", label="Test reward")
l3, = a1b.plot(TOPK.topk, TOPK.reduction_pct, marker="^", ms=5, ls=":", color="#EF6C00",
               label="Search-space reduction")
a1.axvline(RESULTS["topk_selected"], color="red", ls=":", lw=1.0)
a1.set_xticks(TOPK.topk.tolist())
a1.set_xlabel("Candidates evaluated (topk)"); a1.set_ylabel("Reward")
a1b.set_ylabel("Search-space reduction (%)", color="#EF6C00", labelpad=10)
a1b.tick_params(axis="y", colors="#EF6C00", labelsize=8)
a1.margins(y=0.30); a1b.margins(y=0.30)
a1.set_title("Candidate-set size (selected: %d)" % RESULTS["topk_selected"], fontsize=10)
a1.legend(handles=[l1, l2, l3], fontsize=7, loc="lower right", framealpha=0.95)
a1.grid(alpha=0.3)

a2.plot(ALPHA.alpha, ALPHA.val_reward, marker="o", ms=5, color="#2E7D32", label="Validation")
a2.plot(ALPHA.alpha, ALPHA.test_reward, marker="s", ms=5, ls="--", color="#0D47A1", label="Test")
a2.axvline(RESULTS["alpha_selected"], color="red", ls=":", lw=1.0,
           label="selected = %.2f" % RESULTS["alpha_selected"])
a2.set_xticks(ALPHA.alpha.tolist())
a2.set_xlabel("Fusion weight $\\alpha$  (1 = pure Q-network)"); a2.set_ylabel("Reward")
a2.margins(y=0.30)
a2.set_title("Q-network / HPFOO-utility fusion", fontsize=10)
a2.legend(fontsize=7, loc="lower left", framealpha=0.95); a2.grid(alpha=0.3)

fig.suptitle("Sensitivity analyses (all selection on validation)", fontsize=11)
save(fig, "output11_sensitivity.png")

  saved output1_physics_validation.png (1000 dpi)
  saved output2_headroom.png (1000 dpi)
  saved output3_convergence.png (1000 dpi)
  saved output4_pareto.png (1000 dpi)
  saved output5_mobility.png (1000 dpi)
  saved output6_lsh_scalability.png (1000 dpi)
  saved output7_ablation.png (1000 dpi)
  saved output8_multiseed.png (1000 dpi)
  saved output9_topology.png (1000 dpi)
  saved output10_summary_table.png (1000 dpi)
  saved output11_sensitivity.png (1000 dpi)


### §10b — Manuscript figures (publication style, V5)

Figures 3, 9, 12, 13, 14 and 15 replace the V4 versions with multi-seed data. Figures 16–21 are new.

In [35]:
# ===== §10b: manuscript figures (publication style) =====
# Rules: serif font (DejaVu Serif), 1000 DPI, bold axis titles, legend below the x-axis,
# no figure title, full border, spacing via subplots_adjust, file name
# "Figure_<n> – <Topic>.png" saved in the plots folder. Numbers follow the manuscript:
# 3, 9, 12-15 replace the V4 versions; 16-21 are new.
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def pub_style():
    plt.rcdefaults()
    plt.rcParams.update({"font.family": "serif", "font.serif": ["DejaVu Serif"], "font.size": 10,
                         "axes.labelsize": 11, "axes.labelweight": "bold", "legend.fontsize": 8.5,
                         "xtick.labelsize": 9, "ytick.labelsize": 9, "axes.spines.top": True,
                         "axes.spines.right": True, "axes.linewidth": 0.9})

def pub_save(fig, num, topic):
    name = "Figure_%s – %s.png" % (num, topic)
    fig.savefig(os.path.join(PLOTS_DIR, name), dpi=PLOT_DPI)
    plt.close(fig); print("  saved", name)

def legend_below(fig, handles, labels, ncol=4, y=0.01):
    fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, y), ncol=ncol, frameon=False)

def panel(ax, letter):
    ax.text(0.01, 0.98, "(%s)" % letter, transform=ax.transAxes, va="top", ha="left", fontweight="bold")

pub_style()
_sd = lambda x: np.std(x, ddof=1) if len(x) > 1 else 0.0

# ---- Figure 3: multi-seed reward of every method ----
_o3 = ["all_cloud", "local_only", "random", "dqn", "gat_ppo", "greedy", "maxmin", "no_graph", "grl",
       "lsh_grl", "lsh_grl_hpfoo"]
fig, ax = plt.subplots(figsize=(9, 5.2))
_m = [np.mean(multi[k]) for k in _o3]; _s = [_sd(multi[k]) for k in _o3]
bars = ax.bar(range(len(_o3)), _m, yerr=_s, capsize=3, color=[COLORS[k] for k in _o3],
              edgecolor="black", linewidth=0.5)
_ol = ax.axhline(np.mean(multi["oracle"]), ls="--", c="black", lw=1.1)
ax.set_xticks([]); ax.set_ylabel("Mean held-out reward"); ax.set_xlabel("Method")
ax.set_ylim(0, max(_m)*1.12)
for i, (m_, s_) in enumerate(zip(_m, _s)):
    ax.text(i, m_ + s_ + 0.01, "%.3f" % m_, ha="center", fontsize=7.5)
fig.subplots_adjust(left=0.09, right=0.98, top=0.97, bottom=0.30)
legend_below(fig, list(bars) + [_ol], [LABELS[k] for k in _o3] + [LABELS["oracle"]], ncol=4)
pub_save(fig, 3, "Multi-Seed Reward Comparison")

# ---- Figure 9: mobility over seeds ----
fig, ax = plt.subplots(figsize=(8, 5))
_lv = [l for l, _ in MOB_LEVELS]; hs = []
for k in ["lsh_grl_hpfoo", "grl", "greedy", "maxmin", "random", "local_only", "all_cloud", "oracle"]:
    mu = np.array([MOB_SEEDS[l][k].mean() for l in _lv]); sdv = np.array([_sd(MOB_SEEDS[l][k]) for l in _lv])
    h, = ax.plot(_lv, mu, marker="o", ms=4, lw=1.5, color=COLORS[k], ls="--" if k == "oracle" else "-")
    ax.fill_between(_lv, mu-sdv, mu+sdv, color=COLORS[k], alpha=0.15, lw=0); hs.append(h)
ax.set_xlabel("Device mobility level"); ax.set_ylabel("Mean held-out reward")
fig.subplots_adjust(left=0.1, right=0.98, top=0.97, bottom=0.27)
legend_below(fig, hs, [LABELS[k] for k in ["lsh_grl_hpfoo", "grl", "greedy", "maxmin", "random",
                                            "local_only", "all_cloud", "oracle"]], ncol=4)
pub_save(fig, 9, "Mobility Robustness")

# ---- Figure 12: reward weights ----
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8))
_x = np.arange(4); _w = 0.38
b1 = a1.bar(_x-_w/2, W_USED, _w, color="#0D47A1", edgecolor="black", linewidth=0.5)
b2 = a1.bar(_x+_w/2, W_ENT, _w, color="#90A4AE", edgecolor="black", linewidth=0.5)
for i in range(4): a1.text(_x[i]+_w/2, W_ENT[i]+0.01, "H=%.3f" % _E[i], ha="center", fontsize=7.5)
a1.set_xticks(_x); a1.set_xticklabels(["Deadline", "Latency", "Energy", "Cost"])
a1.set_xlabel("Reward term"); a1.set_ylabel("Weight"); panel(a1, "a")
_ws = WEIGHTSENS.copy()
_ws["margin"] = _ws[LABELS["lsh_grl_hpfoo"]] - _ws[LABELS["greedy"]]
_band = float(np.std(np.array(multi["lsh_grl_hpfoo"]) - np.array(multi["greedy"]), ddof=1)) if len(SEEDS) > 1 else 0.005
_ws = _ws.sort_values("margin")
a2.barh(_ws.scenario, _ws.margin, color=["#0D47A1" if v > _band else ("#C62828" if v < -_band else "#9E9E9E")
                                         for v in _ws.margin], edgecolor="black", linewidth=0.5)
a2.axvspan(-_band, _band, color="#E0E0E0", zorder=0); a2.axvline(0, c="black", lw=0.8)
a2.set_xlabel("Reward margin (proposed − greedy)"); a2.set_ylabel("Weighting"); panel(a2, "b")
fig.subplots_adjust(left=0.07, right=0.98, top=0.95, bottom=0.21, wspace=0.42)
legend_below(fig, [b1, b2, Patch(color="#E0E0E0")],
             ["Adopted weights", "Entropy-derived weights", "± s.d. of seed-to-seed margin (tie band)"], ncol=3)
pub_save(fig, 12, "Reward Weight Sensitivity")

# ---- Figure 13: Pufferfish step-size schedules ----
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8))
hs = []
for sch, c in [("legacy", "#0D47A1"), ("reset", "#EF6C00"), ("annealed", "#2E7D32")]:
    if sch in RADIUS_LOG and RADIUS_LOG[sch]:
        h, = a1.plot(RADIUS_LOG[sch], lw=1.0, color=c); hs.append(h)
a1.set_yscale("log"); a1.set_xlabel("Pufferfish iteration (whole training run)")
a1.set_ylabel("Perturbation radius"); panel(a1, "a")
a2.bar(SCHEDULE.schedule, SCHEDULE.reward, yerr=SCHEDULE["std"], capsize=4,
       color=["#0D47A1", "#EF6C00", "#2E7D32"], edgecolor="black", linewidth=0.5)
a2.set_ylim(SCHEDULE.reward.min()-0.02, SCHEDULE.reward.max()+0.015)
a2.set_xlabel("Schedule"); a2.set_ylabel("Mean held-out reward"); panel(a2, "b")
fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.2, wspace=0.3)
legend_below(fig, hs, ["Legacy (adopted)", "Reset", "Annealed"][:len(hs)], ncol=3)
pub_save(fig, 13, "Pufferfish Step-Size Schedules")

# ---- Figure 14: latency decomposition ----
_terms = ["uplink", "backhaul", "prop", "queue", "compute", "downlink"]
_tc = ["#1565C0", "#90CAF9", "#455A64", "#C62828", "#EF6C00", "#7CB342"]
fig, axs = plt.subplots(1, 2, figsize=(10, 4.8))
for ax, (pn, xl), letter in zip(axs, [("data_size_bytes", "Data size (bytes)"),
                                      ("node_backlog", "Queued work at server (cycles)")], "ab"):
    d = PARAMSENS[PARAMSENS.parameter == pn]
    bottom = np.zeros(len(d)); hs = []
    for t, c in zip(_terms, _tc):
        h = ax.bar(range(len(d)), d[t].values, bottom=bottom, color=c, edgecolor="black", linewidth=0.3)
        bottom += d[t].values; hs.append(h)
    ax.set_yscale("log"); ax.set_xticks(range(len(d)))
    ax.set_xticklabels(["%.0e" % v for v in d.value], rotation=30)
    ax.set_xlabel(xl); ax.set_ylabel("End-to-end latency (ms)"); panel(ax, letter)
fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.26, wspace=0.3)
legend_below(fig, hs, ["Uplink", "Backhaul", "Propagation", "Queueing", "Compute", "Downlink"], ncol=6)
pub_save(fig, 14, "Latency Term Decomposition")

# ---- Figure 15: perturbed operating conditions over seeds ----
fig, axs = plt.subplots(1, 3, figsize=(12, 4.6))
for ax, (pn, xl), letter in zip(axs, [("deadline_scale", "Deadline multiplier"),
                                      ("interarrival", "Arrival-gap multiplier"),
                                      ("snr_offset_db", "Radio quality offset (dB)")], "abc"):
    g = _of[_of.parameter == pn].groupby("value"); hs = []
    for k, c, ls in [("proposed", COLORS["lsh_grl_hpfoo"], "-"), ("greedy", COLORS["greedy"], "-"),
                     ("maxmin", COLORS["maxmin"], "-"), ("oracle", "black", "--")]:
        mu, sdv = g[k].mean(), g[k].std(ddof=1).fillna(0)
        h, = ax.plot(mu.index, mu.values, marker="o", ms=4, lw=1.4, color=c, ls=ls)
        ax.fill_between(mu.index, mu-sdv, mu+sdv, color=c, alpha=0.15, lw=0); hs.append(h)
    ax.set_xlabel(xl); ax.set_ylabel("Mean held-out reward"); panel(ax, letter)
fig.subplots_adjust(left=0.06, right=0.98, top=0.95, bottom=0.25, wspace=0.32)
legend_below(fig, hs, [LABELS["lsh_grl_hpfoo"], LABELS["greedy"], LABELS["maxmin"], LABELS["oracle"]], ncol=4)
pub_save(fig, 15, "Perturbed Operating Conditions")

# ---- Figure 16: fairness ----
_fk = ["lsh_grl_hpfoo", "greedy", "maxmin", "gat_ppo", "oracle"]
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4.8))
_x = np.arange(len(SOURCES)); _w = 0.8/len(_fk); hs = []
for i, k in enumerate(_fk):
    vals = np.array([[SEED_RECS[sd][("learned" if k in LEARNER_KINDS else "ref")][k]["miss_by_source"].get(g, np.nan)
                      for g in SOURCES] for sd in SEEDS])*100
    h = a1.bar(_x + (i-(len(_fk)-1)/2)*_w, np.nanmean(vals, 0), _w,
               yerr=np.nanstd(vals, 0, ddof=1) if len(SEEDS) > 1 else None, capsize=2,
               color=COLORS[k], edgecolor="black", linewidth=0.4); hs.append(h)
a1.set_xticks(_x); a1.set_xticklabels(["Cellular", "Vehicular", "Azure LLM", "Borg"])
a1.set_xlabel("Tenant (workload source)"); a1.set_ylabel("Deadline miss rate (%)"); panel(a1, "a")
_jm = [("jain_tenant_success", "Tenant\nsuccess"), ("jain_server_util", "Server\nload"), ("jain_edge_util", "Edge\nload")]
_x2 = np.arange(len(_jm))
for i, k in enumerate(_fk):
    a2.bar(_x2 + (i-(len(_fk)-1)/2)*_w, [np.mean(V(k, m)) for m, _ in _jm], _w,
           yerr=[_sd(V(k, m)) for m, _ in _jm], capsize=2, color=COLORS[k], edgecolor="black", linewidth=0.4)
a2.set_xticks(_x2); a2.set_xticklabels([l for _, l in _jm]); a2.set_ylim(0, 1.08)
a2.set_xlabel("Jain's fairness index"); a2.set_ylabel("Index value (1 = perfectly fair)"); panel(a2, "b")
fig.subplots_adjust(left=0.07, right=0.98, top=0.95, bottom=0.27, wspace=0.28)
legend_below(fig, hs, [LABELS[k] for k in _fk], ncol=5)
pub_save(fig, 16, "Fairness Across Tenants and Servers")

# ---- Figure 17: refiner comparison ----
if RUN["refiners"] and len(REFINER_RECS):
    _rk = [("hpfoo", "HPFOO", COLORS["lsh_grl_hpfoo"]), ("ga", "GA", COLORS["lsh_grl_ga"]),
           ("pso", "PSO", COLORS["lsh_grl_pso"]), ("gwo", "GWO", COLORS["lsh_grl_gwo"]),
           ("none", "No refiner", COLORS["lsh_grl"])]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8))
    hs = [a1.bar(i, RV(a).mean(), yerr=_sd(RV(a)), capsize=4, color=c, edgecolor="black", linewidth=0.5)
          for i, (a, _, c) in enumerate(_rk)]
    _lo = min(RV(a).mean()-_sd(RV(a)) for a, _, _ in _rk)
    a1.set_ylim(_lo-0.01, max(RV(a).mean()+_sd(RV(a)) for a, _, _ in _rk)+0.006)
    a1.set_xticks([]); a1.set_xlabel("Refinement stage"); a1.set_ylabel("Mean held-out reward"); panel(a1, "a")
    for i, (a, _, c) in enumerate(_rk[:4]):
        a2.bar(i, 1000*(RV(a, "opt_time_s")/np.maximum(RV(a, "opt_calls"), 1)).mean(), color=c,
               edgecolor="black", linewidth=0.5)
    a2.set_xticks([]); a2.set_xlabel("Refinement stage"); a2.set_ylabel("Optimiser time per call (ms)"); panel(a2, "b")
    fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.2, wspace=0.3)
    legend_below(fig, hs, [n for _, n, _ in _rk], ncol=5)
    pub_save(fig, 17, "Refiner Comparison")

# ---- Figure 18: leave-one-source-out ----
if RUN["loso"] and len(LOSO_RECS):
    _lk = ["lsh_grl_hpfoo", "grl", "gat_ppo", "dqn", "greedy", "maxmin", "oracle"]
    fig, ax = plt.subplots(figsize=(10, 5)); _x = np.arange(len(SOURCES)); _w = 0.8/len(_lk); hs = []
    for i, k in enumerate(_lk):
        vals = np.array([[LOSO_RECS[(g, sd)][k]["reward"] for sd in P["LOSO_SEEDS"]] for g in SOURCES])
        hs.append(ax.bar(_x + (i-(len(_lk)-1)/2)*_w, vals.mean(1), _w,
                         yerr=vals.std(1, ddof=1) if vals.shape[1] > 1 else None, capsize=2,
                         color=COLORS[k], edgecolor="black", linewidth=0.4))
    ax.set_xticks(_x); ax.set_xticklabels(["Cellular", "Vehicular", "Azure LLM", "Borg"])
    ax.set_xlabel("Held-out workload source (never seen in training)"); ax.set_ylabel("Mean test reward")
    ax.set_ylim(0, 1.0)
    fig.subplots_adjust(left=0.08, right=0.98, top=0.97, bottom=0.25)
    legend_below(fig, hs, [LABELS[k] for k in _lk], ncol=4)
    pub_save(fig, 18, "Leave-One-Source-Out Generalisation")

# ---- Figures 19-20: scalability ----
if RUN["scalability"]:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8))
    h1, = a1.plot(SCALE_TBL.nodes, SCALE_TBL.decision_ms, marker="o", color=COLORS["lsh_grl_hpfoo"])
    h2, = a1.plot(SCALE_TBL.nodes, SCALE_TBL.decision_p95_ms, marker="o", ls=":", color=COLORS["lsh_grl_hpfoo"])
    h3, = a1.plot(SCALE_TBL.nodes, SCALE_TBL.greedy_decision_ms, marker="s", color=COLORS["greedy"])
    h4, = a1.plot(SCALE_TBL.nodes, SCALE_TBL.oracle_decision_ms, marker="^", ls="--", color="black")
    a1.set_xscale("log"); a1.set_yscale("log"); a1.set_xlabel("Number of machines")
    a1.set_ylabel("Decision time (ms)"); panel(a1, "a")
    a2.plot(SCALE_TBL.nodes, SCALE_TBL.train_s, marker="o", color=COLORS["lsh_grl_hpfoo"])
    a2.set_xscale("log"); a2.set_yscale("log"); a2.set_xlabel("Number of machines")
    a2.set_ylabel("Training time (s)"); panel(a2, "b")
    fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.2, wspace=0.3)
    legend_below(fig, [h1, h2, h3, h4], ["Proposed (mean)", "Proposed (P95)", "Greedy", "Myopic oracle"], ncol=4)
    pub_save(fig, 19, "Scalability with Network Size")

    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4.8)); hs = []
    for nm, k in [("proposed", "lsh_grl_hpfoo"), ("greedy", "greedy"), ("oracle", "oracle")]:
        d = TASKCOUNT_TBL[TASKCOUNT_TBL.method == nm]
        h, = a1.plot(d.tasks, d.tasks_per_s, marker="o", color=COLORS[k], ls="--" if nm == "oracle" else "-")
        hs.append(h)
    a1.set_xscale("log"); a1.set_yscale("log"); a1.set_xlabel("Number of tasks")
    a1.set_ylabel("Throughput (tasks/s)"); panel(a1, "a")
    a2.plot(BUDGET_TBL.training_decisions, BUDGET_TBL.train_s, marker="o", color=COLORS["lsh_grl_hpfoo"])
    a2.set_xscale("log"); a2.set_yscale("log"); a2.set_xlabel("Training decisions")
    a2.set_ylabel("Training time (s)"); panel(a2, "b")
    fig.subplots_adjust(left=0.08, right=0.98, top=0.95, bottom=0.2, wspace=0.3)
    legend_below(fig, hs, ["Proposed", "Greedy", "Myopic oracle"], ncol=3)
    pub_save(fig, 20, "Throughput and Training Cost")

# ---- Figure 21: arrival-time distributions ----
_ak2 = [("proposed", "lsh_grl_hpfoo"), ("grl", "grl"), ("greedy", "greedy"), ("maxmin", "maxmin"), ("oracle", "oracle")]
fig, ax = plt.subplots(figsize=(9, 4.8)); _x = np.arange(len(ARRIVAL_PATTERNS)); _w = 0.8/len(_ak2); hs = []
for i, (k, lk) in enumerate(_ak2):
    hs.append(ax.bar(_x + (i-(len(_ak2)-1)/2)*_w, [ARR_SEEDS[p][k].mean() for p in ARRIVAL_PATTERNS], _w,
                     yerr=[_sd(ARR_SEEDS[p][k]) for p in ARRIVAL_PATTERNS], capsize=2,
                     color=COLORS[lk], edgecolor="black", linewidth=0.4))
ax.set_xticks(_x); ax.set_xticklabels(["Poisson", "Periodic", "Bursty (H2)", "Heavy-tailed (Pareto)"])
ax.set_xlabel("Job arrival-time distribution (same mean load)"); ax.set_ylabel("Mean held-out reward")
_allv = np.concatenate([ARR_SEEDS[p][k] for p in ARRIVAL_PATTERNS for k, _ in _ak2])
ax.set_ylim(max(0, _allv.min()-0.05), min(1, _allv.max()+0.02))
fig.subplots_adjust(left=0.09, right=0.98, top=0.97, bottom=0.24)
legend_below(fig, hs, [LABELS[lk] for _, lk in _ak2], ncol=5)
pub_save(fig, 21, "Arrival Pattern Robustness")
plt.rcdefaults()

  saved Figure_3 – Multi-Seed Reward Comparison.png
  saved Figure_9 – Mobility Robustness.png
  saved Figure_12 – Reward Weight Sensitivity.png
  saved Figure_13 – Pufferfish Step-Size Schedules.png
  saved Figure_14 – Latency Term Decomposition.png
  saved Figure_15 – Perturbed Operating Conditions.png
  saved Figure_16 – Fairness Across Tenants and Servers.png
  saved Figure_17 – Refiner Comparison.png
  saved Figure_18 – Leave-One-Source-Out Generalisation.png
  saved Figure_19 – Scalability with Network Size.png
  saved Figure_20 – Throughput and Training Cost.png
  saved Figure_21 – Arrival Pattern Robustness.png


## Save results and final report

In [36]:
RESULTS["config"] = dict(profile=PROFILE, n_tasks=len(df), episodes=P["EPISODES"],
    ep_len=P["EP_LEN"], seeds=P["SEEDS"], interarrival_ms=float(ia_ms),
    deadline_scale=float(dl_scale), cpu_scale=float(CPU_SCALE), weights=dict(success=W_SUCCESS, latency=W_LAT,
    energy=W_ENERGY, cost=W_COST), edge_util_target=TARGET_EDGE_UTIL,
    pufferfish_schedule=PUFFERFISH_SCHEDULE, adj_from_seed_env=ADJ_FROM_SEED_ENV,
    code_version=CODE_VERSION, device=str(DEV), torch=torch.__version__,
    python=sys.version.split()[0], numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
    hpfoo=dict(population=14, iterations=6, bounds=[-3, 3], invoked_every_steps=250, fitness_window=2000,
               evals_per_call=182),
    lsh=dict(tables=4, bits=12, sim_threshold=0.99, min_count=2, bucket_cap=40, load_buckets=4),
    dqn=dict(gamma=0.95, lr=3e-4, batch=128, replay=20000, target_update=200, learn_every=2,
             eps_floor=0.05, eps_decay_fraction=0.6, gat_layers=2, heads=HEADS, emb=EMB_DIM, hidden=HIDDEN),
    ppo=dict(gamma=0.95, lam=0.95, clip=0.2, epochs=4, minibatch=64, ent=0.01, vf=0.5, lr=3e-4))
for _k in list(RESULTS["main_comparison"]): RESULTS["main_comparison"][_k].pop("actions", None)
with open(os.path.join(OUT_DIR, "results_all.json"), "w") as f:
    json.dump(RESULTS, f, indent=2, default=lambda o: float(o) if np.isscalar(o) else str(o))
MAIN.to_csv(os.path.join(OUT_DIR, "table_main_comparison.csv"))
ABLATION.to_csv(os.path.join(OUT_DIR, "table_ablation.csv"), index=False)
TOPK.to_csv(os.path.join(OUT_DIR, "table_sensitivity_topk.csv"), index=False)
ALPHA.to_csv(os.path.join(OUT_DIR, "table_sensitivity_alpha.csv"), index=False)
MULTI.to_csv(os.path.join(OUT_DIR, "table_multiseed.csv"))
T4.to_csv(os.path.join(OUT_DIR, "table4_multiseed_tests_holm.csv"), index=False)
COMPONENT_TBL.to_csv(os.path.join(OUT_DIR, "table_component_contributions.csv"), index=False)
FAIRNESS.to_csv(os.path.join(OUT_DIR, "table_fairness.csv"))
FAIR_TESTS.to_csv(os.path.join(OUT_DIR, "table_fairness_tests.csv"), index=False)
TENANT_TBL.to_csv(os.path.join(OUT_DIR, "table_fairness_per_tenant.csv"), index=False)
MOBILITY.to_csv(os.path.join(OUT_DIR, "table_mobility_multiseed.csv"))
MOB_TESTS.to_csv(os.path.join(OUT_DIR, "table_mobility_tests.csv"), index=False)
ARRIVAL_TBL.to_csv(os.path.join(OUT_DIR, "table_arrival_patterns.csv"), index=False)
LSH_SUMMARY.to_csv(os.path.join(OUT_DIR, "table_lsh_inference.csv"), index=False)
LSH_INF.to_csv(os.path.join(OUT_DIR, "table_lsh_inference_per_seed.csv"))
LSH_SCALE.to_csv(os.path.join(OUT_DIR, "table_lsh_scalability.csv"), index=False)
# per-seed record of every method and every measure (for the manuscript's per-seed appendix)
_ps = []
for sd in SEEDS:
    for grp in ["learned", "ref"]:
        for k, m in SEED_RECS[sd][grp].items():
            _ps.append(dict(seed=sd, method=LABELS[k], **{x: y for x, y in m.items()
                                                         if isinstance(y, (int, float)) and not isinstance(y, bool)}))
pd.DataFrame(_ps).to_csv(os.path.join(OUT_DIR, "table_per_seed_all_methods.csv"), index=False)

# ============================ VERIFICATION SUMMARY ============================
CHECKS = []
def chk(name, ok, detail=""):
    CHECKS.append((name, bool(ok), detail))
chk("Gate 1 physics checks", all(RESULTS["gate1"].values()), "%d checks" % len(RESULTS["gate1"]))
chk("Gate 2 headroom >= 5%", RESULTS["headroom_pct"] >= 5, "%.2f%%" % RESULTS["headroom_pct"])
if isinstance(REGRESSION, pd.DataFrame) and len(REGRESSION):
    chk("Regression vs V4 (required values)", REGRESSION[REGRESSION.must_match].ok.all(),
        "%d/%d match" % (REGRESSION[REGRESSION.must_match].ok.sum(), REGRESSION.must_match.sum()))
chk("Evaluation deterministic (Poisson replay = test)", max(_pc) < 1e-9)
chk("Local-only identical on every seed", np.ptp(multi["local_only"]) < 1e-12)
_allr = np.concatenate([np.asarray(v) for v in multi.values()])
chk("All rewards inside [0, 1]", (_allr >= 0).all() and (_allr <= 1).all())
_jn = np.concatenate([V(k, m) for k in ALL_KINDS for m in ["jain_tenant_success", "jain_server_util"]])
chk("Jain indices inside (0, 1]", (_jn > 0).all() and (_jn <= 1 + 1e-12).all())
chk("Topology restored after §17", N_NODES == 19)
if RUN["refiners"] and len(REFINER_RECS):
    chk("Equal refiner budgets", len({round(float(np.mean(v)), 6) for v in _epc.values()}) == 1)
if RUN["loso"] and len(LOSO_RECS):
    chk("LOSO folds complete", len(LOSO_RECS) == len(SOURCES)*len(P["LOSO_SEEDS"]))
_seen_oracle = sum(SEED_RECS[sd]["ref"]["oracle"]["reward"] >= SEED_RECS[sd]["learned"]["lsh_grl_hpfoo"]["reward"]
                   for sd in SEEDS)
chk("Oracle >= proposed at nominal load (expected, not required)", _seen_oracle == len(SEEDS),
    "%d/%d seeds" % (_seen_oracle, len(SEEDS)))

print("="*76); print("VERIFICATION SUMMARY".center(76)); print("="*76)
for name, ok, det in CHECKS:
    print("  [%s] %-52s %s" % ("PASS" if ok else "FAIL", name, det))
print("="*76)
print("\nKEY NUMBERS FOR THE MANUSCRIPT (profile %s, %d seeds)" % (PROFILE, len(SEEDS)))
print("  proposed reward              : %s" % MULTI.loc[LABELS["lsh_grl_hpfoo"], "reported"])
print("  greedy / max-min / GAT-PPO   : %s | %s | %s" % (MULTI.loc[LABELS["greedy"], "reported"],
      MULTI.loc[LABELS["maxmin"], "reported"], MULTI.loc[LABELS["gat_ppo"], "reported"]))
print("  oracle                       : %s" % MULTI.loc[LABELS["oracle"], "reported"])
print("  gap closed (multi-seed means)   : %.1f%%" % RESULTS["gap_closed_multiseed_pct"])
print("  significant after Holm       : %d of %d comparisons" % ((T4.p_holm < 0.05).sum(), len(T4)))
print("  cost ratio proposed / greedy : %.3f / %.3f" % (np.mean(V("lsh_grl_hpfoo", "cost_ratio")),
                                                        np.mean(V("greedy", "cost_ratio"))))
print("  Jain tenant success prop/gr/mm: %.4f / %.4f / %.4f" % tuple(np.mean(V(k, "jain_tenant_success"))
                                                                   for k in ["lsh_grl_hpfoo", "greedy", "maxmin"]))
if RUN["refiners"] and len(REFINER_RECS):
    print("  refiners (reward)            : " + ", ".join("%s %.4f" % (r.refiner, r.reward) for r in REFINER_TBL.itertuples()))
if RUN["loso"] and len(LOSO_RECS):
    print("  LOSO gap closed per fold     : " + ", ".join("%s %.0f%%" % (r.held_out, r.gap_closed) for r in LOSO_TBL.itertuples()))
if RUN["scalability"]:
    print("  decision ms at %s machines : %s" % (list(SCALE_TBL.nodes), [round(x, 3) for x in SCALE_TBL.decision_ms]))
print("\nArtefacts written to %s" % OUT_DIR)
pd.DataFrame(CHECKS, columns=["check", "pass", "detail"]).to_csv(os.path.join(OUT_DIR, "check_verification.csv"), index=False)

                            VERIFICATION SUMMARY                            
  [PASS] Gate 1 physics checks                                7 checks
  [PASS] Gate 2 headroom >= 5%                                7.52%
  [FAIL] Regression vs V4 (required values)                   51/68 match
  [PASS] Evaluation deterministic (Poisson replay = test)     
  [PASS] Local-only identical on every seed                   
  [PASS] All rewards inside [0, 1]                            
  [PASS] Jain indices inside (0, 1]                           
  [PASS] Topology restored after §17                          
  [PASS] Equal refiner budgets                                
  [PASS] LOSO folds complete                                  
  [PASS] Oracle >= proposed at nominal load (expected, not required) 10/10 seeds

KEY NUMBERS FOR THE MANUSCRIPT (profile full, 10 seeds)
  proposed reward              : 0.8736 ± 0.0045
  greedy / max-min / GAT-PPO   : 0.8248 ± 0.0089 | 0.8454 ± 0.0083 | 0.8343 ± 0.01

---

## Notes for the writing team (V5)

**Use these files for the revision**
- `table4_multiseed_tests_holm.csv` → Table 4 (now with Holm-corrected p, 95% CI of the difference, Cohen's d, wins, and GAT-PPO / max-min rows).
- `table_component_contributions.csv` → Table 6 (every component with paired CI, d and Holm p).
- `table_fairness*.csv`, Figure 16 → new fairness subsection (reviewer 02).
- `table_refiners.csv`, Figure 17 → new refiner comparison (reviewer 03).
- `table_mobility_*`, `table_param_sensitivity_ofat.csv`, `table_arrival_patterns.csv`, Figures 9, 15, 21 → robustness (reviewer 05).
- `table_loso_generalisation.csv`, Figure 18 → generalisation (reviewer 06).
- `table_scalability_*`, `table_training_budget.csv`, Figures 19–20, the §17 complexity table → reviewer 07.
- `table_lsh_inference.csv` → the reviewer 04 question on LSH decision time.
- `results_all.json["config"]` → full hyperparameter / software list (reviewer 07).
- `table_per_seed_all_methods.csv` → per-seed appendix (reviewer 05).

**Do not copy numbers from V4 or from the current manuscript.** Every multi-seed number changes in V5 because
of the graph fix. Only seed-0 single-run values (Table 5) and the reference policies stay the same.

**Honest framing that still applies:** the oracle is a myopic per-step bound, not a global optimum; trace-to-task
mappings are modelling assumptions; simulation only; single-cell radio; independent jobs; no security layer.
If a V5 result goes against the method (for example, a refiner matching HPFOO, or a tenant made worse off),
report it as found.

In [37]:
import shutil, os
if IN_COLAB:
    try:
        os.makedirs(DRIVE_OUT, exist_ok=True)
        shutil.copytree(OUT_DIR, DRIVE_OUT, dirs_exist_ok=True)
        print("copied to", DRIVE_OUT)
        print("plots:", sorted(os.listdir(os.path.join(DRIVE_OUT, "plots"))))
    except Exception as e:
        print("Drive copy failed:", e)
else:
    print("not on Colab - results are in", os.path.abspath(OUT_DIR))

copied to /content/drive/MyDrive/5G/results_V5
plots: ['Figure_12 – Reward Weight Sensitivity.png', 'Figure_13 – Pufferfish Step-Size Schedules.png', 'Figure_14 – Latency Term Decomposition.png', 'Figure_15 – Perturbed Operating Conditions.png', 'Figure_16 – Fairness Across Tenants and Servers.png', 'Figure_17 – Refiner Comparison.png', 'Figure_18 – Leave-One-Source-Out Generalisation.png', 'Figure_19 – Scalability with Network Size.png', 'Figure_20 – Throughput and Training Cost.png', 'Figure_21 – Arrival Pattern Robustness.png', 'Figure_3 – Multi-Seed Reward Comparison.png', 'Figure_9 – Mobility Robustness.png', 'output10_summary_table.png', 'output11_sensitivity.png', 'output1_physics_validation.png', 'output2_headroom.png', 'output3_convergence.png', 'output4_pareto.png', 'output5_mobility.png', 'output6_lsh_scalability.png', 'output7_ablation.png', 'output8_multiseed.png', 'output9_topology.png']
